# NSUCEC NeuraHack 2026 Preliminary Round — ScamShield BD

**Team:** `Turtlers` — Akib Hasan Pyil, K M Mubin, Abdullah Omar Faruk  
**Final Selected Kaggle Submission:** `submission_v9_final.csv` (**Public Score: `0.93661`**; second selection: `submission_v8_hotfix.csv`, **`0.93552`**)  

This notebook is a **100% self-contained, reproducible pipeline** implementing adversarial multilingual normalization, guide-priority rules (`1 > 2 > 4 > 3 > 5 > 0`), programmatic scenario generation (`72,000` conversations across `48` scenario families), scenario-disjoint CV, base teacher consensus domain adaptation (`mDeBERTa-v3-base` + `mE5-base`), diverse student distillation (`mMiniLM`, `mE5-small`, glossed `BGE-small`, `BiGRU-v5`, `DA-Tab LGBM/XGB/CatBoost`), selective `mDeBERTa-full` escalation, zero-leakage meta-calibration, and final `v8/v9` obfuscation refinement.

> **Execution Mode:** By default, `RUN_FULL_TRAINING = False` executes full EDA, normalization demonstrations, rule verification on `sample.csv`, cached OOD model evaluation, and strict schema validation on `submission.csv` (`~30s`). Set `RUN_FULL_TRAINING = True` to re-run end-to-end training & inference from scratch.

## 1. Environment Configuration, Declarations & Reproducibility Seeds (Sections 14–15)

### Mandatory External Resource & Compliance Declarations
- **Datasets**: Only the competition dataset (`sample.csv`, `test.csv`, `sample_submission.csv`) is used. No external labeled datasets are imported.
- **Programmatic Augmentation (Section 14)**: Synthetic data generator `generate_dataset_v4` (`seed=4242`) creates `72,000` labeled scenarios across `48` scenario families. Unlabeled `test.csv` text is hashed into `285,810` canonical structural skeletons; `45,000` cluster prototypes are scored by 2-fold scenario-disjoint teachers and consensus-filtered (`39,618` rows, `mean_conf=0.8201`) for domain adaptation. At inference time (`predict_inductive_v4`), predictions run **strictly inductively** per conversation without test-cluster lookup.
- **Open-Weight Pretrained Models (Hugging Face, MIT License)**:
  1. `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`
  2. `intfloat/multilingual-e5-base`
  3. `MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli`
  4. `intfloat/multilingual-e5-small`
  5. `BAAI/bge-small-en-v1.5`
- **External Hosted APIs**: None (`0` API calls; `.env` is intentionally empty).
- **Random Seeds**: `4242` (synth), `42`/`123`/`2026` (SVD/GBDT/meta), `101+f` & `201+f` (teachers), `70+f`/`909` (BiGRU), `300+f`/`500`/`777` (students & full mDeBERTa).

In [1]:
import os
import re
import gc
import sys
import math
import time
import json
import random
import hashlib
import warnings
import unicodedata
import argparse
from pathlib import Path
from typing import Dict, List, Tuple
from concurrent.futures import ProcessPoolExecutor

warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.sparse import hstack, vstack, csr_matrix
from scipy.optimize import minimize
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score, brier_score_loss, confusion_matrix
try:
    import lightgbm as lgb
except ImportError:
    lgb = None
try:
    import xgboost as xgb
except ImportError:
    xgb = None
try:
    from catboost import CatBoostClassifier
    HAS_CB = True
except ImportError:
    HAS_CB = False
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, get_cosine_schedule_with_warmup

# Reproducibility toggle: False runs EDA, rule/OOD audits & submission validation cleanly in <1 min
RUN_FULL_TRAINING = False

def _find_data_dir():
    for cand in [
        Path("."),
        Path("./dataset"),
        Path("./Main Preli Round/dataset"),
        Path("../Main Preli Round/dataset"),
        Path("/kaggle/input/nsucec-neurahack-2026"),
        Path("/kaggle/input/scamshield-bd"),
    ]:
        if (cand / "sample.csv").exists() and (cand / "test.csv").exists():
            return cand.resolve()
    raise FileNotFoundError("Could not locate sample.csv and test.csv!")

DATA_DIR = _find_data_dir()
WORK_DIR = Path.cwd()
ROOT = DATA_DIR.parent.parent if DATA_DIR.name == "dataset" and DATA_DIR.parent.name == "Main Preli Round" else WORK_DIR

# When inspecting existing workspace verification caches without overwriting package folder:
EXISTING_V2_CACHE = ROOT / "preli" / "v2" / "cache"
if (WORK_DIR / "cache").exists() or not EXISTING_V2_CACHE.exists():
    CACHE_DIR = WORK_DIR / "cache"
else:
    CACHE_DIR = EXISTING_V2_CACHE

CKPT_DIR = CACHE_DIR / "ckpt_v4"
OUT_DIR = CACHE_DIR / "preds_v4"
V2_DIR = WORK_DIR
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR_HF = os.environ.get("HF_HOME", r"D:\Datathon\hf_cache" if Path(r"D:\Datathon\hf_cache").exists() else str(WORK_DIR / "hf_cache"))
HF_LOCAL_ONLY = os.environ.get("HF_LOCAL_ONLY", "0") == "1"
DEV = "cuda" if torch.cuda.is_available() else "cpu"
XGB_DEV = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Python: {sys.version.split()[0]} | PyTorch: {torch.__version__} | Device: {DEV} | HF_LOCAL_ONLY={HF_LOCAL_ONLY}")
print(f"Transformers: {__import__('transformers').__version__} | LightGBM: {lgb.__version__ if lgb else 'optional(CPU-env)'} | XGBoost: {xgb.__version__ if xgb else 'optional(CPU-env)'}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"CACHE_DIR: {CACHE_DIR}")

Python: 3.13.9 | PyTorch: 2.11.0+cu128 | Device: cuda | HF_LOCAL_ONLY=False
Transformers: 5.16.1 | LightGBM: optional(CPU-env) | XGBoost: 3.2.0
DATA_DIR: D:\Datathon\nsucec_neurahackl\Main Preli Round\dataset
CACHE_DIR: D:\Datathon\nsucec_neurahackl\preli\v2\cache


## 2. Exploratory Data Analysis (Executed)

We inspect both `sample.csv` (`86` labeled records) and `test.csv` (`500,000` unlabeled records) across class distributions, multi-turn `|||` conversation density, Bangla script vs Banglish/English usage, and adversarial corruption vectors.

In [2]:
sample_df = pd.read_csv(DATA_DIR / "sample.csv")
test_raw_df = pd.read_csv(DATA_DIR / "test.csv", usecols=["id", "text"])
raw_tx = test_raw_df["text"].fillna("").astype(str)

print(f"sample.csv: {len(sample_df)} rows | label counts: {sample_df['label'].value_counts().sort_index().to_dict()}")
print(f"sample multi-turn (|||): {int(sample_df['text'].str.contains(r'\|\|\|').sum())} / {len(sample_df)}")
print(f"test.csv:   {len(test_raw_df):,} rows")
print(f"  Multi-turn (|||):        {int(raw_tx.str.contains(r'\|\|\|').sum()):,} ({100*raw_tx.str.contains(r'\|\|\|').mean():.2f}%)")
print(f"  Bangla script (U+0980):  {int(raw_tx.str.contains(r'[\u0980-\u09FF]').sum()):,} ({100*raw_tx.str.contains(r'[\u0980-\u09FF]').mean():.2f}%)")
print(f"  Zero-width obfuscation:  {int(raw_tx.str.contains(r'[\u200b-\u200f\u2060\ufeff\ufe0f\u00ad]').sum()):,} ({100*raw_tx.str.contains(r'[\u200b-\u200f\u2060\ufeff\ufe0f\u00ad]').mean():.2f}%)")
print(f"  Cyrillic/Greek homoglyph:{int(raw_tx.str.contains(r'[\u0400-\u04ff\u0370-\u03ff]').sum()):,} ({100*raw_tx.str.contains(r'[\u0400-\u04ff\u0370-\u03ff]').mean():.2f}%)")
lens = raw_tx.str.len()
print(f"  Char lengths -> min={int(lens.min())}, median={int(lens.median())}, p95={int(lens.quantile(0.95))}, max={int(lens.max())}")

sample.csv: 86 rows | label counts: {0: 32, 1: 12, 2: 10, 3: 11, 4: 9, 5: 12}
sample multi-turn (|||): 8 / 86
test.csv:   500,000 rows


  Multi-turn (|||):        107,248 (21.45%)


  Bangla script (U+0980):  54,837 (10.97%)


  Zero-width obfuscation:  54,310 (10.86%)


  Cyrillic/Greek homoglyph:42,538 (8.51%)
  Char lengths -> min=4, median=96, p95=255, max=616


## 3. Multilingual Normalization, Domain Masking & Feature Extraction (`normalizer_v4`)

Cleans zero-width joiners, homoglyphs, noise tokens, character floods, leetspeak, un-defangs URLs (`hxxps://`, `[.]`), preserves semantic emoji tokens (`<EMOJI_GIFT>`), prevents URL domain poisoning (`mask_urls_for_text_rules`), transliterates/glosses Bangla (`bn_to_en_roman_v4`), formats turn structure (`format_turn_aware`), and extracts `24` domain-general structural features (`extract_base_struct_v4`).

In [3]:
import re
import unicodedata
from typing import Dict, List, Tuple

ZW_RE = re.compile(r"[\u200b-\u200f\u2060\ufeff\ufe0f\u00ad]")
CONFUSABLES = str.maketrans({
    "\u0430": "a", "\u0435": "e", "\u043e": "o", "\u0456": "i",
    "\u0441": "c", "\u0440": "p", "\u0443": "y", "\u0445": "x",
    "\u0455": "s", "\u0501": "d", "\u04cf": "l",
    "\u0410": "A", "\u0415": "E", "\u041e": "O", "\u0406": "I",
    "\u0421": "C", "\u0420": "P", "\u0423": "Y", "\u0425": "X",
    "\u03bf": "o", "\u03b9": "i", "\u03b1": "a",
    "\u0964": ".",
})

EMOJI_MAP = {
    "\U0001f4b0": " <EMOJI_MONEY> ",
    "\U0001f381": " <EMOJI_GIFT> ",
    "\U0001f64f": " <EMOJI_PRAY> ",
    "\u26a0": " <EMOJI_WARN> ",
    "\U0001f4f2": " <EMOJI_MOBILE> ",
    "\U0001f525": " <EMOJI_FIRE> ",
    "\U0001f60a": " <EMOJI_SMILE> ",
}
EMOJI_RE = re.compile("|".join(re.escape(k) for k in EMOJI_MAP))
ANY_EMOJI_RE = re.compile(r"[\U0001F300-\U0001FAFF\u2600-\u27BF]+")

NOISE_TOKENS_RE = re.compile(
    r"\b(?:zzqwpo|wkrjfejejrn|mnbvcx|qzxvbn|xkcdpl|asdkjh|lkjsdfoiu|xcvbnm|zzqpwoeiru|qundvpyfjqx|wbmtzmh|qwertyz|asdfghj)\b",
    flags=re.IGNORECASE,
)

KEEP_DOUBLE_WORDS = {
    "free", "fee", "offer", "book", "good", "soon", "need", "keep", "see", "call",
    "bill", "will", "all", "full", "sell", "tell", "well", "pass", "less", "boss",
    "hello", "address", "account", "apply", "app", "support", "access", "confirm",
    "commission", "winning", "winner", "innocent", "sorry", "shipping", "hurry",
    "arriving", "arrived", "received", "success", "successful", "install", "installment",
    "whatsapp", "telegram", "message", "press", "correct", "collect", "off", "staff",
    "poor", "room", "too", "look", "door", "cool", "took", "week", "weekly", "speed",
    "guaranteed", "deen", "allah", "assalamu", "walaikum", "korben", "taka", "bolen",
    "connect", "appreciate", "allow", "allowance", "arrears", "attach", "attached",
    "officer", "approved", "official", "borrow", "tomorrow", "messenger", "shopping",
    "annual", "add", "missed", "error", "reverse", "review", "ambassador", "committee",
}

LEET_TRANS = str.maketrans({"0": "o", "1": "i", "3": "e", "4": "a", "5": "s", "7": "t"})

CORE_LEXICON = {
    "bkash", "nagad", "rocket", "upay", "surecash", "mcash", "tap", "ok", "wallet",
    "pathao", "daraz", "sundarban", "steadfast", "redx", "paperfly", "ecourier",
    "fedex", "dhl", "paribahan", "karatoa", "sheba", "shwapno", "aarong", "robi",
    "grameenphone", "airtel", "banglalink", "teletalk", "skitto", "unilever", "walton",
    "beximco", "jamuna", "bank", "islami", "prime", "brac", "city", "dbbl", "ucb",
    "ebl", "pubali", "sonali", "desco", "nesco", "wasa", "titas", "btcl", "dpdc", "breb",
    "agora", "yellow", "foodpanda", "chaldal", "pickaboo", "rokomari", "ajkerdeal",
    "pin", "otp", "cvv", "nid", "kyc", "card", "password", "passcode", "account",
    "verify", "verification", "login", "blocked", "suspend", "suspended", "expired",
    "expire", "expires", "flagged", "limit", "upgrade", "helpline", "security",
    "support", "agent", "staff", "officer", "department", "fraud", "control",
    "refund", "mistake", "wrong", "reverse", "reimbursement", "vul", "bhul", "ferot",
    "return", "cancel", "cancellation", "bolen", "bolo", "bolun", "pathan", "pathaben",
    "din", "dao", "deben", "share", "send", "enter", "reply", "click", "call", "dial",
    "read", "out", "winner", "won", "win", "prize", "lottery", "lucky", "draw",
    "cashback", "bonus", "gift", "reward", "rewards", "smartphone", "iphone", "car",
    "tv", "claim", "congratulations", "congrats", "jitechen", "jiteche", "free",
    "spin", "airdrop", "usdt", "survey", "mystery", "box", "parcel", "package",
    "courier", "delivery", "deliver", "delivered", "customs", "duty", "fee", "charge",
    "clearance", "storage", "address", "incomplete", "failed", "atke", "ase", "held",
    "confirm", "update", "track", "tracking", "cod", "cash", "ready", "rakhun",
    "rider", "shipped", "hub", "collect", "job", "part", "time", "task", "tasks",
    "earn", "earning", "income", "salary", "beton", "deposit", "invest", "investment",
    "crypto", "profit", "double", "guaranteed", "whatsapp", "telegram", "group",
    "join", "apply", "register", "registration", "offer", "sale", "discount", "chhar",
    "code", "voucher", "eid", "winter", "flash", "recharge", "flexiload", "gb",
    "bill", "due", "paid", "successful", "balance", "trxid", "credited", "debited",
    "prepaid", "token", "meter", "out", "received", "add", "hoyeche", "brother",
    "sir", "madam", "apu", "mama", "bhai", "chacha", "khala", "friend", "hello",
    "good", "morning", "number", "from", "help", "lagbe", "kotha", "chilo", "busy",
    "ekhon", "amar", "apnar", "apni", "koren", "korun", "korben", "kauke", "na",
    "never", "anyone", "scam", "phishing", "alert", "warning", "sabdhan", "last",
    "hurry", "immediately", "taratari", "within", "hours", "minutes", "today",
    "only", "limited", "nearest", "outlet", "use", "via", "with", "your", "you",
    "new", "exciting", "features", "cooperation", "thank", "subject", "sender",
    "caller", "kal", "tay", "dekha", "hobe", "happy", "birthday", "dinner",
    "tonight", "treat", "ielts", "course", "admission", "cholche", "exam",
    "routine", "bazar", "school", "college", "class", "meeting", "interview",
    "library", "reserved", "available", "appointment", "dentist", "reschedule",
    "book", "flat", "apartment", "dream", "hair", "oil", "pizza", "combo",
    "drink", "days", "abroad", "malaysia", "visa", "position", "office", "copy",
    "bring", "welcome", "install", "order", "valid", "minute", "minutes", "invalid",
    "block", "report", "someone", "messaged", "scholarship", "allowance", "arrears",
    "citizen", "student", "approved", "uniform", "kit", "ambassador", "handler",
    "tuition", "portal", "spring", "bank", "booth", "letter", "attached", "hr",
    "sign", "joining", "date", "renew", "pack", "ota", "ashche", "ashe", "je",
    "seta", "theke", "detect", "borrow", "trouble", "deactivated", "sim", "poto",
    "photo", "both", "sides", "review", "proceed", "receive", "transfer", "kindly",
    "kamai", "unlock", "arbitrage", "bot", "app", "update", "exciting", "features",
    "weekend", "fiber", "router", "birthday", "tuition", "booth", "apartment",
    "installment", "delivered", "release", "helpline", "limited", "winners",
}

EXACT_TYPO_REPAIRS = {
    "bksah": "bkash", "bksh": "bkash", "bkah": "bkash", "baksh": "bkash",
    "bkashop": "bkash otp", "srecash": "surecash", "upyause": "upay use",
    "upayuse": "upay use", "chrtered": "chartered", "recahrge": "recharge",
    "pge": "page", "agnt": "agent", "dal": "dial", "dil": "dial",
    "jitechne": "jitechen", "daivd": "david", "firend": "friend",
    "fooooor": "for", "yu": "you", "yor": "your", "yoru": "your",
    "cad": "card", "adn": "and", "ad": "and", "olny": "only",
    "hsbeen": "has been", "reutrnit": "return it", "helplabe": "help lagbe",
    "hepl": "help", "rpely": "reply", "contiune": "continue", "hep": "help",
    "etka": "ekta", "ekon": "ekhon", "bsuy": "busy", "tyadeha": "tay dekha",
    "dekhahobe": "dekha hobe", "nearestoutlet": "nearest outlet",
    "withexciting": "with exciting", "yourcooperation": "your cooperation",
    "numberfrom": "number from", "whichfriend": "which friend",
    "kihelp": "ki help", "releasekorte": "release korte", "forour": "for our",
    "assecurity": "as security", "bhulnumber": "bhul number", "deha": "dekha",
    "ist": "is", "iste": "is the", "si,rapni": "sir apni",
    "brothe,etka": "brother ekta", "brothe,ekta": "brother ekta",
    "13l75": "ielts", "ieltsstuy": "ielts study",
    "brthday": "birthday", "dinnertonight": "dinner tonight",
    "currentpin": "current pin", "admissioncholche": "admission cholche",
    "coursee": "course", "shre": "share", "sharekroun": "share korun",
    "korbn": "korben", "dibenna": "diben na", "kuke": "kauke",
    "kuake": "kauke", "neevr": "never", "prcel": "parcel", "obth": "both",
    "addrss": "address", "poto": "photo", "mnths": "months", "thnks": "thanks",
    "jecoooode": "je code", "jecooode": "je code", "jecode": "je code",
    "ashcheota": "ashche ota", "thatis": "that is", "nevershare": "never share",
    "yourpin": "your pin", "pinta": "pin ta", "detecthoyeceh": "detect hoyeche",
    "applykorun": "apply korun", "andcvv": "and cvv", "passwordat": "password at",
    "airt3l": "airtel", "4ir73l": "airtel", "5k1tto": "skitto",
    "4dm15s10n": "admission", "c4shb4ck": "cashback",
    "securitydepartmenttheke": "security department theke",
    "accountwillbe": "account will be", "willbe": "will be",
    "sharekorben": "share korben", "bolbenna": "bolben na",
    "hobecod": "hobe cod", "apnaraccount": "apnar account",
    "takinle": "ta kinle", "tafree": "ta free", "shorttoprojojjo": "shortto projojjo",
    "releaseit": "release it", "unlockkorun": "unlock korun", "sendingnid": "sending nid",
    "potoand": "photo and", "couldnt": "could not", "deliverd": "delivered",
    "ktakotha": "ekta kotha", "kothachilo": "kotha chilo", "kiupay": "ki upay",
    "usekoren": "use koren", "seeender": "sender", "sennnder": "sender",
    "goodmoring": "good morning", "frmo": "from", "taks": "tasks",
    "newapp": "new app", "updte": "update", "bnak": "bank", "toapply": "to apply",
    "helpla": "help lagbe", "bhulnumber-e": "bhul number", "feedin": "fee din", "cashout": "cash out",
}

SUSPICIOUS_TLDS = {"top", "xyz", "site", "online", "vip", "link", "click", "info", "co", "cc", "icu", "live"}
SHORTENER_DOMAINS = {"bit.ly", "tinyurl.com", "t.co", "rb.gy", "cutt.ly", "shorturl.at"}
OFFICIAL_BRAND_DOMAINS = {
    "bkash.com", "bkash.com.bd", "bkash.org.bd", "bkash.net",
    "nagad.com", "nagad.com.bd", "nagad.net", "nagad.org.bd",
    "rocket.com.bd", "rocket.org.bd", "upay.com.bd", "surecash.com", "surecash.com.bd", "surecash.org.bd",
    "tap.com", "tap.org.bd", "okwallet.com.bd", "pathaopay.org.bd", "pathaopay.com.bd",
    "pathaofood.com.bd", "daraz.com.bd", "robi.com", "robi.com.bd", "robi.net", "robi.org.bd",
    "grameenphone.com", "grameenphone.com.bd", "grameenphone.net", "grameenphone.org.bd",
    "gp.com", "gp.com.bd", "gp.net", "gp.org.bd", "airtel.com", "airtel.com.bd", "airtel.net", "airtel.org.bd",
    "banglalink.com", "banglalink.com.bd", "banglalink.net", "banglalink.org.bd",
    "teletalk.com", "teletalk.com.bd", "teletalk.net", "teletalk.org.bd",
    "skitto.com", "skitto.com.bd", "skitto.net", "skitto.org.bd",
    "payment.breb.org.bd", "ecourier.com.bd", "citybank.org.bd", "ebl.net",
    "bracbank.net", "shebadelivery.net", "agora.net", "yellow.org.bd", "foodpanda.org.bd",
    "mcash.net",
}

BN_SPLIT_RE = re.compile(
    r"(ওটিপি|পিন|কোড|টাকা|লিংকে|পার্সেল|ক্লিক|যোগাযোগ)(দিন|করুন|নিন|পাঠান|হবে|আটকে|দিয়ে|করতে)"
)
PHONE_RE = re.compile(r"(?:\+?880[\s-]?)?01\d[\dxX]{6,9}\b")


def deobfuscate_basic(text: str) -> str:
    t = unicodedata.normalize("NFKC", str(text))
    t = ZW_RE.sub("", t).translate(CONFUSABLES)
    t = EMOJI_RE.sub(lambda m: EMOJI_MAP[m.group(0)], t)
    t = ANY_EMOJI_RE.sub(" ", t)
    t = re.sub(r"\[\s*\.\s*\]|\[\s*\]", ".", t)
    t = re.sub(r"\bh[t7x]{2}p(s|5)?://", lambda m: "https://" if m.group(1) else "http://", t, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", t).strip()


def _split_glued_ascii(tok: str) -> str:
    if len(tok) < 7 or tok in CORE_LEXICON or not tok.isalpha():
        return tok
    # Greedy 2- or 3-way split on CORE_LEXICON
    n = len(tok)
    dp = [None] * (n + 1)
    dp[0] = []
    for i in range(n):
        if dp[i] is None:
            continue
        for j in range(i + 2, n + 1):
            sub = tok[i:j]
            if sub in CORE_LEXICON:
                cand = dp[i] + [sub]
                if dp[j] is None or len(cand) < len(dp[j]):
                    dp[j] = cand
    if dp[n] is not None and 2 <= len(dp[n]) <= 4:
        return " ".join(dp[n])
    return tok


def _repair_token(tok: str) -> str:
    tl = tok.lower()
    if tl in EXACT_TYPO_REPAIRS:
        return EXACT_TYPO_REPAIRS[tl]
    if PHONE_RE.fullmatch(tok):
        return tok
    if re.search(r"([a-z])\1{2,}", tl):
        c2 = re.sub(r"([a-z])\1{2,}", r"\1\1", tl)
        c1 = re.sub(r"([a-z])\1{2,}", r"\1", tl)
        tl = c2 if c2 in KEEP_DOUBLE_WORDS else c1
        if tl in EXACT_TYPO_REPAIRS:
            return EXACT_TYPO_REPAIRS[tl]
    if re.search(r"[a-z]", tok) and re.search(r"[013457]", tok):
        # Preserve promo/trx codes like SAVE20, EID10, SP74153, FLASH1575
        if not re.match(r"^(?:save|deal|eid|winter|welcome|flash|sp|cn|pf|sc|ec|rx|sf)\d+$", tl) and not re.match(r"^[a-z]{1,3}\d{4,}$", tl):
            leet_cand = tl.translate(LEET_TRANS)
            if leet_cand in CORE_LEXICON or (re.fullmatch(r"[a-z]{3,}", leet_cand) and _split_glued_ascii(leet_cand) != leet_cand):
                tl = leet_cand
                if tl in EXACT_TYPO_REPAIRS:
                    return EXACT_TYPO_REPAIRS[tl]
    return _split_glued_ascii(tl)


def clean_for_model(text: str) -> str:
    t = deobfuscate_basic(text)
    t = NOISE_TOKENS_RE.sub(" ", t)
    # Fix Bengali glued post-positions/verbs like "ওটিপিদিন" -> "ওটিপি দিন"
    t = BN_SPLIT_RE.sub(r"\1 \2", t)
    # Split colon/comma/slash glued alphanumeric tokens outside URLs & |||
    parts = re.split(r"(\|\|\||https?://\S+|\s+)", t)
    out = []
    for p in parts:
        if not p or p.isspace():
            continue
        if p == "|||" or p.startswith("http"):
            # Repair domain-level leetspeak like 5k1tto.com.bd
            if p.startswith("http"):
                m = re.match(r"^(https?://)([^/\s]+)(.*)$", p, flags=re.I)
                if m:
                    dom = m.group(2).lower().replace("5k1tto", "skitto").replace("4ir73l", "airtel").replace("c0m", "com")
                    p = m.group(1) + dom + m.group(3)
            out.append(p)
        elif PHONE_RE.fullmatch(p):
            out.append(p)
        else:
            # Handle hyphen/colon-joined pieces gently unless USSD or short code
            if re.match(r"^\*\d+(?:\*\d+)?#$", p):
                out.append(p)
            else:
                # Split punctuation attached like Part-time, Robi:Special, bhulnumber-e
                subparts = re.split(r"([:,/()]+)", p)
                rebuilt = []
                for sp in subparts:
                    if not sp or re.fullmatch(r"[:,/()]+", sp):
                        if sp:
                            rebuilt.append(sp)
                    elif re.fullmatch(r"[A-Za-z0-9']+", sp):
                        rebuilt.append(_repair_token(sp))
                    else:
                        m = re.match(r"^([^A-Za-z0-9']*)([A-Za-z0-9']+)([^A-Za-z0-9']*)$", sp)
                        if m:
                            rebuilt.append(m.group(1) + _repair_token(m.group(2)) + m.group(3))
                        else:
                            rebuilt.append(sp)
                joined = "".join(rebuilt)
                # Drop unknown random 7+ consonant-heavy gibberish tokens
                if re.fullmatch(r"[bcdfghjklmnpqrstvwxz]{6,}", joined):
                    continue
                out.append(joined)
    res = re.sub(r"\s+", " ", " ".join(out)).strip()
    return res if res else "<NOISE>"


def mask_urls_for_text_rules(clean_text: str) -> Tuple[str, Dict[str, int]]:
    urls = re.findall(r"https?://([^\s/]+)(/[^\s\"']*)?", clean_text.lower())
    flags = {"has_url": int(len(urls) > 0), "has_susp_tld": 0, "has_official_dom": 0, "has_shortener": 0, "has_apk": int(".apk" in clean_text.lower()), "has_tg_wa": int(bool(re.search(r"t\.me/|wa\.me/", clean_text.lower())))}
    def _repl(m):
        dom = m.group(1).lower()
        path = (m.group(2) or "").lower()
        tld = dom.split(".")[-1] if "." in dom else ""
        if ".apk" in path:
            flags["has_apk"] = 1
            return " <URL_APK> "
        if "t.me" in dom or "wa.me" in dom:
            flags["has_tg_wa"] = 1
            return " <URL_TGWA> "
        if dom in SHORTENER_DOMAINS:
            flags["has_shortener"] = 1
            return " <URL_SHORT> "
        if dom in OFFICIAL_BRAND_DOMAINS or dom.endswith((".edu.bd", ".ac.bd", ".gov.bd", ".edu", ".gov")):
            flags["has_official_dom"] = 1
            return " <URL_OFFICIAL> "
        if tld in SUSPICIOUS_TLDS:
            flags["has_susp_tld"] = 1
            return " <URL_SUSP> "
        return " <URL> "
    masked = re.sub(r"https?://([^\s/]+)(/[^\s\"']*)?", _repl, clean_text)
    # Also mask scheme-less shorteners/tg/wa links
    masked = re.sub(r"\b(?:bit\.ly|tinyurl\.com|t\.co|rb\.gy|cutt\.ly|shorturl\.at)/\S+", lambda m: _set_flag_short(flags), masked)
    masked = re.sub(r"\b(?:t\.me|wa\.me)/\S+", lambda m: _set_flag_tgwa(flags), masked)
    return masked, flags


def _set_flag_short(flags):
    flags["has_url"] = 1
    flags["has_shortener"] = 1
    return " <URL_SHORT> "


def _set_flag_tgwa(flags):
    flags["has_url"] = 1
    flags["has_tg_wa"] = 1
    return " <URL_TGWA> "


def extract_base_struct_v4(raw_text: str, clean_text: str) -> list:
    """24-dim structural, obfuscation, OOV-proxy, and turn-structure features."""
    raw_s = str(raw_text)
    cl_low = clean_text.lower()
    masked, uflags = mask_urls_for_text_rules(clean_text)
    turns = clean_text.split("|||")
    zw_count = len(ZW_RE.findall(raw_s))
    cyr_count = sum(1 for c in raw_s if "\u0400" <= c <= "\u04ff")
    alpha_tokens = re.findall(r"[a-z]{3,}", cl_low)
    has_bn = int(bool(re.search(r"[\u0980-\u09FF]", clean_text)))
    known_hits = sum(1 for w in alpha_tokens if w in CORE_LEXICON)
    known_ratio = float(known_hits / max(1, len(alpha_tokens)))
    has_phone = int(bool(re.search(r"(?:\+?880\s*)?01\d[\dxX]{5,9}|\b\d{5}-[\dxX]+|\b\*\d{3}(?:\*\d+)?#", cl_low)))
    has_short_code = int(bool(re.search(r"\b(?:16247|16216|16263|16167)\b|\*\d{3}#", cl_low)))
    has_money = int(bool(re.search(r"\b(?:tk|bdt|taka)\b|টাকা|\$", cl_low)))
    has_hxxp = int(bool(re.search(r"h[t7x]{2}p", raw_s, flags=re.I)))
    has_noise_tok = int(bool(NOISE_TOKENS_RE.search(raw_s)))
    is_noise_only = int(clean_text == "<NOISE>" or ((not has_bn) and known_hits == 0 and uflags["has_url"] == 0 and not re.search(r"\d{3,}", cl_low) and len(alpha_tokens) <= 4))
    has_innocent_wrap = int(bool(re.search(r"hope this message finds you well|dear valued customer|regular communication|thank you for your cooperation", cl_low)))
    return [
        float(len(raw_s)), float(len(clean_text)), float(len(turns)), float(has_bn),
        float(uflags["has_url"]), float(has_hxxp), float(uflags["has_susp_tld"]),
        float(uflags["has_official_dom"]), float(uflags["has_shortener"]), float(uflags["has_apk"]),
        float(uflags["has_tg_wa"]), float(has_phone), float(has_short_code), float(has_money),
        float(has_noise_tok), float(is_noise_only), float(zw_count), float(cyr_count),
        float(known_hits), float(known_ratio), float(len(alpha_tokens)), float(has_innocent_wrap),
        float("|CTX|" in format_turn_aware(clean_text)), float("[" in raw_s and "]" in raw_s),
    ]


def format_turn_aware(clean_text: str, mask_urls: bool = True) -> str:
    """Mask untrusted URLs (killing spoofed brand/gift domains) & put key/late turns upfront."""
    t_in = mask_urls_for_text_rules(clean_text)[0] if mask_urls else clean_text
    t_in = re.sub(r"\be(\d+%)", r"e \1", t_in)
    if "|||" not in t_in or len(t_in) <= 180:
        return t_in
    turns = [x.strip() for x in t_in.split("|||") if x.strip()]
    if len(turns) <= 2:
        return t_in
    sig_re = re.compile(r"pin|otp|cvv|nid|password|code|fee|pay|deposit|won|prize|gift|parcel|customs|link|<URL|job|income|পিন|ওটিপি|টাকা|ফি|জিতে|পার্সেল|চাকরি|যোগ|scam|phishing|never share|card number|ludo", re.I)
    key_turns = [t for t in turns if sig_re.search(t)]
    tail_turns = turns[-2:]
    picked = []
    for t in key_turns + tail_turns:
        if t not in picked:
            picked.append(t)
    head = " ||| ".join(picked[-3:])
    return f"{head} |CTX| {t_in}"


BN_GLOSS = {
    "বিকাশ": "bkash", "নগদ": "nagad", "রকেট": "rocket", "উপায়": "upay",
    "পিন": "pin", "ওটিপি": "otp", "কোড": "code", "এনআইডি": "nid",
    "কার্ড": "card", "পাসওয়ার্ড": "password", "টাকা": "taka money",
    "হাজার": "thousand", "লাখ": "lakh", "জিতেছেন": "won prize",
    "জিতেছে": "won prize", "জিতার": "win prize", "লটারি": "lottery",
    "লাকি": "lucky", "ড্র": "draw", "ক্যাশব্যাক": "cashback",
    "বোনাস": "bonus", "পুরস্কার": "prize reward", "উপহার": "gift",
    "অভিনন্দন": "congratulations", "পার্সেল": "parcel", "ডেলিভারি": "delivery",
    "কাস্টমস": "customs", "শুল্ক": "duty fee", "আটকে": "held stuck",
    "ঠিকানা": "address", "ব্যর্থ": "failed", "আপডেট": "update",
    "লিংকে": "link", "লিংক": "link", "ক্লিক": "click", "যান": "visit",
    "পাঠান": "send", "বলুন": "tell", "দিন": "give", "করুন": "do",
    "নিন": "take", "দাও": "give", "দেবেন": "give", "জানান": "inform",
    "অ্যাকাউন্ট": "account", "একাউন্ট": "account", "বন্ধ": "closed blocked",
    "ব্লক": "block", "সতর্ক": "warning alert", "ভুয়া": "fake",
    "প্রতারণা": "scam fraud", "স্ক্যাম": "scam", "সাবধান": "beware",
    "আয়": "earn income", "বিনিয়োগ": "invest", "দ্বিগুণ": "double",
    "নিশ্চিত": "guaranteed", "লাভ": "profit", "চাকরি": "job",
    "কাজ": "work task", "বেতন": "salary", "লোক": "staff hiring",
    "নিচ্ছে": "hiring", "রেজিস্ট্রেশন": "registration", "ঘরে": "home",
    "বসে": "sitting", "হোয়াটসঅ্যাপে": "whatsapp", "টেলিগ্রামে": "telegram",
    "যোগ": "join", "যোগাযোগ": "contact call", "কল": "call",
    "ছাড়": "discount off", "অফার": "offer", "বিল": "bill",
    "পেমেন্টে": "payment", "রিচার্জে": "recharge", "পান": "get",
    "ডায়াল": "dial", "ক্যাশ": "cash", "আউট": "out", "সফল": "successful",
    "ব্যালেন্স": "balance", "জমা": "credited deposit", "পেয়েছেন": "received",
    "ট্রানজেকশন": "transaction", "গ্রাহক": "customer", "প্রিয়": "dear",
    "এখনই": "immediately now", "আজই": "today", "আজ": "today",
    "নইলে": "otherwise", "ফি": "fee", "চার্জ": "charge", "প্রসেসিং": "processing",
    "পরিশোধ": "pay", "ভুল": "mistake wrong", "ভুলে": "by mistake",
    "ফেরত": "return", "দয়া": "please", "করে": "by", "কাউকে": "anyone",
    "শেয়ার": "share", "কখনো": "never", "বৈধ": "valid", "মিনিট": "minutes",
    "ডিসেম্বর": "december", "নভেম্বর": "november", "ভাতা": "allowance",
    "বয়স্ক": "old age", "নম্বরটি": "number", "নম্বরে": "number",
}
BN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
BN_CHAR = {
    "অ": "o", "আ": "a", "ই": "i", "ঈ": "i", "উ": "u", "ঊ": "u", "ঋ": "ri",
    "এ": "e", "ঐ": "oi", "ও": "o", "ঔ": "ou",
    "া": "a", "ি": "i", "ী": "i", "ু": "u", "ূ": "u", "ৃ": "ri",
    "ে": "e", "ৈ": "oi", "ো": "o", "ৌ": "ou",
    "ক": "k", "খ": "kh", "গ": "g", "ঘ": "gh", "ঙ": "ng",
    "চ": "ch", "ছ": "chh", "জ": "j", "ঝ": "jh", "ঞ": "n",
    "ট": "t", "ঠ": "th", "ড": "d", "ঢ": "dh", "ণ": "n",
    "ত": "t", "থ": "th", "দ": "d", "ধ": "dh", "ন": "n",
    "প": "p", "ফ": "f", "ব": "b", "ভ": "bh", "ম": "m",
    "য": "j", "র": "r", "ল": "l", "শ": "sh", "ষ": "sh", "স": "s", "হ": "h",
    "ড়": "r", "ঢ়": "rh", "য়": "y", "ৎ": "t", "ং": "ng", "ঃ": "h", "ঁ": "n", "্": "",
    "৳": "Tk ",
}
BN_CONSONANTS = set("কখগঘঙচছজঝঞটঠডঢণতথদধনপফবভমযরলশষসহড়ঢ়য়")


def bn_to_en_roman_v4(text: str) -> str:
    s = str(text).translate(BN_DIGITS)
    if not re.search(r"[\u0980-\u09FF৳]", s):
        return s

    def repl_word(m):
        w = m.group(0)
        gloss = BN_GLOSS.get(w)
        out = []
        n = len(w)
        for i, ch in enumerate(w):
            out.append(BN_CHAR.get(ch, ch))
            if ch in BN_CONSONANTS and i + 1 < n and w[i + 1] in BN_CONSONANTS:
                out.append("a")
        rw = "".join(out)
        if gloss and gloss != rw:
            return f"{rw} ({gloss})"
        return rw

    return re.sub(r"[\u0980-\u09FF]+|৳", lambda m: "Tk " if m.group(0) == "৳" else repl_word(m), s)

In [4]:
# Live adversarial normalization demonstration (raw obfuscated -> normalized clean -> rule/BGE gloss)
demo_msgs = [
    "mnbvcx Your bKash PIIIN will expire today Rpely witttth your currentP1N ad a new PIN to 💰 contiune service",
    "866970 is y\u200bournagad lоgіncode don07 share it w\u200bith anyoe even mnbvcx na\u200bgad 5taff",
    "khala ami,pohne hariyeche 💰 Tomr nubmer e je cooodeaschhe oa bOLo",
    "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ১৫০ টাকা পরিশোধ করতে hxxps://tap-track[.]site/7be4 এ ক্লিক করুন",
]
for i, msg in enumerate(demo_msgs, 1):
    c = clean_for_model(msg)
    m, u = mask_urls_for_text_rules(c)
    print(f"[{i}] CLEAN: {c}")
    if m != c:
        print(f"    MASKD: {m}")

[1] CLEAN: your bkash pin will expire today reply with your currentp1n and a new pin to <EMOJI_MONEY> continue service
[2] CLEAN: 866970 is your nagad login code don07 share it with anyoe even nagad staff
[3] CLEAN: khala ami,pohne hariyeche <EMOJI_MONEY> tomr nubmer e je codeaschhe oa bolo
[4] CLEAN: আপনার পার্সেল আটকে আছে. ডেলিভারি চার্জ ১৫০ টাকা পরিশোধ করতে https://tap-track.site/7be4 এ ক্লিক করুন
    MASKD: আপনার পার্সেল আটকে আছে. ডেলিভারি চার্জ ১৫০ টাকা পরিশোধ করতে  <URL_SUSP>  এ ক্লিক করুন


## 4. Scenario-Disjoint Validation Splits & Section 10 Metric (`cv_ood`)

Implements consistent post-decoding (`action=0` & `NONE` iff `label in {0, 5}`), coarse scenario-family assignment, balanced 3-fold scenario-disjoint partitioning (`get_balanced_3folds` retaining 67% of every class per fold), and `score_section10_proxy`.

In [5]:
import re
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

TECH_LIST = ["T1", "T2", "T3", "T4", "T5", "T6"]


def tech_str_to_vec(t_str: str) -> np.ndarray:
    v = np.zeros(6, dtype=np.int32)
    s = str(t_str).strip()
    if s and s != "NONE":
        for code in s.split("|"):
            if code in TECH_LIST:
                v[TECH_LIST.index(code)] = 1
    return v


def vec_to_tech_str(v: np.ndarray) -> str:
    codes = [TECH_LIST[i] for i in range(6) if v[i] > 0]
    return "|".join(codes) if codes else "NONE"


def decode_consistent(p_lbl, p_tech, p_act, tech_th=0.5):
    pred_l = p_lbl.argmax(axis=1).astype(int)
    pred_a = np.where(np.isin(pred_l, [0, 5]), 0, 1 + p_act[:, 1:].argmax(axis=1)).astype(int)
    if np.ndim(tech_th) == 0:
        pred_t = (p_tech >= tech_th).astype(int)
    else:
        pred_t = (p_tech >= np.asarray(tech_th)[None, :]).astype(int)
    is_np = np.isin(pred_l, [0, 5])
    pred_t[is_np] = 0
    need_one = (~is_np) & (pred_t.sum(axis=1) == 0)
    if need_one.any():
        pred_t[need_one, p_tech[need_one].argmax(axis=1)] = 1
    return pred_l, pred_t, pred_a


def assign_sample_scenario_family(clean_text: str, label: int) -> str:
    """Coarse scenario-family clustering on sample.csv for leak-free Leave-One-Scenario-Out CV."""
    t = clean_text.lower()
    if label == 1:
        if any(k in t for k in ("vul", "bhul", "mistake", "wrong", "ferot")):
            return "mfs_mistake"
        if any(k in t for k in ("jitechen", "won", "bonus", "prize", "parcel", "atke", "customs", "ghor bose")):
            return "pri1_cross_cat"
        return "mfs_cred_harvest"
    if label == 2:
        if "cashback" in t or "ক্যাশব্যাক" in t or "jitechen" in t or "জিতেছেন" in t:
            return "prize_cashback_lakh"
        return "prize_gift_voucher"
    if label == 3:
        if any(k in t for k in ("data entry", "unilever", "bdjobs", "beximco", "abroad", "malaysia")):
            return "job_recruitment_fee"
        if any(k in t for k in ("crypto", "invest", "double", "arbitrage")):
            return "job_crypto_invest"
        return "job_task_earn"
    if label == 4:
        if any(k in t for k in ("customs", "duty", "china", "malaysia")):
            return "deliv_customs_duty"
        if any(k in t for k in ("failed", "address", "delivery attempt")):
            return "deliv_addr_phish"
        return "deliv_fee_link"
    if label == 5:
        if any(k in t for k in ("sale", "discount", "off", "চাড়", "ছাড়", "11.11", "eid")):
            return "promo_sale_disc"
        return "promo_offer_ad"
    # label == 0 sub-families
    if any(k in t for k in ("scam", "phishing", "sabdhan", "block")):
        return "legit_scam_warn"
    if any(k in t for k in ("otp", "login code", "verification code")):
        return "legit_otp_warn"
    if any(k in t for k in ("cash out", "cash-out", "received", "debited", "credited", "bill", "recharge", "বিল", "ক্যাশ আউট")):
        return "legit_tx_bill"
    if any(k in t for k in ("parcel", "order", "delivered", "rider", "পার্সেল", "ডেলিভারি")):
        return "legit_deliv_ok"
    return "legit_chat_other"


# Scenario-Disjoint 2-Fold Assignment (matching sample.csv & synth_64k_v4 families share zero scenario leakage across folds!)
FOLD_1_FAMILIES = {
    "legit_tx_bill", "legit_hr_doc", "legit_tuition", "legit_chat_faq", "legit_chat_other",
    "mfs_account_block", "mfs_limit_aml", "mfs_cred_harvest", "wrong_tx_readout",
    "borrowed_wallet_pin", "pri1_parcel_pin", "pri1_prize_pin", "pri1_cross_cat",
    "prize_link_bn", "prize_mystery_box", "prize_fee_bn", "prize_call_wa_bn_en", "prize_pii_reply", "prize_cashback_lakh",
    "job_crypto_arb", "job_package_ambassador", "job_double_invest", "job_data_entry", "job_crypto_invest",
    "deliv_storage_charge", "deliv_addr_url", "deliv_call_phone", "deliv_addr_phish",
    "promo_app_bogo", "promo_telco", "promo_offer_ad",
}


def get_scenario_disjoint_folds(families) -> np.ndarray:
    """Return binary fold IDs (0 or 1) with zero scenario overlap between fold 0 and fold 1."""
    return np.array([1 if str(f) in FOLD_1_FAMILIES else 0 for f in families], dtype=np.int32)


BALANCED_3FOLD_MAP = {
    # Fold 0
    "legit_otp": 0, "legit_otp_warn": 0, "legit_chat_other": 0, "legit_noise": 0,
    "mfs_alert_card": 0, "wrong_tx_pin": 0, "pri1_scholarship_pin": 0, "mfs_mistake": 0,
    "prize_link_en": 0, "prize_router_allowance": 0, "prize_apk_rewards": 0, "prize_cashback_lakh": 0,
    "job_task_like": 0, "job_bn_hiring_reg": 0, "job_double_invest": 0, "job_recruitment_fee": 0,
    "deliv_customs_duty": 0, "deliv_addr_phish": 0, "promo_sale": 0, "promo_sale_disc": 0,
    # Fold 1
    "legit_tx_bill": 1, "legit_cod": 1, "legit_deliv_ok": 1, "legit_chat_faq": 1,
    "mfs_account_block": 1, "wrong_tx_return_money": 1, "pri1_parcel_pin": 1, "pri1_prize_pin": 1, "pri1_cross_cat": 1,
    "prize_link_bn": 1, "prize_fee_en": 1, "prize_call_wa_bn_en": 1,
    "job_crypto_arb": 1, "job_package_ambassador": 1, "job_call_visa_bn": 1, "job_task_earn": 1,
    "deliv_storage_charge": 1, "deliv_bn_url": 1, "deliv_fee_link": 1, "promo_app_bogo": 1, "promo_realestate_edu": 1,
    # Fold 2
    "legit_agent_cashout": 2, "legit_hr_doc": 2, "legit_tuition": 2, "legit_scam_warn": 2,
    "mfs_limit_aml": 2, "mfs_breb_western": 2, "wrong_tx_readout": 2, "borrowed_wallet_pin": 2, "pri1_job_pin": 2, "mfs_cred_harvest": 2,
    "prize_usdt_survey": 2, "prize_mystery_box": 2, "prize_fee_bn": 2, "prize_pii_reply": 2, "prize_gift_voucher": 2,
    "job_volunteer_review": 2, "job_data_entry": 2, "job_crypto_invest": 2,
    "deliv_addr_url": 2, "deliv_call_phone": 2, "promo_telco": 2, "promo_offer_ad": 2,
}


def get_balanced_3folds(families) -> np.ndarray:
    """Balanced 3-fold scenario-disjoint partition (every fold keeps 67% of every class!)."""
    return np.array([BALANCED_3FOLD_MAP.get(str(f), hash(str(f)) % 3) for f in families], dtype=np.int32)


def score_section10_proxy(
    y_l_primary, p_l_primary, y_t_primary, p_t_primary, y_a_primary, p_a_primary, conf_primary,
    y_l_ood=None, p_l_ood=None, y_t_ood=None, p_t_ood=None, y_a_ood=None, p_a_ood=None, conf_ood=None,
    y_l_adv=None, p_l_adv=None, conf_adv=None,
    y_l_multi=None, p_l_multi=None,
):
    """Exact Section 10 composite metric across Primary, OOD Domain-Shift/Novel, Adversarial, and Calibration."""
    fa = 100.0 * f1_score(y_l_primary, p_l_primary, average="macro", zero_division=0)
    fb = 100.0 * f1_score(y_t_primary, p_t_primary, average="macro", zero_division=0)
    fc = 100.0 * f1_score(y_a_primary, p_a_primary, average="macro", zero_division=0)

    fd = 100.0 * f1_score(y_l_ood, p_l_ood, average="macro", zero_division=0) if y_l_ood is not None else fa
    fe = 100.0 * f1_score(y_l_adv, p_l_adv, average="macro", zero_division=0) if y_l_adv is not None else fa
    ff = 100.0 * f1_score(y_l_multi, p_l_multi, average="macro", zero_division=0) if y_l_multi is not None else fd

    if conf_ood is not None and y_l_ood is not None:
        all_corr = np.concatenate([(y_l_primary == p_l_primary), (y_l_ood == p_l_ood)]).astype(np.float64)
        all_conf = np.concatenate([conf_primary, conf_ood]).astype(np.float64)
    else:
        all_corr = (y_l_primary == p_l_primary).astype(np.float64)
        all_conf = np.asarray(conf_primary, dtype=np.float64)
    brier = float(np.mean((all_conf - all_corr) ** 2))
    c_score = 100.0 * max(0.0, 1.0 - brier)
    comp = 0.40 * fa + 0.15 * fb + 0.10 * fc + 0.10 * fd + 0.10 * fe + 0.05 * c_score + 0.10 * ff
    return {
        "Composite": round(comp, 3),
        "F_A_primary": round(fa, 2),
        "F_B_tech": round(fb, 2),
        "F_C_action": round(fc, 2),
        "F_D_ood": round(fd, 2),
        "F_E_adv": round(fe, 2),
        "F_F_multi_novel": round(ff, 2),
        "C_conf": round(c_score, 2),
        "Brier": round(brier, 5),
        "Acc_primary": round(100.0 * accuracy_score(y_l_primary, p_l_primary), 2),
    }

## 5. Guide-Aligned Rule Teacher & 58-Dim Feature Vector (`rules_v4` + Live Verification)

Implements strict Problem Statement Section 4 priority ordering (`1 > 2 > 4 > 3 > 5 > 0`), multi-clause negation-aware credential harvest checks, hard-legitimate guards (safe OTP warnings, COD cash notices, balance receipts, HR interview letters), and validates untuned accuracy directly on `sample.csv` (`86` labeled records).

In [6]:
import os
import re
import sys
from pathlib import Path
import numpy as np


# Strictly word-boundary-safe general linguistic & policy patterns (Problem Statement Sections 2-4)
SENS_RE = re.compile(
    r"\b(?:pin|otp|password|passcode|cvv|card number|card details|nid|verification code|login code|security code|"
    r"6[- ]?digit(?: code)?|\d[- ]digit code|date of birth|dob|internet banking password|wallet pin)\b"
    r"|ওটিপি|পিন|পাসওয়ার্ড|এনআইডি|কার্ড নম্বর",
    flags=re.IGNORECASE,
)
CODE_PLAIN_RE = re.compile(r"\bcode\b|কোড", flags=re.IGNORECASE)

ASK_SECRET_RE = re.compile(
    r"\b(?:send|sending|share|sharing|give|giving|tell|reply|reply with|provide|enter|entering|submit|forward|"
    r"confirm|verify|verify by|read out|bolen|bolo|bolun|pathan|pathaben|din|den|dao|deben|dibi|ta bolen|ta din|dite hobe|bolte hobe)\b"
    r"|জানান|পাঠান|দিন|বলুন|বলো|দেবেন|দাও|পাঠাও",
    flags=re.IGNORECASE,
)
SPEAKER_PREFIX_RE = re.compile(
    r"\[(?:sender|you|caller|agent)\]\s*:|(?:^|\|\|\|\s*)(?:sender|caller|you|s|r|a|b)\s*:",
    flags=re.IGNORECASE,
)

NEG_WARN_RE = re.compile(
    r"\b(?:only with (?:the )?rider|trip pin|do not share|don'?t share|never share|not share|never ask|never call|will never|"
    r"do not reply|don'?t reply|please do not reply|do not pay anyone|reply stop to opt out|"
    r"kauke\b.{0,18}\b(?:share|bolben|korben|diben|dien|bolo)\b.{0,6}\bna|"
    r"(?:bolben|dibi|dien|diben|korben)\s+na)\b"
    r"|কাউকে.{0,22}(?:দেবেন|করবেন|বলবেন|শেয়ার).{0,6}না|কখনো.{0,12}না",
    flags=re.IGNORECASE,
)

SCAM_WARN_THREAD_RE = re.compile(
    r"(?:someone|ekta sms|ekjon).{0,60}(?:messaged|ashche|pathaise).*?\|\|\|"
    r"|\b(?:eta scam|this is a scam|it'?s a scam|that is phishing|phishing|scam bhai|block and report|ignore korsi|scam hote pare|keu pin chaile)\b"
    r"|স্ক্যাম|প্রতারণা",
    flags=re.IGNORECASE,
)

MISTAKE_RE = re.compile(
    r"\b(?:accident(?:ally)?|by mistake|wrong transfer|wrong transaction|transfer.{0,20}wrong|sent.{0,20}wrong|"
    r"vul kore|bhul kore|vul e|bhul e|bhul hoye|vul hoye|wrong number|bhul number|vul number|mistaken|bhul transaction|vul transaction)\b"
    r"|ভুল করে|ভুলে|ভুল নম্বর",
    flags=re.IGNORECASE,
)
RETURN_RE = re.compile(
    r"\b(?:send (?:it )?back|return|ferot|refund|reverse it|pathiye din|flexiload|kindly return)\b"
    r"|ফেরত|পাঠিয়ে দিন",
    flags=re.IGNORECASE,
)

STRONG_PRIZE_RE = re.compile(
    r"\b(?:you won|you'?ve won|won|winner|winners|win a (?:tk|voucher|prize|gift|iphone|smartphone|car)|jitechen|jitechhen|jiteche|jitar|lucky draw|lottery|giveaway|airdrop|usdt|"
    r"mystery box|spin and win|free iphone|free smartphone|free router|free wi[- ]?fi router|10gb data gift|"
    r"pm education fund|stipend|old[- ]?age allowance|allowance arrears|bumper|"
    r"(?:number\s*ti|toyota\s*car|iphone\s*\d*|smart\s*tv|prize|bonus|gift|cashback|stipend|allowance).{0,35}select(?:ed)?\s*(?:hoyechen|hoyeche|for)|"
    r"loyalty reward|prize|cashback.{0,15}(?:jitechen|won|claim))\b"
    r"|জিতেছেন|জিতেছে|জিতার|লাকি ড্র|লটারি|পুরস্কার",
    flags=re.IGNORECASE,
)
SOFT_PRIZE_RE = re.compile(
    r"\b(?:cashback|gift|reward|rewards|bonus|eid bonus|congratulations|congrats)\b"
    r"|ক্যাশব্যাক|উপহার|অভিনন্দন|ঈদ বোনাস|গিফট",
    flags=re.IGNORECASE,
)
CLAIM_RE = re.compile(
    r"\b(?:claim|click|tap|link|fee|charge|(?<!pathao\s)pay|processing|survey|connect (?:your )?wallet|whatsapp|telegram|call|dial|install|download|reply|address|group|<URL_?\w*>|answer \d+ questions|submit your bkash)\b"
    r"|ক্লেইম|ক্লিক|লিংকে|লিংক|ফি|চার্জ|যোগাযোগ|যান|ইনস্টল|পাঠান|নিচের|দিন",
    flags=re.IGNORECASE,
)

PARCEL_RE = re.compile(
    r"\b(?:parcel|package|courier|customs|shipment|consignment|tracking|redelivery|storage fee|clearance charge|delivery attempt|out for delivery|was delivered|has been delivered)\b"
    r"|পার্সেল|কুরিয়ার|কাস্টমস|শুল্ক|ডেলিভারি হবে|ডেলিভারি ব্যর্থ|ডেলিভারি চার্জ",
    flags=re.IGNORECASE,
)
PARCEL_PHISH_CUE_RE = re.compile(
    r"\b(?:customs|duty|storage fee|redelivery|clearance charge|wrong address|incomplete|address confirmation|confirm(?: new)? address|"
    r"update address|address.{0,12}(?:vul|bhul)|(?:vul|bhul).{0,10}address|"
    r"could not be delivered|couldn'?t be delivered|attempt failed|release|"
    r"held|atke ase|failed|fee|charge|(?<!pathao\s)pay|click|tap|<URL_SUSP>|<URL_SHORT>|<URL>)\b"
    r"|আটকে|ব্যর্থ|চার্জ|ফি|পরিশোধ|শুল্ক|কাস্টমস|ঠিকানা আপডেট|ক্লিক|লিংকে",
    flags=re.IGNORECASE,
)

JOB_RE = re.compile(
    r"(?<!amar\s)(?<!my\s)\bjob\b(?!\s*jabe)|\b(?:hiring|vacancy|work[- ]?from[- ]?home|work frmo home|part[- ]?time|data entry|package handler|brand ambassador|"
    r"relief volunteer|typing job|agent license|earn(?:ing)?|income|kamai|kamao|salary|beton|task|tasks|watch and like|"
    r"like korle|facebook page|page like|per like|product review|review products|unlock tasks?|"
    r"commission|invest|investment|double your money|crypto signals?|arbitrage|bot|ludo|cash game|"
    r"freelanc(?:ing|e)|abroad job|trading crypto)\b"
    r"|লোক নিচ্ছে|ঘরে বসে|আয়|বেতন|বিনিয়োগ|দ্বিগুণ|ভাতা|হোয়াটসঅ্যাপে যোগ দিন",
    flags=re.IGNORECASE,
)
JOB_TRAP_RE = re.compile(
    r"\b(?:fee|deposit|security deposit|uniform fee|registration fee|clearance charge|service charge|visa fee|kit|"
    r"id card|invest|enroll now|per task|per post|per day|per like|like e \d+|per review|daily|weekly profit|profit per day|"
    r"guarantees?\s+\d+%|deposit tk \d+.{0,35}get tk \d+|10 minutes\.\s*guaranteed|commission|dite hobe|unlock korun|apply korun|"
    r"join (?:whatsapp|telegram|the group|vip)|group e join|whatsapp e join|join korun|whatsapp us|complete \d+ tasks?)\b"
    r"|রেজিস্ট্রেশন ফি|ডিপোজিট|দ্বিগুণ|নিশ্চিত লাভ|প্রতিদিন|যোগ দিন|যোগাযোগ করুন|সার্ভিস চার্জ|ভিসা ফি|টাকা দিন|হোয়াটসঅ্যাপে|টেলিগ্রামে",
    flags=re.IGNORECASE,
)

PROMO_RE = re.compile(
    r"\b(?:discount|sale|eid offer|winter sale|flash sale|11\.11|end of season|mega offer|1 ta kinle 1 ta free|"
    r"buy \d+ get \d+ free|\d+% (?:off|discount|cashback)|(?:tk|bdt|৳|\$)\s*\d+\s*off|off your first order|"
    r"0% emi|emi|chhar|voucher|vouchers up to|use code|code:\s*[a-z0-9]+|promo|shortto projojjo|t&c apply|"
    r"book your dream (?:flat|apartment|home)|easy installment|admission (?:going on|cholche)|learn english|combo|refer a friend|"
    r"special offer|\d+\s*gb\s*(?:for|at)\s*tk|"
    r"recharge tk \d+ and get|\d+ ?gb (?:extra|bonus)|new app update|exciting features|nearest outlet|limited time only)\b"
    r"|ছাড়|অফার|শর্ত প্রযোজ্য|রিচার্জে পান|বিল পেমেন্টে|আজই কিনুন|কিনুন",
    flags=re.IGNORECASE,
)

# High-precision Hard Legitimate patterns
HARD_LEGIT_COD_RE = re.compile(
    r"\b(?:cod tk [\d,]+ ready rakhun|keep tk [\d,]+ ready for cod|cash on delivery|collected in cash by our rider|"
    r"no online payment is needed|pay when it arrives)\b",
    flags=re.IGNORECASE,
)
HARD_LEGIT_OTP_RE = re.compile(
    r"(?:\b\d{4,8}\b.{0,45}\b(?:otp|code)\b|\b(?:otp|verification code|login code|confirmation code)\b.{0,35}\b\d{4,8}\b"
    r"|ওটিপি কোড.{0,20}\d{4,8})",
    flags=re.IGNORECASE,
)
HARD_LEGIT_TX_RE = re.compile(
    r"\b(?:cash[- ]?out.{0,25}successful|add money.{0,25}successful|mobile recharge.{0,25}successful|"
    r"payment.{0,20}successful|prepaid recharge.{0,20}successful|credited to a/?c|debited from a/?c|avl bal|"
    r"you have received tk|apni tk [\d,]+ peyechen|account e tk [\d,]+ add hoyeche|"
    r"has been credited|earned tk [\d,]+ cashback on your bill payment|current balance tk|"
    r"pathano hoyeche|trxid|bill.{0,25}(?:due|last date|shesh tarikh)|"
    r"(?:is this the .{0,20}agent.{0,40}cash\s*out)|send money to agent number|received,\s*here is your cash)\b"
    r"|ক্যাশ আউট সফল|পেয়েছেন|ট্রানজেকশন আইডি|অ্যাকাউন্টে.*জমা হয়েছে|ব্যালেন্স.*টাকা",
    flags=re.IGNORECASE,
)
HARD_LEGIT_HR_DOC_RE = re.compile(
    r"\b(?:offer letter attached|joining date:|application.{0,25}has been received|shortlisted candidates|"
    r"do not pay anyone for this process|interview schedule|bring cv and nid|cv ar nid copy niye ashben|"
    r"pay tuition via bank booth|official portal only)\b",
    flags=re.IGNORECASE,
)
HARD_LEGIT_DELIV_RE = re.compile(
    r"\b(?:out for delivery today|has been shipped|ship hoyeche|was delivered|has been delivered|arrived at .{0,20}hub|"
    r"collect from branch|will arrive|will retry tomorrow.{0,30}(?:call|reschedule)|096\d{2}-.{0,20}to reschedule)\b"
    r"|আজ ডেলিভারি হবে|আজকে ডেলিভারি হবে|ডেলিভারি হবে",
    flags=re.IGNORECASE,
)
HARD_LEGIT_CHAT_RE = re.compile(
    r"\b(?:forgot.{0,18}pin|how can i reset|choose\s*reset\s*pin|did you get my .{0,20}transfer|"
    r"dekh to ashche kina|bazar korish|p2p|dekha hobe|happy birthday|dinner tonight|"
    r"library: the book|dentist'?s office|meeting moved to|class kal|exam er routine|"
    r"study group|pack expires on)\b",
    flags=re.IGNORECASE,
)

# Entity Impersonation (strict T1: pretending to BE an entity/officer/helpline, not casual mention)
ENTITY_T1_RE = re.compile(
    r"(?:^(?:bkash|nagad|rocket|upay|surecash|mcash|tap|ok wallet|pathao|daraz|bdjobs|"
    r"sundarban|sa paribahan|paribahan|redx|steadfast|paperfly|ecourier|dhl|fedex|"
    r"gp|grameenphone|robi|banglalink|airtel|teletalk|skitto|jamuna|dbbl|brac|islami|city|prime)\b|"
    r"\b(?:courier notice|customs hold|security team|security department|fraud control|customer care|refund team|"
    r"this is (?:dhl|fedex|sundarban|pathao|redx|steadfast|bkash|nagad|rocket|upay)|"
    r"from (?:gp|grameenphone|robi|banglalink|airtel|daraz)|your (?:bkash|nagad|rocket|upay|dbbl|city bank|islami bank)|"
    r"helpline|officer|representative|unilever bangladesh|beximco|bdjobs|bangladesh bank|nbr|btrc|"
    r"desco|nesco|wasa|titas|breb|dpdc|western union|pm education|govt|government|"
    r"ami .{0,15}agent|dear (?:nagad|bkash|rocket|upay|gp|robi) (?:user|customer)|grameenphone lucky draw|"
    r"robi eid lucky draw|bkash anniversary|urgent bkash|urgent rocket|sim off)\b)"
    r"|বিকাশ|নগদ|রকেট|উপায়|গ্রামীণফোন|রবি|বাংলালিংক|দারাজ|পাঠাও|সুন্দরবন|সরকার|হেল্পলাইন|কর্মকর্তা",
    flags=re.IGNORECASE,
)

URGENCY_T5_RE = re.compile(
    r"\b(?:within \d+|24 ?h|12 ?h|1 hour|24 ghonta|expir(?:e|es|ed)|will be block|will be returned|will be deactivated|"
    r"band hobe|bondho|off hoye|suspend|last warning|final notice|flagged|immediately|taratari|hurry|"
    r"bacht|bachte|bipode|trouble|job jabe|avoid cancellation|ekhoni|today only|asap|na hole|claim within)\b"
    r"|এখনই|নইলে|আজই|বন্ধ হয়ে|বাতিল|তাড়াতাড়ি|সতর্ক|জরুরি|বিপদে|ঘণ্ট|মিনিট",
    flags=re.IGNORECASE,
)


def extract_rule_prediction(raw_text: str):
    """Inductive Guide-aligned classifier & 58-dim feature vector (24 base + 34 rule feats)."""
    clean = clean_for_model(raw_text)
    masked, uflags = mask_urls_for_text_rules(clean)
    # Strip speaker prefixes and decorative EMOJI tokens before semantic rule checks
    t = SPEAKER_PREFIX_RE.sub(" ", masked)
    t = re.sub(r"<EMOJI_\w+>", " ", t)

    scam_warn_thread = bool(SCAM_WARN_THREAD_RE.search(t))
    mistake = bool(MISTAKE_RE.search(t)) and bool(RETURN_RE.search(t)) and bool(
        re.search(r"\b(?:tk|bdt|taka|money)\b|টাকা|\d{2,}", t, re.I)
    )

    # Segment-level active credential harvest check
    req_secret = False
    for seg in re.split(r"[.!?\n\|]+", t):
        seg = seg.strip()
        if not seg:
            continue
        has_sens = bool(SENS_RE.search(seg)) or (
            bool(CODE_PLAIN_RE.search(seg))
            and not bool(re.search(r"\b(?:use code|code:|promo|voucher)\b", seg, re.I))
        )
        if has_sens and ASK_SECRET_RE.search(seg) and not NEG_WARN_RE.search(seg):
            if not HARD_LEGIT_HR_DOC_RE.search(seg):
                req_secret = True
                break
    if not req_secret:
        # Cross-clause explicit harvest (guarded against legit OTP/scam warnings without imperative ask)
        for seg in re.split(r"[.!?\n\|]+", t):
            seg = seg.strip()
            if seg and not NEG_WARN_RE.search(seg) and re.search(
                r"(?:nid.{0,15}both sides|photo of your nid|nid photo|card number.{0,20}cvv|cvv den|"
                r"enter(?:ing)? (?:your )?(?:pin|otp)|confirm(?: your)? (?:internet banking )?(?:password|otp|pin)|"
                r"send (?:your )?(?:pin|otp|nid|wallet pin)|tell me your pin|read out the code)",
                seg,
                re.I,
            ):
                req_secret = True
                break

    # Safe OTP warning vs Scam warning thread (never override unquoted active credential harvest)
    neg_warn = bool(NEG_WARN_RE.search(t))
    quoted_scam_framing = bool(
        ("|||" in clean and scam_warn_thread)
        or re.search(r"(?:someone|ekta sms|ekjon).{0,60}(?:messaged|ashche|pathaise|je)|that is phishing|eta scam|keu pin chaile", t, re.I)
    )
    is_scamwarn_only = bool(scam_warn_thread and (not req_secret or quoted_scam_framing))
    is_cod = bool(HARD_LEGIT_COD_RE.search(t)) and not req_secret
    is_hr_legit = (
        bool(HARD_LEGIT_HR_DOC_RE.search(t))
        and not req_secret
        and not bool(re.search(r"\b(?:pay tk|registration fee|uniform fee|deposit)\b", t, re.I))
    )
    is_tx_legit = (
        bool(HARD_LEGIT_TX_RE.search(t))
        and not req_secret
        and not mistake
        and not bool(re.search(r"\b(?:verify|claim|click|fee|charge|prosessing)\b|প্রসেসিং|ক্লেইম|<URL_SUSP>|<URL_SHORT>", t, re.I))
    )
    is_otp_legit = bool(HARD_LEGIT_OTP_RE.search(t)) and neg_warn and not req_secret
    is_deliv_legit = (
        bool(HARD_LEGIT_DELIV_RE.search(t))
        and not req_secret
        and not (
            PARCEL_PHISH_CUE_RE.search(t)
            and not bool(re.search(r"will retry tomorrow|096\d{2}-", t, re.I))
        )
    )
    is_chat_legit = bool(HARD_LEGIT_CHAT_RE.search(t)) and not req_secret and not mistake
    hard_legit = is_cod or is_hr_legit or is_tx_legit or is_otp_legit or is_deliv_legit or is_chat_legit or (clean == "<NOISE>")

    has_parcel = bool(PARCEL_RE.search(t))
    has_parcel_phish = bool(PARCEL_PHISH_CUE_RE.search(t))
    has_job = bool(JOB_RE.search(t))
    has_job_trap = bool(JOB_TRAP_RE.search(t))
    has_claim = bool(CLAIM_RE.search(t))
    has_promo = bool(PROMO_RE.search(t))

    # Strict Priority Prize vs Parcel/Job conflict resolution
    has_strong_prize = bool(STRONG_PRIZE_RE.search(t))
    has_soft_prize = bool(SOFT_PRIZE_RE.search(t)) and has_claim and not (has_parcel and has_parcel_phish) and not (has_job and has_job_trap)
    has_prize = has_strong_prize or has_soft_prize

    # Strict Section 4 Priority Order: 1 > 2 > 4 > 3 > 5 > 0
    if is_scamwarn_only:
        lbl = 0
    elif req_secret or mistake:
        lbl = 1
    elif hard_legit:
        lbl = 0
    elif has_promo and not (
        (has_parcel and has_parcel_phish)
        or (has_job and has_job_trap)
        or uflags["has_susp_tld"]
        or uflags["has_shortener"]
        or req_secret
        or has_strong_prize
        or re.search(r"\b(?:processing fee|survey|airdrop)\b|প্রসেসিং ফি", t, re.I)
    ):
        lbl = 5
    elif has_prize and (has_claim or uflags["has_url"]) and not (has_parcel and has_parcel_phish and (not re.search(r"\b(?:won|lottery|lucky draw)\b|লাকি ড্র|লটারি", t, re.I) or bool(re.search(r"limited winners", t, re.I)))) and not (has_job and has_job_trap):
        lbl = 2
    elif has_parcel and (has_parcel_phish or uflags["has_susp_tld"] or uflags["has_shortener"]) and not re.search(r"\b(?:package handler|salary|ambassador)\b", t, re.I):
        lbl = 4
    elif has_job and has_job_trap:
        lbl = 3
    elif has_promo:
        lbl = 5
    else:
        lbl = 0

    # Derive Rule-aligned Task C Action (1 > 2 > 4 > 3 > 5) and Task B Techniques
    if lbl in (0, 5):
        act = 0
        tvec = np.zeros(6, dtype=np.int32)
    else:
        req_pii_reply = bool(re.search(r"\b(?:reply|send|share|give)\b.{0,25}\b(?:name and address|full name|address|date of birth|dob)\b", t, re.I))
        has_pay = bool(re.search(
            r"\b(?:(?<!pathao\s)pay|deposit|invest|ferot|return|send (?:it )?back|reverse|enroll now with tk)\b"
            r"|\b(?:registration|processing|delivery|redelivery|storage|service|clearance)\s*(?:fee|charge|duty)\b.{0,22}\b(?:tk|bdt|\d|dite|(?<!pathao\s)pay|din|baki)"
            r"|\b(?:customs|storage|clearance)\s*duty\b.{0,25}\b(?:tk|bdt|\d|bkash)"
            r"|\b(?:tk|bdt|\$)\s*[\d,]+\s*(?:fee|duty|deposit|charge|ferot|dite hobe|bkash korun|nagad korun)\b"
            r"|পরিশোধ|ডিপোজিট|বিনিয়োগ|ফেরত|(?:টাকা|ফি|চার্জ)\s*(?:দিন|দিতে হবে)",
            t,
            re.I,
        ))
        # Distinguish "visa fee Tk 80,000 ... Call 01..." (CTA=Call) from "Pay Tk 3,500 fee" (CTA=Pay)
        has_pay_verb = bool(re.search(r"\b(?:(?<!pathao\s)pay|deposit|invest|enroll now with|ferot|send (?:it )?back|reverse|bkash korun|nagad korun)\b|পরিশোধ|ডিপোজিট|বিনিয়োগ|ফেরত|দিতে হবে|টাকা দিন", t, re.I))
        has_join_apk = bool(re.search(
            r"\b(?:install|download|join (?:the )?(?:group|vip|whatsapp|telegram)|whatsapp e join|telegram e join|group e join|join korun|"
            r"complete \d+ tasks?|watch and like|buy products|review products|rewards app|relief volunteer)\b|<URL_APK>|<URL_TGWA>"
            r"|যোগ দিন|হোয়াটসঅ্যাপে যোগ|টেলিগ্রামে যোগ|গ্রুপে",
            t,
            re.I,
        ))
        has_url_click = bool(uflags["has_url"] or re.search(r"\b(?:click|tap|link|visit|survey|connect (?:your )?wallet)\b|ক্লিক|লিংকে|যান", t, re.I))
        has_call_wa = bool(re.search(r"\b(?:call|whatsapp|contact|dial)\b|যোগাযোগ|কল করুন", t, re.I))

        if lbl == 1:
            act = 1 if (req_secret or not mistake) else 2
        elif req_pii_reply and not has_pay_verb:
            act = 1
        elif has_call_wa and not has_pay_verb and not has_join_apk and not has_url_click:
            act = 5
        elif has_pay:
            act = 2
        elif has_join_apk:
            act = 4
        elif has_url_click and not (has_call_wa and not uflags["has_url"]):
            act = 3
        elif has_call_wa:
            act = 5
        else:
            act = {2: 3, 3: 4, 4: 3}[lbl]

        tvec = np.zeros(6, dtype=np.int32)
        if ENTITY_T1_RE.search(t) and not (lbl == 3 and not re.search(r"\b(?:license|ambassador|typing job|data entry|officer|hr|unilever|bdjobs|beximco|walton|rokomari|pickaboo)\b", t, re.I)):
            tvec[0] = 1
        if lbl == 1 and (req_secret or not mistake):
            tvec[1] = 1
        if (act == 2) or (lbl != 1 and has_pay) or (lbl == 3 and re.search(r"\b(?:invest|profit|salary|fee|deposit)\b", t, re.I)):
            tvec[2] = 1
        if lbl == 2 or has_prize:
            tvec[3] = 1
        if URGENCY_T5_RE.search(t) or mistake:
            tvec[4] = 1
        if lbl == 3 or has_job:
            tvec[5] = 1
        if tvec.sum() == 0:
            tvec[{1: 1, 2: 3, 3: 5, 4: 2}[lbl]] = 1

    rule_feats = [
        float(lbl == 0), float(lbl == 1), float(lbl == 2), float(lbl == 3), float(lbl == 4), float(lbl == 5),
        float(req_secret), float(mistake), float(scam_warn_thread), float(hard_legit),
        float(is_cod), float(is_hr_legit), float(is_tx_legit), float(is_otp_legit),
        float(has_prize), float(has_claim), float(has_parcel and has_parcel_phish),
        float(has_job and has_job_trap), float(has_promo),
        float(tvec[0]), float(tvec[1]), float(tvec[2]), float(tvec[3]), float(tvec[4]), float(tvec[5]),
        float(act == 1), float(act == 2), float(act == 3), float(act == 4), float(act == 5),
        float(is_deliv_legit), float(is_chat_legit), float(neg_warn), float(lbl != 0 and lbl != 5),
    ]
    feats = np.array(extract_base_struct_v4(raw_text, clean) + rule_feats, dtype=np.float32)
    return lbl, tvec, act, feats

In [7]:
# Live validation of Guide-aligned rules_v4 on labeled sample.csv (86 rows)
r_pl, r_pt, r_pa = [], [], []
for _, row in sample_df.iterrows():
    l, tv, a, _ = extract_rule_prediction(row["text"])
    r_pl.append(l); r_pt.append(tv); r_pa.append(a)
r_pl, r_pt, r_pa = np.array(r_pl), np.vstack(r_pt), np.array(r_pa)
y_st = np.vstack([tech_str_to_vec(x) for x in sample_df["techniques"]])
res_r4 = score_section10_proxy(
    sample_df["label"].values, r_pl, y_st, r_pt, sample_df["action"].values, r_pa,
    np.where(r_pl == sample_df["label"].values, 0.95, 0.25),
)
print("rules_v4 on sample.csv:", res_r4)

rules_v4 on sample.csv:

 {'Composite': 99.458, 'F_A_primary': 100.0, 'F_B_tech': 96.47, 'F_C_action': 100.0, 'F_D_ood': 100.0, 'F_E_adv': 100.0, 'F_F_multi_novel': 100.0, 'C_conf': 99.75, 'Brier': 0.0025, 'Acc_primary': 100.0}


## 6. Programmatic Synthetic Generator (`72,000` Rows Across `48` Families) & Caching (`synth_generator_v4`, `prepare_v4_cache`)

Generates balanced multilingual/code-mixed single & multi-turn conversations with realistic adversarial noise/homoglyph corruption (`corrupt_prob=0.68`, `seed=4242`) and hashes `test.csv` into structural clusters (`test_skel_clusters.npz`).

In [8]:
import random
import re
import sys
from pathlib import Path
import numpy as np
import pandas as pd


MFS_BRANDS = ["bKash", "Nagad", "Rocket", "Upay", "SureCash", "mCash", "Tap", "OK Wallet", "Pathao Pay"]
BANKS = ["DBBL", "Dutch-Bangla Bank", "Islami Bank", "BRAC Bank", "City Bank", "Prime Bank", "Pubali Bank", "Jamuna Bank", "Standard Chartered", "UCB", "EBL", "Trust Bank"]
COURIERS = ["Sundarban Courier", "SA Paribahan", "Pathao Courier", "RedX", "Steadfast", "Paperfly", "Karatoa", "eCourier", "DHL", "FedEx", "Janani Courier", "Sheba Delivery"]
TELECOMS = ["GP", "Grameenphone", "Robi", "Banglalink", "Airtel", "Teletalk", "Skitto"]
ECOMMS = ["Daraz", "Rokomari", "Ajkerdeal", "Star Tech", "Aarong", "Shwapno", "Chaldal", "Pickaboo"]
EMPLOYERS = ["Unilever Bangladesh", "Beximco", "Walton", "Square Group", "PRAN-RFL", "Apex", "Robi Axiata", "bKash Limited", "Daraz"]
GOV_AUTHORITIES = ["Bangladesh Bank", "NBR Tax Office", "NID Wing", "BTRC", "Customs House", "Passport Office", "PM Education Fund", "Social Welfare Office"]
UTILITIES = ["DESCO", "NESCO", "WASA", "Titas Gas", "BTCL", "DPDC", "BREB"]
SCAM_TLDS = ["top", "xyz", "site", "online", "vip", "link", "click", "info", "co", "cc", "icu", "live"]
SHORTENERS = ["bit.ly", "tinyurl.com", "cutt.ly", "rb.gy"]

SPEAKER_STYLES = [
    ("S: ", "R: "),
    ("A: ", "B: "),
    ("[Sender]: ", "[You]: "),
    ("[sender]: ", "[you]: "),
    ("Caller: ", "You: "),
    ("caller: ", "you: "),
    ("", ""),
]

INNOCENT_WRAPPERS = [
    ("We hope this message finds you well. ", ""),
    ("Subject: Account & service update. Dear Valued Customer, ", " Thanks, Customer Service"),
    ("As part of our regular communication, please read carefully. ", " Thank you for your cooperation."),
    ("Subject: Message from {brand}. Dear Customer, ", " Reply STOP to opt out."),
]


def _rand_phone(rng):
    op = rng.choice(["013", "014", "015", "016", "017", "018", "019"])
    p = f"{op}XXXXXX{rng.randint(10,99)}"
    return f"+880 {p}" if rng.random() < 0.35 else p


def _rand_code(rng):
    return "".join(rng.choices("0123456789ABCDEFGHJKLMNPQRSTUVWXYZ", k=rng.randint(6, 10)))


def _rand_scam_url(rng, brand=None):
    if rng.random() < 0.16:
        return f"https://{rng.choice(SHORTENERS)}/{''.join(rng.choices('abcdefghijklmnopqrstuvwxyz0123456789', k=6))}"
    b = (brand or rng.choice(MFS_BRANDS + COURIERS + TELECOMS + ECOMMS)).lower().replace(" ", "")
    tag = rng.choice(["verify", "kyc", "claim", "track", "fee", "help", "support", "login", "secure", "reward", "bonus"])
    num = str(rng.randint(1, 99)) if rng.random() < 0.5 else ""
    tld = rng.choice(SCAM_TLDS)
    path = rng.choice(["/pay", "/verify", "/claim", "/login", "/" + "".join(rng.choices("abcdefghijklmnopqrstuvwxyz0123456789", k=5)), ""])
    return f"https://{b}-{tag}{num}.{tld}{path}"


def _rand_official_url(rng, brand):
    b = brand.lower().replace(" ", "").replace("-", "")
    tld = rng.choice(["com", "com.bd", "org.bd", "net"])
    path = rng.choice(["/track", "/offers", "/pay", "/app", "/careers", ""])
    return f"https://{b}.{tld}{path}"


def wrap_multiturn_v4(payload_msgs, rng: random.Random, allow_group=False, min_intro=1, max_intro=4):
    if allow_group and rng.random() < 0.25:
        names = ["James", "Nusrat", "Rahim", "Michael", "Karim", _rand_phone(rng)]
        n1, n2, n3 = rng.sample(names, 3)
        turns = [
            f"[Group] {n1}: {rng.choice(['anyone has the slides?', 'hi everyone', 'good evening', 'did you check the email?'])}",
            f"[Group] {n2}: {rng.choice(['i will share tonight', 'check the thread', 'just saw it', 'wait a second'])}",
        ]
        for m in payload_msgs:
            turns.append(f"[Group] {n3}: {m}")
        return " ||| ".join(turns)

    s_pfx, r_pfx = rng.choice(SPEAKER_STYLES)
    turns = []
    if rng.random() < 0.65:
        intros = [
            ("Hello", "Hi, who is this?"),
            ("Good morning sir", "Good morning, who's this?"),
            ("Brother, ekta help lagbe", "ki help?"),
            ("Hi, I got your number from a friend", "Which friend?"),
            ("Assalamu alaikum", "Walaikum assalam, ke bolchen?"),
            ("Apu, ekta kotha chilo", "bolen"),
            (f"Sir, apni ki {rng.choice(MFS_BRANDS)} use koren?", "ha, keno?"),
            ("Hello dear, found your number on Facebook", "who are you?"),
        ]
        for q, a in rng.sample(intros, rng.randint(min_intro, min(max_intro, len(intros)))):
            turns.append(f"{s_pfx}{q}")
            turns.append(f"{r_pfx}{a}")
    for i, m in enumerate(payload_msgs):
        turns.append(f"{s_pfx}{m}")
        if i < len(payload_msgs) - 1 and rng.random() < 0.70:
            turns.append(f"{r_pfx}{rng.choice(['ki?', 'kivabe?', 'kon company?', 'kothay?', 'thik ase, bolen', 'ok', 'ami to kichu bujhtesi na'])}")
    if rng.random() < 0.30:
        turns.append(f"{r_pfx}{rng.choice(['ami ekhon busy', 'really?', 'sure', 'ok let me check', 'seat kom!'])}")
    return " ||| ".join(turns)


def corrupt_text_v4(text: str, rng: random.Random) -> str:
    NOISE_WORDS = ["zzqwpo", "wkrjfejejrn", "mnbvcx", "qzxvbn", "xkcdpl", "asdkjh", "qundvpyfjqx", "wbmtzmh", "lkjsdfoiu"]
    EMOJIS = ["💰", "🎁", "🙏", "⚠️", "📲", "🔥", "😊"]
    CYR = {"a": "\u0430", "e": "\u0435", "o": "\u043e", "i": "\u0456", "c": "\u0441", "p": "\u0440", "y": "\u0443", "x": "\u0445"}
    LEET = {"a": "4", "e": "3", "i": "1", "o": "0", "s": "5", "t": "7"}

    out = text
    r_url = rng.random()
    if r_url < 0.45 and "http" in out.lower():
        if r_url < 0.25:
            out = re.sub(r"https?://", lambda m: "hxxps://" if "s" in m.group(0).lower() else "hxxp://", out, flags=re.I)
            out = out.replace(".", "[.]")
        else:
            out = re.sub(r"https?://", lambda m: "h77p5://" if "s" in m.group(0).lower() else "h77p://", out, flags=re.I)
            out = out.replace(".", rng.choice(["[.]", "[]"]))

    words = out.split(" ")
    new_words = []
    for w in words:
        if w == "|||" or "http" in w.lower() or "hxxp" in w.lower() or "h77p" in w.lower():
            new_words.append(w)
            continue
        if rng.random() < 0.07:
            new_words.append(rng.choice(NOISE_WORDS))
        if w.isascii() and len(w) >= 3 and rng.random() < 0.28:
            chars = list(w)
            op = rng.random()
            if op < 0.24:
                pos = rng.randint(1, len(chars) - 1)
                chars.insert(pos, rng.choice(["\u200b", "\u200c"]))
            elif op < 0.48:
                for j, c in enumerate(chars):
                    if c.lower() in CYR and rng.random() < 0.35:
                        chars[j] = CYR[c.lower()]
            elif op < 0.68:
                for j, c in enumerate(chars):
                    if c.lower() in LEET and rng.random() < 0.30:
                        chars[j] = LEET[c.lower()]
            elif op < 0.85:
                j = rng.randint(0, len(chars) - 1)
                if chars[j].isalpha():
                    chars[j] = chars[j] * rng.randint(3, 4)
            elif len(chars) >= 4:
                j = rng.randint(0, len(chars) - 2)
                chars[j], chars[j + 1] = chars[j + 1], chars[j]
            w = "".join(chars)
        elif re.search(r"[\u0980-\u09FF]", w) and rng.random() < 0.14 and len(w) >= 2:
            pos = rng.randint(1, len(w) - 1)
            w = w[:pos] + "\u200c" + w[pos:]
        new_words.append(w)
        if rng.random() < 0.05:
            new_words.append(rng.choice(EMOJIS))

    res = " ".join(new_words)
    if rng.random() < 0.22:
        res = re.sub(r" ([a-z]{2,5}) ", lambda m: m.group(1) + "", res, count=1)
    rc = rng.random()
    if rc < 0.07:
        res = res.upper()
    elif rc < 0.22:
        res = res.lower()
    return res


def _wrap_innocent(txt, brand, rng, allow_rate=0.14):
    if "|||" in txt or rng.random() >= allow_rate:
        return txt
    pfx, sfx = rng.choice(INNOCENT_WRAPPERS)
    return f"{pfx.format(brand=brand)}{txt}{sfx}"


def gen_one_v4(rng: random.Random):
    """Generate one diverse sample with (text, label, tech, action, scenario_family)."""
    r = rng.random()
    if r < 0.31:
        lbl = 0
    elif r < 0.49:
        lbl = 1
    elif r < 0.63:
        lbl = 2
    elif r < 0.77:
        lbl = 3
    elif r < 0.89:
        lbl = 4
    else:
        lbl = 5

    mfs = rng.choice(MFS_BRANDS)
    bank = rng.choice(BANKS)
    cour = rng.choice(COURIERS)
    tel = rng.choice(TELECOMS)
    ecom = rng.choice(ECOMMS)
    emp = rng.choice(EMPLOYERS)
    util = rng.choice(UTILITIES)
    amt = rng.choice([100, 250, 450, 500, 1200, 1800, 2500, 4500, 10000, 25000])
    bal = amt + rng.randint(50, 50000)
    otp = rng.randint(100000, 999999)

    if lbl == 0:
        sub = rng.random()
        if sub < 0.18:
            fam = "legit_otp"
            txt = rng.choice([
                f"{mfs} OTP: {otp}. Kauke share korben na. {rng.randint(2,5)} minute valid.",
                f"Your {mfs} verification code is {otp}. {mfs} will never call you to ask for it.",
                f"{otp} is your {bank} login code. Do not share it with anyone, even {bank} staff.",
                f"{otp} ashe apnar {mfs} login code. Eta kauke bolben na. Meyad {rng.randint(2,5)} minute.",
                f"{bank} OTP {otp} for Tk {amt:,} payment to {ecom}. Never share this OTP with anyone.",
                f"আপনার {mfs} ওটিপি কোড {otp}। এটি কখনো কারো সাথে শেয়ার করবেন না।",
                f"Security alert: new device logged in to your {mfs} account. If this was you, ignore. We never ask for your PIN or OTP.",
            ])
        elif sub < 0.44:
            if rng.random() < 0.45:
                fam = "legit_agent_cashout"
                s_p, r_p = rng.choice(SPEAKER_STYLES[:5])
                txt = (
                    f"{s_p}Hello, is this the {mfs} agent? ||| {r_p}yes sir ||| "
                    f"{s_p}I want cash out {amt:,} ||| {r_p}ok send money to agent number {_rand_phone(rng)} ||| "
                    f"{s_p}sent ||| {r_p}received, here is your cash"
                )
            else:
                fam = "legit_tx_bill"
                txt = rng.choice([
                    f"{mfs}: Cash-out Tk {amt:,} successful. Fee Tk 0.00. Balance Tk {bal:,}. TrxID {_rand_code(rng)}.",
                    f"{mfs}: You have received Tk {amt:,} from {_rand_phone(rng)}. Balance Tk {bal:,}. TrxID {_rand_code(rng)}.",
                    f"{mfs}: Apni Tk {amt:,} peyechen {_rand_phone(rng)} theke. Current balance Tk {bal:,}.",
                    f"{mfs}: Account e Tk {amt:,} add hoyeche {_rand_phone(rng)} theke. TrxID {_rand_code(rng)}.",
                    f"{mfs}: আপনার {amt:,} টাকা ক্যাশ আউট সফল হয়েছে। বর্তমান ব্যালেন্স {bal:,} টাকা। ট্রানজেকশন আইডি {_rand_code(rng)}।",
                    f"{bank}: BDT {amt:,} debited from A/C XXXX{rng.randint(1000,9999)} at {ecom}. Avl bal BDT {bal:,}.",
                    f"{util}: Your bill for November is Tk {amt:,}. Last date {rng.randint(1,28)} Dec. Pay via app or {_rand_official_url(rng, util)}.",
                    f"{util}: Prepaid recharge Tk {amt:,} successful. Token: {rng.randint(1000,9999)}-{rng.randint(1000,9999)}-{rng.randint(1000,9999)}.",
                    f"Earned Tk {rng.choice([15, 25, 50])} cashback on your {util} bill payment via {mfs}. TrxID {_rand_code(rng)}.",
                    f"Your {ecom} refund of Tk {amt:,} has been credited to your {mfs} account.",
                    f"Dear customer, your {tel} {rng.choice([5,10,20])}GB pack expires on {rng.randint(1,28)} Nov. Dial *123*1# to renew.",
                ])
        elif sub < 0.60:
            fam = rng.choice(["legit_hr_doc", "legit_cod", "legit_tuition"])
            pid = f"{rng.choice(['SC','PF','RX','DZ','SP'])}{rng.randint(10000,99999)}"
            if fam == "legit_cod":
                txt = rng.choice([
                    f"{cour}: Your parcel {pid} is out for delivery today. COD Tk {amt:,} ready rakhun.",
                    f"{ecom}: Order #{pid} out for delivery. Keep Tk {amt:,} ready for COD. No online payment is needed.",
                    f"{cour}: Delivery today between 2-6 PM. Payment will be collected in cash by our rider.",
                    f"R: Is my order #{pid} here? ||| S: Rider called, will deliver at {rng.randint(3,8)} PM ||| R: ok I'll pay cash on delivery",
                    f"{cour}: Parcel {pid} arrived at Dhaka hub. Collect from branch with this SMS and a photo ID.",
                    f"{cour}: আপনার পার্সেল {pid} আজ ডেলিভারি হবে।",
                    f"Pathao: Your rider is arriving in {rng.randint(2,10)} min. Share your trip PIN only with the rider.",
                    f"{cour}: Delivery attempt unsuccessful. We will retry tomorrow. Call 09611-XXXXXX to reschedule.",
                ])
            elif fam == "legit_hr_doc":
                txt = rng.choice([
                    f"Offer letter attached. Joining date: 1 Jan. Salary BDT {rng.choice([25000,30000,45000,60000,80000]):,}. Please sign and return. - HR, {emp}",
                    f"Dear Applicant, your application for Engineer at {emp} has been received. Shortlisted candidates will be contacted via official email. Do not pay anyone for this process.",
                    f"Interview schedule: Tuesday 10 AM, {emp} office. Bring CV and NID copy. Reply YES to confirm. - HR, {emp}",
                    f"{emp} HR: Apnar interview kal 11 tay. CV ar NID copy niye ashben.",
                ])
            else:
                txt = f"NSU Accounts: Pay tuition for Spring via bank booth or official portal only ({_rand_official_url(rng, 'nsu')}). Do not pay any third party."
        elif sub < 0.72:
            fam = "legit_scam_warn"
            scam_q = rng.choice([
                "your SIM registration is incomplete. Share the 6-digit OTP you just received",
                "apnar bkash PIN din account block hoye jabe",
                "dear winner, you won a smart tv. pay tk 500 delivery charge",
                "apnar card flagged hoyeche, PIN ar CVV pathan",
            ])
            if rng.random() < 0.65:
                s_p, r_p = rng.choice(SPEAKER_STYLES[:5])
                warn = rng.choice([
                    "eta scam bhai, kauke PIN ba OTP dibi na, block kor",
                    "that is phishing. never share your PIN or OTP. block and report.",
                    "scam eta, kono taka ba code dibi na",
                    "ignore korsi, eto obvious scam",
                ])
                txt = f"{s_p}Someone from '{mfs} helpline' messaged me: '{scam_q}' ||| {r_p}{warn} ||| {s_p}thanks, blocking"
            else:
                txt = f"Sabdhan! Ekta SMS ashche je '{scam_q}'. Eta scam. Keu PIN ba OTP chaile dibi na."
        elif sub < 0.94:
            fam = "legit_chat_faq"
            s_p, r_p = rng.choice(SPEAKER_STYLES[:5])
            txt = rng.choice([
                f"{s_p}I forgot my {mfs} PIN. How can I reset it? ||| {r_p}Dial *247# and choose Reset PIN. Never share your PIN with anyone, including our staff.",
                f"Did you get my {mfs} transfer of Tk {amt:,}?",
                f"Ami ekhon {mfs.lower()} e {amt} taka pathaisi, dekh to ashche kina.",
                f"Kal {rng.randint(8,11)} tay dekha hobe, ok? ||| thik ase.",
                f"A: bhai, {amt} taka ferot dibi kobe? ||| B: sunday dibo, {mfs.lower()} e pathaya dibo ||| A: ok thik ase",
                f"Mama, ajke raate bari asbo. Bazar theke kichu lagbe?",
                f"NSU Library: The book you reserved is available. Please collect within 2 days.",
                f"Boss: Meeting moved to 3 PM. Please join the Google Meet link.",
                f"Happy birthday! Dinner tonight, my treat!",
            ])
        else:
            fam = "legit_noise"
            txt = " ".join("".join(rng.choices("bcdfghjklmnpqrstvwxzqaioeu", k=rng.randint(5, 13))) for _ in range(rng.randint(1, 3)))
            return txt, 0, "NONE", 0, fam
        txt = _wrap_innocent(txt, mfs, rng, allow_rate=0.18)
        return txt, 0, "NONE", 0, fam

    elif lbl == 1:
        # Priority #1 MFS / Credential Harvesting (no decoy leakage!)
        url = _rand_scam_url(rng, mfs)
        sub = rng.random()
        if sub < 0.42:
            fam = rng.choice(["mfs_alert_card", "mfs_account_block", "mfs_limit_aml", "mfs_breb_western"])
            t_set = {"T1", "T2"}
            if fam == "mfs_alert_card":
                t_set.add("T5")
                txt = rng.choice([
                    f"{mfs} Alert: your card ending {rng.randint(1000,9999)} has been flagged. Send your NID number and date of birth.",
                    f"{mfs} Alert: your card ending {rng.randint(1000,9999)} has been flagged. Share the 6-digit OTP you just received. Otherwise account will be suspended.",
                    f"{mfs} Helpline: Apnar account e unusual login detect hoyeche. Block theke bachte 6 digit code ta ei number e reply korun.",
                    f"Dear {mfs} customer, an unusual login was detected from another device. Enter your PIN at {url}",
                    f"{mfs} refund team: send card number, expiry and CVV to receive your Tk {amt:,} refund.",
                ])
            elif fam == "mfs_limit_aml":
                t_set.add("T5")
                txt = rng.choice([
                    f"{mfs} Security Team: your wallet limit upgrade has been approved. Confirm your password and OTP here: {url}",
                    f"{bank}: Your account is flagged under anti-money laundering review. Confirm your internet banking password at {url}",
                    f"SABDHAN! {mfs.upper()}: APNAR WALLET LIMIT BARANO HOBE. CARD NUMBER, EXPIRY AR CVV DEN. KAUKE BOLBEN NA.",
                    f"Dear {mfs} user, KYC update na korle account 24 ghontar moddhe band hobe. Update korte NID number o PIN pathan.",
                ])
            elif fam == "mfs_breb_western":
                t_set.add("T5")
                txt = rng.choice([
                    f"Western Union: Tk {amt:,} remittance from your brother abroad is ready. Verify by sending NID photo and {mfs} wallet PIN.",
                    f"BREB Meter notice: Your prepaid meter balance is critically low. Enter OTP at {url} to avoid disconnection.",
                    f"প্রিয় গ্রাহক, আপনার {mfs} অ্যাকাউন্ট বন্ধ হয়ে যাবে। এখনই আপনার পিন নম্বর ও ওটিপি পাঠান।",
                ])
            else:
                if rng.random() < 0.7:
                    t_set.add("T5")
                txt = rng.choice([
                    f"Apnar {mfs} account verify korte NID er front back photo ar OTP pathan, na hole SIM off hoye jabe.",
                    f"Your {mfs} PIN will expire today. Reply with your current PIN and a new PIN to continue service.",
                ])
            if rng.random() < 0.35:
                txt = wrap_multiturn_v4([txt], rng, allow_group=True)
            if rng.random() < 0.22:
                txt = txt + " Do not tell anyone about this."
            txt = _wrap_innocent(txt, mfs, rng, allow_rate=0.14)
            return txt, 1, vec_to_tech_str(np.array([int(c in t_set) for c in TECH_LIST])), 1, fam

        elif sub < 0.74:
            # Wrong-transfer (PIN theft act=1 vs ferot money act=2) + borrowed wallet
            if rng.random() < 0.55:
                fam = rng.choice(["wrong_tx_pin", "wrong_tx_readout", "borrowed_wallet_pin"])
                t_set = {"T2", "T5"}
                if fam == "wrong_tx_readout":
                    t_set.add("T1")
                    txt = rng.choice([
                        f"URGENT {mfs}: a wrong transaction of Tk {amt:,} was made from your account. Read out the code that was just sent to you.",
                        f"S: {mfs} agent bolchi, vul kore apnar number e Tk {amt:,} chole gese ||| S: account verify korte apnar PIN ar OTP ta bolen.",
                    ])
                elif fam == "borrowed_wallet_pin":
                    txt = rng.choice([
                        f"Mama ami bipode poreche, {mfs.lower()} e taka nai. Tomar PIN ta dao ami ektu send kore nei.",
                        f"Brother amar wallet limit ses, tomar {mfs} number ar PIN ta ektu diba please? Amar khub emergency.",
                    ])
                else:
                    t_set.add("T1")
                    txt = f"Ami {mfs} agent, bhul number-e Tk {amt} pathaisi. Ferot din, PIN ta bolen."
                if rng.random() < 0.35:
                    txt = wrap_multiturn_v4([txt], rng, allow_group=True)
                txt = _wrap_innocent(txt, mfs, rng, allow_rate=0.15)
                return txt, 1, vec_to_tech_str(np.array([int(c in t_set) for c in TECH_LIST])), 1, fam
            else:
                fam = "wrong_tx_return_money"
                txt = rng.choice([
                    f"Bhai ekta bhul hoye geche. Amar {amt} taka vul kore apnar number e chole gese. Please ferot den {_rand_phone(rng)} e. Ami onek bipode.",
                    f"Hi, I accidentally sent Tk {amt:,} to your number. Please send it back to {_rand_phone(rng)}.",
                    f"ভাই ভুল করে আপনার নম্বরে {amt:,} টাকা ফ্লেক্সিলোড হয়ে গেছে, দয়া করে {_rand_phone(rng)} নম্বরে ফেরত পাঠান।",
                    f"Mistaken transfer of Tk {amt:,} sent to your account. Kindly reverse it or return via {mfs} to {_rand_phone(rng)} immediately.",
                ])
                if rng.random() < 0.35:
                    txt = wrap_multiturn_v4([txt], rng, allow_group=True)
                txt = _wrap_innocent(txt, mfs, rng, allow_rate=0.22)  # Explicit fix for INNOCENT_WRAPPERS + wrong tx!
                return txt, 1, "T3|T5", 2, fam

        else:
            # Priority-1 cross-category traps (Scholarship/Prize/Parcel/Job asking PIN/OTP/CVV!)
            fam = rng.choice(["pri1_scholarship_pin", "pri1_parcel_pin", "pri1_prize_pin", "pri1_job_pin"])
            if fam == "pri1_scholarship_pin":
                t_set = {"T1", "T2", "T5"}
                txt = f"HSC Scholarship board: Apnar Tk {amt:,} scholarship approved. Dispurse korte {mfs} PIN o OTP ekhoni pathan."
            elif fam == "pri1_parcel_pin":
                t_set = {"T1", "T2"}
                txt = rng.choice([
                    f"A friend abroad sent you a gift worth 1800 pounds held at customs. To proceed send your {mfs} PIN and NID photo.",
                    f"{cour}: Apnar parcel customs e atke ase. Release korte PIN o OTP pathan.",
                ])
            elif fam == "pri1_prize_pin":
                t_set = {"T1", "T2", "T4"}
                txt = rng.choice([
                    f"Congratulations! Apni {mfs} draw te Tk {amt:,} jitechen. Claim korte apnar OTP share korun.",
                    f"{mfs} bonus: apni Tk {amt:,} bonus peyechen. Wallet e add korte *167# dial kore PIN din.",
                    f"BN National Prize draw: Apni 1 lakh taka jitechen. Claim korte apnar {mfs} OTP bolun.",
                ])
            else:
                t_set = {"T2", "T5", "T6"}
                txt = f"Ghor bose {amt} taka daily income! Register korte apnar {mfs} PIN pathan, seat ses!"
            txt = _wrap_innocent(txt, mfs, rng, allow_rate=0.14)
            return txt, 1, vec_to_tech_str(np.array([int(c in t_set) for c in TECH_LIST])), 1, fam

    elif lbl == 2:
        brand = rng.choice(MFS_BRANDS + TELECOMS + ECOMMS + GOV_AUTHORITIES)
        fee = rng.choice([50, 99, 150, 250, 300, 500])
        prize = rng.choice([f"Tk {amt:,} cashback", f"Tk {amt:,} prize", "iPhone 15", "Smart TV", "Toyota car", "free smartphone", "10GB data gift", "Free Wi-Fi router"])
        url = _rand_scam_url(rng, brand)
        act = rng.choices([3, 2, 5, 4, 1], weights=[0.32, 0.28, 0.20, 0.12, 0.08])[0]
        t_set = {"T4"}
        if act == 3:
            fam = rng.choice(["prize_link_en", "prize_link_bn", "prize_usdt_survey", "prize_mystery_box"])
            if fam == "prize_usdt_survey":
                txt = rng.choice([
                    f"USDT Airdrop: Claim 500 USDT reward! Connect your wallet here: {url}",
                    f"2-min customer survey: Complete to claim Tk {amt:,} gift voucher! Click {url}",
                ])
            elif fam == "prize_mystery_box":
                txt = f"{ecom} anniversary gift: won a Mystery Box worth Tk {amt:,}! Claim at {url}"
                t_set.add("T1")
            elif fam == "prize_link_bn":
                txt = f"অভিনন্দন! আপনি {amt:,} টাকা ক্যাশব্যাক জিতেছেন। পেতে নিচের লিংকে ক্লিক করুন। {url if rng.random()<0.6 else ''}"
                t_set.add("T5")
            else:
                txt = rng.choice([
                    f"Free {prize} from {brand} for you! Tap {url} now, offer expires in 1 hour.",
                    f"Apni {amt:,} taka jitechen. Claim korte link e click korun. {url}",
                ])
                t_set.update({"T1", "T5"})
        elif act == 2:
            fam = rng.choice(["prize_fee_en", "prize_router_allowance", "prize_fee_bn"])
            t_set.add("T3")
            if fam == "prize_router_allowance":
                txt = rng.choice([
                    f"Free Wi-Fi router for all {tel} fiber users! Pay only Tk {fee} delivery fee via {_rand_phone(rng)}.",
                    f"Dear citizen, you are eligible for old-age allowance arrears of Tk {amt:,}. Pay Tk {fee} service charge to release the amount.",
                    f"PM Education Fund: your child is selected for a Tk {amt:,} stipend. Submit your bKash number and pay a Tk {fee} verification fee.",
                    f"Apni {prize} er jonno select hoyechen. Pete Tk {fee} processing fee din. Taratari korun.",
                ])
                t_set.add("T1")
            elif fam == "prize_fee_bn":
                txt = f"অভিনন্দন! আপনি {prize} জিতেছেন। পেতে প্রসেসিং ফি {fee} টাকা দিন {_rand_phone(rng)}।"
            else:
                txt = f"You won {prize} from {brand} lucky draw! Pay Tk {fee} processing fee to claim via {_rand_phone(rng)}."
                t_set.add("T1")
        elif act == 5:
            # Explicit high coverage of Bangladeshi Prize Action=5 CTA ("যোগাযোগ করুন" / "WhatsApp korun" / "Call")!
            fam = "prize_call_wa_bn_en"
            t_set.add("T1")
            txt = rng.choice([
                f"আপনার নম্বরটি {brand} লাকি ড্রতে নির্বাচিত হয়েছে। পুরস্কার পেতে এই নম্বরে যোগাযোগ করুন {_rand_phone(rng)}।",
                f"অভিনন্দন! আপনি {amt:,} টাকা বাম্পার পুরস্কার জিতেছেন। সংগ্রহ করতে এখনই কল করুন {_rand_phone(rng)}।",
                f"Congratulations! Apnar number ti {brand} Eid lucky draw te select hoyeche. {prize} jitar jonno {_rand_phone(rng)} e WhatsApp korun.",
                f"Lucky draw winner! {brand} Bumper reward Tk {amt:,} jitechen! Call {_rand_phone(rng)} now to claim.",
            ])
        elif act == 4:
            fam = "prize_apk_rewards"
            t_set.add("T1")
            txt = rng.choice([
                f"{brand} anniversary: spin and win Tk {amt:,}! Download the spin app: {url}/app.apk",
                f"Install the {mfs} Rewards app and get Tk {amt:,} instantly: {url}",
            ])
        else:
            fam = "prize_pii_reply"
            txt = f"Lucky draw winner! Reply with your full name and address to receive a {prize}."
        if rng.random() < 0.28:
            txt = wrap_multiturn_v4([txt], rng, allow_group=True)
        txt = _wrap_innocent(txt, brand, rng, allow_rate=0.14)
        return txt, 2, vec_to_tech_str(np.array([int(c in t_set) for c in TECH_LIST])), act, fam

    elif lbl == 3:
        pay = rng.choice([500, 1500, 2000, 3000, 3500, 40000])
        dep = rng.choice([60, 200, 450, 700, 2000, 3500, 5000, 80000])
        t_set = {"T6"}
        act = rng.choices([4, 2, 5], weights=[0.42, 0.44, 0.14])[0]
        if act == 4:
            fam = rng.choice(["job_task_like", "job_crypto_arb", "job_volunteer_review"])
            if fam == "job_crypto_arb":
                t_set.add("T5")
                txt = rng.choice([
                    f"Crypto signals group - 300% weekly profit guaranteed! Join VIP: https://t.me/vip_crypto_bd",
                    f"AI arbitrage bot guarantees 25% profit per day. Join the group: {_rand_scam_url(rng, 'arb')}",
                    f"I earn 2,500 USD weekly trading crypto, I can teach you! Join Telegram: https://t.me/fx_bd_team",
                ])
            elif fam == "job_volunteer_review":
                txt = rng.choice([
                    f"{ecom} product review job: earn Tk {pay:,}/day! Join our WhatsApp group to start tasks.",
                    f"Relief volunteer recruitment: Tk {pay:,}/day allowance! Download volunteer form app: {_rand_scam_url(rng, 'relief')}/app.apk",
                ])
            else:
                txt = rng.choice([
                    f"Part-time job: like YouTube videos, earn Tk {pay} per task. Join Telegram: https://t.me/earn_bd_tasks",
                    f"Facebook page like korle protek like e 50 taka. Group e join korun.",
                    f"ঘরে বসে প্রতিদিন {pay} টাকা আয় করুন। আজই হোয়াটসঅ্যাপে যোগ দিন।",
                    f"Ghor bose din e {pay} taka income. WhatsApp e join korun.",
                    f"Task: buy products on {ecom} via our link, we refund + 20% commission. Complete 5 tasks, get Tk {pay:,}.",
                ])
        elif act == 2:
            fam = rng.choice(["job_bn_hiring_reg", "job_package_ambassador", "job_data_entry", "job_double_invest"])
            t_set.add("T3")
            if fam == "job_bn_hiring_reg":
                t_set.update({"T1", "T5"})
                txt = rng.choice([
                    f"{emp} এ লোক নিচ্ছে! বেতন {pay:,} টাকা। রেজিস্ট্রেশন ফি {dep} টাকা দিন {_rand_phone(rng)}।",
                    f"{emp} e lok niche! Beton {pay:,} taka. Registration fee {dep} taka dite hobe.",
                    f"{ecom} product review diye {pay:,} taka kamai korun! {dep} taka deposit diye task unlock korun. Aajkei apply korun.",
                ])
            elif fam == "job_package_ambassador":
                t_set.add("T1")
                txt = rng.choice([
                    f"{emp} hiring Package Handler: salary Tk {pay:,}/month. Pay Tk {dep} uniform fee to confirm.",
                    f"Brand Ambassador opening at {ecom}! Earn Tk {pay:,}/month. Order your starter kit deposit Tk {dep}.",
                ])
            elif fam == "job_double_invest":
                t_set.add("T5")
                txt = rng.choice([
                    f"Double your money in 7 days! Invest Tk {dep:,}, get Tk {dep*2:,} guaranteed. bKash {_rand_phone(rng)}",
                    f"Ludo cash game: deposit Tk {dep}, get Tk {dep*3} in 10 minutes. Guaranteed.",
                    f"৭ দিনে টাকা দ্বিগুণ! {dep:,} টাকা বিনিয়োগ করলে {dep*2:,} টাকা নিশ্চিত লাভ। বিকাশ করুন {_rand_phone(rng)}।",
                ])
            else:
                t_set.add("T1")
                txt = rng.choice([
                    f"Congratulations, you are selected for Data Entry Officer at {emp}. Pay Tk {dep:,} registration fee to confirm.",
                    f"Freelancing course: earn $500/month from home guaranteed. Enroll now with Tk {dep:,}.",
                    f"Hi, I saw your CV on Bdjobs. Work-from-home 40K/month ||| S: Deposit Tk {dep:,} as security first.",
                ])
        else:
            fam = "job_call_visa_bn"
            t_set.update({"T3", "T5"})
            txt = rng.choice([
                f"Abroad job in Malaysia, visa fee Tk {dep:,}. 100% guaranteed. Call {_rand_phone(rng)}",
                f"সরকারি/বিদেশি চাকরির সুযোগ! বেতন {pay:,} টাকা, বিস্তারিত জানতে কল করুন {_rand_phone(rng)}।",
            ])
        if rng.random() < 0.30 and "|||" not in txt:
            txt = wrap_multiturn_v4([txt], rng, allow_group=True)
        txt = _wrap_innocent(txt, emp, rng, allow_rate=0.14)
        return txt, 3, vec_to_tech_str(np.array([int(c in t_set) for c in TECH_LIST])), act, fam

    elif lbl == 4:
        pid = f"{rng.choice(['SP','CN','PF','SC','SF'])}{rng.randint(10000,99999)}"
        fee = rng.choice([45, 99, 120, 250, 1850, 2200])
        url = _rand_scam_url(rng, cour)
        t_set = set()
        act = rng.choices([2, 3, 5], weights=[0.46, 0.40, 0.14])[0]
        if act == 2:
            fam = rng.choice(["deliv_customs_duty", "deliv_storage_charge"])
            t_set.add("T3")
            if fam == "deliv_customs_duty":
                t_set.add("T1")
                txt = rng.choice([
                    f"{cour}: Parcel {pid} held at customs. Pay Tk {fee} duty/storage fee within 24h via bKash {_rand_phone(rng)} or {url}",
                    f"Customs hold: your parcel from {rng.choice(['China','UK','Malaysia','Dubai'])} needs Tk {fee:,} duty. Pay via bKash to release.",
                    f"Hello, this is {cour}. You have a package from Malaysia ||| R: I didn't order anything ||| Pay Tk {fee:,} customs today or it will be returned.",
                ])
                if "24h" in txt or "today" in txt:
                    t_set.add("T5")
            else:
                txt = rng.choice([
                    f"Parcel atke ase. Delivery fee dite link e jan. {url}",
                    f"আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ {fee} টাকা পরিশোধ করতে লিংকে ক্লিক করুন। {url}",
                    f"{cour}: Parcel {pid} couldnt be delivered. Pay Tk {fee} fee to release it: {url}",
                    f"Apnar package e customs duty baki. Customs duty Tk {fee:,} bkash korun {_rand_phone(rng)} e.",
                    f"Your package could not be delivered. Pay Tk {fee:,} clearance charge to {_rand_phone(rng)}. Hurry, limited winners only.",
                ])
                if cour in txt:
                    t_set.add("T1")
        elif act == 3:
            fam = rng.choice(["deliv_addr_url", "deliv_bn_url"])
            if fam == "deliv_bn_url":
                t_set.add("T1")
                txt = f"{cour}: ডেলিভারি ব্যর্থ হয়েছে। ঠিকানা আপডেট করতে {url} এ যান।"
            else:
                t_set.update({"T1", "T5"})
                txt = rng.choice([
                    f"Courier notice: package #{pid} requires address confirmation. Click {url} within 24h.",
                    f"{cour}: Your parcel could not be delivered. Confirm address at {url}",
                    f"Delivery attempt failed. Update address within 12 hours or parcel will be returned. {url}",
                ])
        else:
            fam = "deliv_call_phone"
            t_set.update({"T1", "T5"})
            txt = rng.choice([
                f"{cour}: Apnar parcel {pid} er address vul. Call korun {_rand_phone(rng)} ekhoni.",
                f"{cour}: আপনার পার্সেল ঠিকানা ভুল, কল করুন {_rand_phone(rng)}।",
            ])
        if rng.random() < 0.28 and "|||" not in txt:
            txt = wrap_multiturn_v4([txt], rng, allow_group=True)
        txt = _wrap_innocent(txt, cour, rng, allow_rate=0.14)
        return txt, 4, vec_to_tech_str(np.array([int(c in t_set) for c in TECH_LIST])), act, fam

    else:
        fam = rng.choice(["promo_sale", "promo_app_bogo", "promo_realestate_edu", "promo_telco"])
        brand = rng.choice(TELECOMS + ECOMMS + BANKS + MFS_BRANDS + ["Pizza Hut", "Star Tech", "Bashundhara Housing"])
        disc = rng.choice([10, 15, 20, 30, 40, 50])
        code = rng.choice(["EID10", "SAVE20", "WINTER30", "DEAL50", "WELCOME50"])
        if fam == "promo_sale":
            txt = rng.choice([
                f"{brand}: Eid offer! Sob item e {disc}% discount. Code: {code}. Shortto projojjo.",
                f"{brand} 11.11 sale starts tonight 12 AM. Vouchers up to Tk {amt:,}.",
                f"Get {disc}% cashback on your next top-up with code {code}.",
                f"Install our app and get Tk 50 off your first order. Code: WELCOME50",
                f"ঈদ উপলক্ষে সব পণ্যে {disc}% ছাড়! আজই কিনুন।",
                f"{brand}: End of season sale up to {disc}% off. Visit your nearest outlet. T&C apply.",
            ])
        elif fam == "promo_app_bogo":
            txt = rng.choice([
                f"{brand}: Buy 1 get 1 free this weekend! Visit your nearest outlet.",
                f"{brand}: 1 ta kinle 1 ta free! Code: {code}. Shortto projojjo.",
                f"{brand}: New app update with exciting features. Open the app to apply. Limited time only.",
                f"Pizza Hut combo: 2 large pizzas + drink for Tk 1,299 today only!",
            ])
        elif fam == "promo_realestate_edu":
            txt = rng.choice([
                f"Book your dream apartment in Bashundhara, easy installment. Call 09611-XXXXXX",
                f"Learn English in 30 days! Admission going on. Call now.",
            ])
        else:
            txt = rng.choice([
                f"{tel} Mega Offer! Buy 1GB get 1GB free. Dial *121*3#",
                f"{tel}: Special offer! 10GB for Tk 199, 7 days. Dial *123*199#",
            ])
        if rng.random() < 0.25 and "|||" not in txt:
            txt = wrap_multiturn_v4([txt], rng, allow_group=True)
        txt = _wrap_innocent(txt, brand, rng, allow_rate=0.14)
        return txt, 5, "NONE", 0, fam


def generate_dataset_v4(n_samples=60000, seed=4242, corrupt_prob=0.68):
    rng = random.Random(seed)
    rows = []
    for idx in range(n_samples):
        txt, lbl, tech, act, fam = gen_one_v4(rng)
        if rng.random() < corrupt_prob and fam != "legit_noise":
            txt = corrupt_text_v4(txt, rng)
        rows.append((1000000 + idx, txt, lbl, tech, act, fam))
    return pd.DataFrame(rows, columns=["id", "text", "label", "techniques", "action", "scenario_family"])

In [9]:
import hashlib
import os
import re
import sys
import time
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path
import numpy as np
import pandas as pd





def _canonical_skel(clean_t: str) -> str:
    s = re.sub(r"https?://\S+", " URL ", clean_t.lower())
    s = re.sub(r"(?:\+?880[\s-]?)?01\d[\dxX]{5,9}", " PHN ", s)
    s = re.sub(r"\d+", " N ", s)
    s = re.sub(r"[^a-z\u0980-\u09FF]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()[:120]


def _worker_chunk(texts):
    cleans, feats, lbls, tvecs, acts, skel_hashes = [], [], [], [], [], []
    for raw in texts:
        c = clean_for_model(raw)
        l, tv, a, f = extract_rule_prediction(raw)
        sk = hashlib.blake2b(_canonical_skel(c).encode("utf-8"), digest_size=8).digest()
        cleans.append(c)
        feats.append(f)
        lbls.append(l)
        tvecs.append(tv)
        acts.append(a)
        skel_hashes.append(np.frombuffer(sk, dtype=np.uint64)[0])
    return (
        cleans,
        np.vstack(feats).astype(np.float32),
        np.array(lbls, dtype=np.uint8),
        np.vstack(tvecs).astype(np.uint8),
        np.array(acts, dtype=np.uint8),
        np.array(skel_hashes, dtype=np.uint64),
    )


def parallel_featurize(texts, chunk_size=5000, workers=10):
    chunks = [texts[i:i + chunk_size] for i in range(0, len(texts), chunk_size)]
    all_c, all_f, all_l, all_t, all_a, all_s = [], [], [], [], [], []
    try:
        with ProcessPoolExecutor(max_workers=workers) as ex:
            for c, f, l, tv, a, s in ex.map(_worker_chunk, chunks):
                all_c.extend(c); all_f.append(f); all_l.append(l); all_t.append(tv); all_a.append(a); all_s.append(s)
    except Exception:
        for ch in chunks:
            c, f, l, tv, a, s = _worker_chunk(ch)
            all_c.extend(c); all_f.append(f); all_l.append(l); all_t.append(tv); all_a.append(a); all_s.append(s)
    return (
        all_c,
        np.vstack(all_f),
        np.concatenate(all_l),
        np.vstack(all_t),
        np.concatenate(all_a),
        np.concatenate(all_s),
    )


def run_prepare_cache():
    t0 = time.time()
    # 1. Sample.csv (86 rows)
    sdf = pd.read_csv(DATA_DIR / "sample.csv")
    sc, sf, sl, st, sa, ss = parallel_featurize(sdf["text"].tolist(), chunk_size=30, workers=2)
    sdf["clean_text"] = sc
    sdf["scenario_family"] = [assign_sample_scenario_family(sc[i], int(sdf["label"].iloc[i])) for i in range(len(sdf))]
    sdf.to_csv(CACHE_DIR / "sample_v4.csv", index=False)
    np.save(CACHE_DIR / "sample_v4_feats.npy", sf)
    np.savez_compressed(CACHE_DIR / "sample_v4_rule_pred.npz", lbl=sl, tvec=st, act=sa, skel=ss)
    print(f"Cached sample_v4 ({len(sdf)} rows, {sf.shape[1]} feats) in {time.time()-t0:.1f}s")

    # 2. Synth 72k v5
    t1 = time.time()
    sy = generate_dataset_v4(n_samples=72000, seed=4242, corrupt_prob=0.68)
    syc, syf, syl, syt, sya, sysk = parallel_featurize(sy["text"].tolist(), chunk_size=4000, workers=10)
    sy["clean_text"] = syc
    sy.to_csv(CACHE_DIR / "synth_64k_v4.csv", index=False)
    np.save(CACHE_DIR / "synth_64k_v4_feats.npy", syf)
    np.savez_compressed(CACHE_DIR / "synth_64k_v4_rule_pred.npz", lbl=syl, tvec=syt, act=sya, skel=sysk)
    print(f"Cached synth_64k_v4 ({len(sy)} rows, rule agree={(syl==sy['label'].values).mean():.4f}) in {time.time()-t1:.1f}s")

    # 3. Test 500k
    t2 = time.time()
    te = pd.read_csv(DATA_DIR / "test.csv")
    tec, tef, tel, tet, tea, tesk = parallel_featurize(te["text"].fillna("").astype(str).tolist(), chunk_size=10000, workers=10)
    pd.DataFrame({"id": te["id"].values, "clean_text": tec}).to_csv(CACHE_DIR / "test_v4_clean.csv", index=False)
    np.save(CACHE_DIR / "test_v4_feats.npy", tef)
    np.savez_compressed(CACHE_DIR / "test_v4_rule_pred.npz", lbl=tel, tvec=tet, act=tea, skel=tesk)

    # Build canonical cluster statistics for domain-adaptation prototype selection (training-only!)
    vals, first_idx, counts = np.unique(tesk, return_index=True, return_counts=True)
    order = np.argsort(-counts)
    vals_sorted = vals[order]
    counts_sorted = counts[order]
    first_sorted = first_idx[order]
    np.savez_compressed(
        CACHE_DIR / "test_skel_clusters.npz",
        uniq_hash=vals_sorted,
        first_idx=first_sorted,
        counts=counts_sorted,
    )
    n_ge10 = int((counts >= 10).sum())
    cov_ge10 = int(counts[counts >= 10].sum())
    print(
        f"Cached test_v4 (500k rows) in {time.time()-t2:.1f}s | "
        f"unique_skels={len(vals)}, clusters>=10: {n_ge10} covering {cov_ge10} ({100*cov_ge10/len(te):.1f}%)"
    )

In [10]:
# Quick programmatic verification of synthetic generator diversity (small deterministic sample)
sy_preview = generate_dataset_v4(n_samples=600, seed=4242, corrupt_prob=0.68)
print(f"Generated preview rows={len(sy_preview)} | distinct scenario families={sy_preview['scenario_family'].nunique()}")
print("Preview label distribution:", sy_preview["label"].value_counts().sort_index().to_dict())
print("3-fold scenario-disjoint split counts:", pd.Series(get_balanced_3folds(sy_preview["scenario_family"].values)).value_counts().sort_index().to_dict())

Generated preview rows=600 | distinct scenario families=48
Preview label distribution: {0: 170, 1: 109, 2: 78, 3: 99, 4: 81, 5: 63}
3-fold scenario-disjoint split counts: {0: 154, 1: 243, 2: 203}


## 7. Multi-Task Neural & Tabular Pipeline (`torch_utils_v4`, `TFIDF+GBDT`, `BiGRU-v5`, `Base Teachers`, `Domain Consensus`, `Students`, `DA-Tab`)

- **`HeadConditionedTransformer`**: Pretrained backbone (`mDeBERTa-v3-base`, `mE5-base`, `mMiniLM`, `mE5-small`, `BGE-small`) concatenated with LayerNorm-scaled `58`-dim structural/rule features, 4x multi-sample dropout, and detached Task-A probabilities conditioning Task B (`techniques`) and Task C (`action`).
- **`BPEBiGRUNet`**: 32k-bucket hashed subword 2-layer BiGRU + Attention/Max pooling for diverse sequence inductive bias.
- **`fit_gbdt_multitask`**: Multi-class `LightGBM + XGBoost + CatBoost` for Task A + per-technique/action LightGBM heads.

In [11]:
import gc
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel

LOG_COLS_V4 = (0, 1, 2, 16, 17, 18, 20)
LOG_DIVS_V4 = (6.0, 6.0, 3.0, 3.0, 3.0, 4.0, 4.0)


def scale_feats_v4(X: np.ndarray) -> np.ndarray:
    """Deterministic log-scale normalization of unbounded count features in 58-dim v4 array."""
    Xd = np.array(X, dtype=np.float32, copy=True)
    for col, div in zip(LOG_COLS_V4, LOG_DIVS_V4):
        Xd[:, col] = np.log1p(np.maximum(0.0, Xd[:, col])) / div
    return np.nan_to_num(Xd, nan=0.0, posinf=5.0, neginf=0.0)


class MultiTaskKDDataset(Dataset):
    """Dataset supporting hard labels and optional soft teacher probabilities for distillation."""

    def __init__(self, texts, feats, yl=None, yt=None, ya=None, soft_l=None, soft_t=None, soft_a=None, weights=None):
        self.texts = list(texts)
        self.feats = np.asarray(feats, dtype=np.float32)
        self.yl = None if yl is None else np.asarray(yl, dtype=np.int64)
        self.yt = None if yt is None else np.asarray(yt, dtype=np.float32)
        self.ya = None if ya is None else np.asarray(ya, dtype=np.int64)
        self.soft_l = None if soft_l is None else np.asarray(soft_l, dtype=np.float32)
        self.soft_t = None if soft_t is None else np.asarray(soft_t, dtype=np.float32)
        self.soft_a = None if soft_a is None else np.asarray(soft_a, dtype=np.float32)
        self.weights = None if weights is None else np.asarray(weights, dtype=np.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = [self.texts[idx], self.feats[idx]]
        if self.yl is not None:
            item.extend([int(self.yl[idx]), self.yt[idx], int(self.ya[idx])])
        if self.soft_l is not None:
            item.extend([self.soft_l[idx], self.soft_t[idx], self.soft_a[idx]])
        if self.weights is not None:
            item.append(float(self.weights[idx]))
        return tuple(item)


class BucketBatchSampler:
    """Length-bucketed sampler minimizing padding waste and avoiding WDDM VRAM spikes."""

    def __init__(self, texts, bs, seed=42, jitter=16.0):
        self.lens = np.array([len(str(t)) for t in texts], dtype=np.float32)
        self.bs = int(bs)
        self.seed = int(seed)
        self.jitter = float(jitter)
        self.epoch = 0

    def __iter__(self):
        rng = np.random.RandomState(self.seed + self.epoch)
        self.epoch += 1
        n = len(self.lens)
        order = np.argsort(self.lens + rng.uniform(0.0, self.jitter, size=n))
        batches = [order[i:i + self.bs].tolist() for i in range(0, n, self.bs)]
        rng.shuffle(batches)
        for b in batches:
            yield b

    def __len__(self):
        return (len(self.lens) + self.bs - 1) // self.bs


def make_collate_fn(tok, max_len=160, has_labels=True, has_soft=False, has_weights=False):
    def collate(batch):
        texts = [b[0] for b in batch]
        feats = torch.from_numpy(np.stack([b[1] for b in batch])).float()
        enc = tok(texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
        out = [enc["input_ids"], enc["attention_mask"], feats]
        pos = 2
        if has_labels:
            out.append(torch.tensor([b[pos] for b in batch], dtype=torch.long))
            out.append(torch.from_numpy(np.stack([b[pos + 1] for b in batch])).float())
            out.append(torch.tensor([b[pos + 2] for b in batch], dtype=torch.long))
            pos += 3
        if has_soft:
            out.append(torch.from_numpy(np.stack([b[pos] for b in batch])).float())
            out.append(torch.from_numpy(np.stack([b[pos + 1] for b in batch])).float())
            out.append(torch.from_numpy(np.stack([b[pos + 2] for b in batch])).float())
            pos += 3
        if has_weights:
            out.append(torch.tensor([b[pos] for b in batch], dtype=torch.float32))
        return tuple(out)

    return collate


class HeadConditionedTransformer(nn.Module):
    """Multi-task transformer with multi-sample dropout and head-conditioned Task B/C."""

    def __init__(self, model_name, n_feats=58, pool_mode="mean", proj_dim=256, num_freeze_layers=0):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name, cache_dir=CACHE_DIR_HF, local_files_only=HF_LOCAL_ONLY, dtype=torch.float32)
        self.pool_mode = pool_mode
        h = self.backbone.config.hidden_size
        pool_dim = 2 * h if pool_mode == "cls_mean" else h

        if num_freeze_layers > 0:
            self._freeze_bottom(num_freeze_layers)

        self.feat_norm = nn.LayerNorm(n_feats)
        self.fuse = nn.Sequential(
            nn.Linear(pool_dim + n_feats, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.GELU(),
        )
        self.dropouts = nn.ModuleList([nn.Dropout(0.1 + 0.04 * i) for i in range(4)])
        self.head_label = nn.Linear(proj_dim, 6)
        self.head_tech = nn.Linear(proj_dim + 6, 6)
        self.head_action = nn.Linear(proj_dim + 6, 6)

    def _freeze_bottom(self, k):
        for name, p in self.backbone.named_parameters():
            if any(f"layer.{i}." in name for i in range(k)) or "embeddings" in name:
                p.requires_grad = False

    def _pool(self, last_hidden, attention_mask):
        if self.pool_mode == "cls":
            return last_hidden[:, 0, :]
        mask = attention_mask.unsqueeze(-1).type_as(last_hidden)
        mean_p = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        if self.pool_mode == "cls_mean":
            return torch.cat([last_hidden[:, 0, :], mean_p], dim=-1)
        return mean_p

    def forward(self, input_ids, attention_mask, feats):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        pooled = self._pool(out, attention_mask)
        fn = self.feat_norm(feats)
        z = self.fuse(torch.cat([pooled, fn], dim=-1))

        logits_l = sum(self.head_label(d(z)) for d in self.dropouts) / len(self.dropouts)
        p_l_cond = F.softmax(logits_l.detach(), dim=-1)
        zc = torch.cat([z, p_l_cond], dim=-1)
        logits_t = sum(self.head_tech(d(zc)) for d in self.dropouts) / len(self.dropouts)
        logits_a = sum(self.head_action(d(zc)) for d in self.dropouts) / len(self.dropouts)
        return logits_l, logits_t, logits_a

    def param_groups(self, lr_backbone=2.5e-5, lr_head=1.5e-4, weight_decay=0.01):
        no_decay = ("bias", "LayerNorm.weight", "LayerNorm.bias")
        bb_d, bb_nd, hd_d, hd_nd = [], [], [], []
        for n, p in self.named_parameters():
            if not p.requires_grad:
                continue
            is_hd = not n.startswith("backbone.")
            is_nd = any(k in n for k in no_decay)
            if is_hd:
                (hd_nd if is_nd else hd_d).append(p)
            else:
                (bb_nd if is_nd else bb_d).append(p)
        return [
            {"params": bb_d, "lr": lr_backbone, "weight_decay": weight_decay},
            {"params": bb_nd, "lr": lr_backbone, "weight_decay": 0.0},
            {"params": hd_d, "lr": lr_head, "weight_decay": weight_decay},
            {"params": hd_nd, "lr": lr_head, "weight_decay": 0.0},
        ]


def compute_multitask_loss(
    logits_l, logits_t, logits_a,
    yl, yt, ya,
    label_weights=None, act_weights=None, sample_w=None,
    soft_l=None, soft_t=None, soft_a=None, kd_alpha=0.45, temp=2.0,
):
    sw = sample_w if sample_w is not None else torch.ones_like(yl, dtype=torch.float32)
    sw = sw / sw.mean().clamp(min=1e-6)

    ce_l = F.cross_entropy(logits_l, yl, weight=label_weights, reduction="none", label_smoothing=0.02)
    loss_l = (ce_l * sw).mean()

    scam_mask = (yl >= 1) & (yl <= 4)
    if scam_mask.any():
        bce_t = F.binary_cross_entropy_with_logits(logits_t[scam_mask], yt[scam_mask], reduction="none").mean(dim=-1)
        loss_t = (bce_t * sw[scam_mask]).mean()
    else:
        loss_t = logits_t.sum() * 0.0

    act_mask = scam_mask & (ya >= 1) & (ya <= 5)
    if act_mask.any():
        # Mask out logit 0 or index on classes 1..5 cleanly
        ya_sub = ya[act_mask] - 1
        aw_sub = act_weights[1:] if act_weights is not None else None
        ce_a = F.cross_entropy(logits_a[act_mask, 1:], ya_sub, weight=aw_sub, reduction="none", label_smoothing=0.02)
        loss_a = (ce_a * sw[act_mask]).mean()
    else:
        loss_a = logits_a.sum() * 0.0

    loss_hard = 1.0 * loss_l + 0.55 * loss_t + 0.65 * loss_a

    if soft_l is not None and kd_alpha > 0:
        log_p_l = F.log_softmax(logits_l / temp, dim=-1)
        kd_l = F.kl_div(log_p_l, soft_l.clamp(min=1e-6), reduction="none").sum(dim=-1) * (temp * temp)
        loss_kd = (kd_l * sw).mean()
        if scam_mask.any():
            bce_kt = F.binary_cross_entropy_with_logits(logits_t[scam_mask], soft_t[scam_mask].clamp(1e-4, 1 - 1e-4), reduction="none").mean(dim=-1)
            loss_kd = loss_kd + 0.50 * (bce_kt * sw[scam_mask]).mean()
        if act_mask.any():
            log_p_a = F.log_softmax(logits_a[act_mask, 1:] / temp, dim=-1)
            p_soft_a = soft_a[act_mask, 1:].clamp(min=1e-6)
            p_soft_a = p_soft_a / p_soft_a.sum(dim=-1, keepdim=True)
            kd_a = F.kl_div(log_p_a, p_soft_a, reduction="none").sum(dim=-1) * (temp * temp)
            loss_kd = loss_kd + 0.55 * (kd_a * sw[act_mask]).mean()
        return (1.0 - kd_alpha) * loss_hard + kd_alpha * loss_kd
    return loss_hard


@torch.no_grad()
def predict_transformer(model, tok, texts, feats, bs=64, max_len=160, device="cuda"):
    model.eval()
    n = len(texts)
    # Sort by length for zero padding waste, then unsort back to original order
    lens = np.array([len(str(t)) for t in texts])
    order = np.argsort(lens)
    inv_order = np.argsort(order)

    ordered_texts = [texts[i] for i in order]
    ordered_feats = feats[order]

    p_l = np.zeros((n, 6), dtype=np.float32)
    p_t = np.zeros((n, 6), dtype=np.float32)
    p_a = np.zeros((n, 6), dtype=np.float32)

    for s in range(0, n, bs):
        e = min(n, s + bs)
        enc = tok(ordered_texts[s:e], padding=True, truncation=True, max_length=max_len, return_tensors="pt")
        ids = enc["input_ids"].to(device, non_blocking=False)
        mask = enc["attention_mask"].to(device, non_blocking=False)
        f_t = torch.from_numpy(ordered_feats[s:e]).to(device, dtype=torch.float32)
        with torch.amp.autocast("cuda", enabled=(device == "cuda")):
            ll, lt, la = model(ids, mask, f_t)
            pl = F.softmax(ll.float(), dim=-1).cpu().numpy()
            pt = torch.sigmoid(lt.float()).cpu().numpy()
            pa_sub = F.softmax(la[:, 1:].float(), dim=-1).cpu().numpy()
        p_l[s:e] = pl
        p_t[s:e] = pt
        p_a[s:e, 1:] = pa_sub

    return p_l[inv_order], p_t[inv_order], p_a[inv_order]


def free_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [12]:
import gc
import os
import sys
import time
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
try:
    from catboost import CatBoostClassifier
    HAS_CB = True
except ImportError:
    HAS_CB = False
try:
    import torch
    XGB_DEV = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    XGB_DEV = "cpu"
from scipy.sparse import hstack, vstack, csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold
from sklearn.preprocessing import StandardScaler



# Map each sample.csv coarse scenario_family to ALL matching synth_64k_v4 families so LOSO CV has ZERO leakage
LOSO_SYNTH_EXCLUSIONS = {
    "mfs_mistake": {"wrong_tx_pin", "wrong_tx_readout", "wrong_tx_return_money"},
    "mfs_cred_harvest": {"mfs_alert_card", "mfs_account_block", "mfs_limit_aml", "mfs_breb_western", "borrowed_wallet_pin"},
    "pri1_cross_cat": {"pri1_scholarship_pin", "pri1_parcel_pin", "pri1_prize_pin", "pri1_job_pin"},
    "prize_cashback_lakh": {"prize_link_bn", "prize_link_en", "prize_fee_bn", "prize_fee_en", "prize_call_wa_bn_en"},
    "prize_gift_voucher": {"prize_usdt_survey", "prize_mystery_box", "prize_router_allowance", "prize_apk_rewards", "prize_pii_reply"},
    "job_recruitment_fee": {"job_data_entry", "job_bn_hiring_reg", "job_package_ambassador", "job_call_visa_bn"},
    "job_crypto_invest": {"job_crypto_arb", "job_double_invest"},
    "job_task_earn": {"job_task_like", "job_volunteer_review"},
    "deliv_customs_duty": {"deliv_customs_duty"},
    "deliv_addr_phish": {"deliv_addr_url", "deliv_bn_url", "deliv_call_phone"},
    "deliv_fee_link": {"deliv_storage_charge"},
    "promo_sale_disc": {"promo_sale", "promo_app_bogo"},
    "promo_offer_ad": {"promo_realestate_edu", "promo_telco"},
    "legit_scam_warn": {"legit_scam_warn"},
    "legit_otp_warn": {"legit_otp"},
    "legit_tx_bill": {"legit_tx_bill", "legit_agent_cashout"},
    "legit_deliv_ok": {"legit_cod"},
    "legit_chat_other": {"legit_chat_faq", "legit_hr_doc", "legit_tuition", "legit_noise"},
}

UNSEEN_HOLDOUT_FAMS = {
    "pri1_scholarship_pin", "wrong_tx_readout", "borrowed_wallet_pin",
    "prize_usdt_survey", "prize_router_allowance", "prize_call_wa_bn_en",
    "job_bn_hiring_reg", "job_package_ambassador",
    "deliv_addr_url", "legit_hr_doc", "legit_cod", "promo_app_bogo",
}


def make_turn_texts(clean_series):
    return [format_turn_aware(str(x)) for x in clean_series]


def fit_predict_linear_stack(X_tr, y_l, y_t, y_a, X_vals):
    """Fit fast calibrated OVR Linear models for Task A, B, C and predict on list of sparse validation/test matrices."""
    clf_l = LogisticRegression(C=2.5, max_iter=180, tol=1e-3, class_weight="balanced", solver="lbfgs")
    clf_l.fit(X_tr, y_l)

    clfs_t = []
    scam_mask = np.isin(y_l, [1, 2, 3, 4])
    for j in range(6):
        c = LogisticRegression(C=2.0, max_iter=150, tol=1e-3, class_weight="balanced")
        c.fit(X_tr[scam_mask], y_t[scam_mask, j])
        clfs_t.append(c)

    act_mask = np.isin(y_l, [1, 2, 3, 4]) & (y_a > 0)
    clf_a = LogisticRegression(C=2.0, max_iter=180, tol=1e-3, class_weight="balanced", solver="lbfgs")
    clf_a.fit(X_tr[act_mask], y_a[act_mask])

    outs = []
    for Xv in X_vals:
        pl = clf_l.predict_proba(Xv).astype(np.float32)
        pt = np.column_stack([c.predict_proba(Xv)[:, 1] for c in clfs_t]).astype(np.float32)
        pa_sub = clf_a.predict_proba(Xv).astype(np.float32)
        pa = np.zeros((Xv.shape[0], 6), dtype=np.float32)
        for col_idx, cls_id in enumerate(clf_a.classes_):
            pa[:, int(cls_id)] = pa_sub[:, col_idx]
        outs.append((pl, pt, pa))
    return outs, (clf_l, clfs_t, clf_a)


def fit_gbdt_multitask(X_tr, y_l, y_t, y_a, X_va=None, y_va_l=None, seed=42, fast=True):
    n_tr_mc = 260 if fast else 380
    lgb_l = lgb.LGBMClassifier(
        n_estimators=n_tr_mc, learning_rate=0.06, num_leaves=47, subsample_for_bin=30000,
        colsample_bytree=0.75, subsample=0.85, class_weight="balanced",
        random_state=seed, n_jobs=8, verbose=-1,
    )
    if X_va is not None:
        lgb_l.fit(X_tr, y_l, eval_set=[(X_va, y_va_l)], callbacks=[lgb.early_stopping(35, verbose=False)])
    else:
        lgb_l.fit(X_tr, y_l)

    xgb_l = xgb.XGBClassifier(
        n_estimators=220 if fast else 320, learning_rate=0.07, max_depth=6, colsample_bytree=0.7, subsample=0.85,
        tree_method="hist", device=XGB_DEV, n_jobs=8, eval_metric="mlogloss", random_state=seed + 1,
    )
    xgb_l.fit(X_tr, y_l)

    cb_l = None
    if HAS_CB:
        cb_l = CatBoostClassifier(
            iterations=220 if fast else 360, learning_rate=0.08, depth=6,
            auto_class_weights="Balanced", thread_count=8, random_seed=seed + 2, verbose=0,
        )
        cb_l.fit(X_tr, y_l)

    scam_mask = np.isin(y_l, [1, 2, 3, 4])
    lgbs_t = []
    for j in range(6):
        c = lgb.LGBMClassifier(
            n_estimators=260, learning_rate=0.05, num_leaves=31,
            colsample_bytree=0.75, subsample=0.85, class_weight="balanced",
            random_state=seed + 10 + j, n_jobs=8, verbose=-1,
        )
        c.fit(X_tr[scam_mask], y_t[scam_mask, j])
        lgbs_t.append(c)

    act_mask = scam_mask & (y_a > 0)
    ya_zero = y_a[act_mask] - 1
    lgb_a = lgb.LGBMClassifier(
        n_estimators=350, learning_rate=0.05, num_leaves=31,
        colsample_bytree=0.75, subsample=0.85, class_weight="balanced",
        random_state=seed + 99, n_jobs=8, verbose=-1,
    )
    lgb_a.fit(X_tr[act_mask], ya_zero)
    return {"lgb_l": lgb_l, "xgb_l": xgb_l, "cb_l": cb_l, "lgbs_t": lgbs_t, "lgb_a": lgb_a}


def predict_gbdt(models, X):
    p_lgb = models["lgb_l"].predict_proba(X).astype(np.float32)
    p_xgb = models["xgb_l"].predict_proba(X).astype(np.float32)
    if models.get("cb_l") is not None:
        p_cb = models["cb_l"].predict_proba(X).astype(np.float32)
        pl = 0.40 * p_lgb + 0.35 * p_xgb + 0.25 * p_cb
    else:
        p_cb = p_lgb
        pl = 0.55 * p_lgb + 0.45 * p_xgb
    pt = np.column_stack([c.predict_proba(X)[:, 1] for c in models["lgbs_t"]]).astype(np.float32)
    pa_sub = models["lgb_a"].predict_proba(X).astype(np.float32)
    pa = np.zeros((X.shape[0], 6), dtype=np.float32)
    for ci, cls_id in enumerate(models["lgb_a"].classes_):
        pa[:, int(cls_id) + 1] = pa_sub[:, ci]
    return p_lgb, p_xgb, p_cb, pl, pt, pa


def run_baseline_v5():
    t0 = time.time()
    sdf = pd.read_csv(CACHE_DIR / "sample_v4.csv")
    sf = np.load(CACHE_DIR / "sample_v4_feats.npy")
    sy = pd.read_csv(CACHE_DIR / "synth_64k_v4.csv")
    syf = np.load(CACHE_DIR / "synth_64k_v4_feats.npy")

    s_txt = make_turn_texts(sdf["clean_text"].fillna(""))
    sy_txt = make_turn_texts(sy["clean_text"].fillna(""))

    s_y_l = sdf["label"].values.astype(int)
    s_y_t = np.vstack([tech_str_to_vec(x) for x in sdf["techniques"]])
    s_y_a = sdf["action"].values.astype(int)
    s_fams = sdf["scenario_family"].values

    sy_y_l = sy["label"].values.astype(int)
    sy_y_t = np.vstack([tech_str_to_vec(x) for x in sy["techniques"]])
    sy_y_a = sy["action"].values.astype(int)
    sy_fams = sy["scenario_family"].values

    # Vectorize TF-IDF on synth + sample cleanly
    vec_w = TfidfVectorizer(ngram_range=(1, 2), max_features=80000, sublinear_tf=True, min_df=2, token_pattern=r"(?u)\b\w+\b|[<>\[\]\|]+")
    vec_c = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=50000, sublinear_tf=True, min_df=3)
    Xw_sy = vec_w.fit_transform(sy_txt)
    Xc_sy = vec_c.fit_transform(sy_txt)
    X_tfidf_sy = hstack([Xw_sy, Xc_sy]).tocsr()

    Xw_s = vec_w.transform(s_txt)
    Xc_s = vec_c.transform(s_txt)
    X_tfidf_s = hstack([Xw_s, Xc_s]).tocsr()

    svd = TruncatedSVD(n_components=128, random_state=42)
    Z_sy = svd.fit_transform(X_tfidf_sy).astype(np.float32)
    Z_s = svd.transform(X_tfidf_s).astype(np.float32)

    scaler = StandardScaler()
    F_sy = scaler.fit_transform(syf).astype(np.float32)
    F_s = scaler.transform(sf).astype(np.float32)

    X_lin_sy = hstack([X_tfidf_sy, csr_matrix(F_sy * 0.25)]).tocsr()
    X_lin_s = hstack([X_tfidf_s, csr_matrix(F_s * 0.25)]).tocsr()

    X_tab_sy = np.hstack([F_sy, Z_sy]).astype(np.float32)
    X_tab_s = np.hstack([F_s, Z_s]).astype(np.float32)

    # ---- PILLAR 1: ZERO-LEAKAGE LOSO ON SAMPLE.CSV ----
    gkf = StratifiedGroupKFold(n_splits=5)
    oof_lin_s_l = np.zeros((len(sdf), 6), dtype=np.float32)
    oof_lin_s_t = np.zeros((len(sdf), 6), dtype=np.float32)
    oof_lin_s_a = np.zeros((len(sdf), 6), dtype=np.float32)
    oof_gb_s_l = np.zeros((len(sdf), 6), dtype=np.float32)
    oof_gb_s_t = np.zeros((len(sdf), 6), dtype=np.float32)
    oof_gb_s_a = np.zeros((len(sdf), 6), dtype=np.float32)

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(sdf, s_y_l, groups=s_fams)):
        va_coarse = set(s_fams[va_idx])
        excluded_synth = set()
        for cf in va_coarse:
            excluded_synth.update(LOSO_SYNTH_EXCLUSIONS.get(cf, set()))
        keep_sy = ~np.isin(sy_fams, list(excluded_synth))

        X_tr_l = vstack([X_lin_sy[keep_sy], X_lin_s[tr_idx]]).tocsr()
        y_tr_l = np.concatenate([sy_y_l[keep_sy], s_y_l[tr_idx]])
        y_tr_t = np.vstack([sy_y_t[keep_sy], s_y_t[tr_idx]])
        y_tr_a = np.concatenate([sy_y_a[keep_sy], s_y_a[tr_idx]])

        [(pl, pt, pa)], _ = fit_predict_linear_stack(X_tr_l, y_tr_l, y_tr_t, y_tr_a, [X_lin_s[va_idx]])
        oof_lin_s_l[va_idx], oof_lin_s_t[va_idx], oof_lin_s_a[va_idx] = pl, pt, pa

        # Train GBDT excluding same leaked families
        X_tr_g = np.vstack([X_tab_sy[keep_sy], np.repeat(X_tab_s[tr_idx], 2, axis=0)])
        yg_l = np.concatenate([sy_y_l[keep_sy], np.repeat(s_y_l[tr_idx], 2)])
        yg_t = np.vstack([sy_y_t[keep_sy], np.repeat(s_y_t[tr_idx], 2, axis=0)])
        yg_a = np.concatenate([sy_y_a[keep_sy], np.repeat(s_y_a[tr_idx], 2)])
        gm = fit_gbdt_multitask(X_tr_g, yg_l, yg_t, yg_a, seed=42 + fold)
        _, _, _, pgl, pgt, pga = predict_gbdt(gm, X_tab_s[va_idx])
        oof_gb_s_l[va_idx], oof_gb_s_t[va_idx], oof_gb_s_a[va_idx] = pgl, pgt, pga
        print(f"LOSO fold {fold}: excluded {len(excluded_synth)} synth fams, kept {keep_sy.sum()} rows | acc_lin={(pl.argmax(1)==s_y_l[va_idx]).mean():.3f} acc_gb={(pgl.argmax(1)==s_y_l[va_idx]).mean():.3f}", flush=True)

    # ---- PILLAR 2: UNSEEN SCENARIO FAMILY HOLDOUT ON SYNTH_64K ----
    ood_mask = np.isin(sy_fams, list(UNSEEN_HOLDOUT_FAMS))
    id_mask = ~ood_mask
    [(pl_ood, pt_ood, pa_ood)], _ = fit_predict_linear_stack(
        X_lin_sy[id_mask], sy_y_l[id_mask], sy_y_t[id_mask], sy_y_a[id_mask], [X_lin_sy[ood_mask]]
    )
    gm_ood = fit_gbdt_multitask(X_tab_sy[id_mask], sy_y_l[id_mask], sy_y_t[id_mask], sy_y_a[id_mask], seed=777)
    _, _, _, pgl_ood, pgt_ood, pga_ood = predict_gbdt(gm_ood, X_tab_sy[ood_mask])

    p_mix_s_l = 0.40 * oof_lin_s_l + 0.60 * oof_gb_s_l
    p_mix_s_t = 0.40 * oof_lin_s_t + 0.60 * oof_gb_s_t
    p_mix_s_a = 0.40 * oof_lin_s_a + 0.60 * oof_gb_s_a
    s_pl, s_pt, s_pa = decode_consistent(p_mix_s_l, p_mix_s_t, p_mix_s_a, tech_th=0.45)

    p_mix_ood_l = 0.40 * pl_ood + 0.60 * pgl_ood
    p_mix_ood_t = 0.40 * pt_ood + 0.60 * pgt_ood
    p_mix_ood_a = 0.40 * pa_ood + 0.60 * pga_ood
    ood_pl, ood_pt, ood_pa = decode_consistent(p_mix_ood_l, p_mix_ood_t, p_mix_ood_a, tech_th=0.45)

    raw_s = sdf["text"].values
    n_turns_ood = syf[ood_mask, 2]
    multi_mask = n_turns_ood >= 3

    scores = score_section10_proxy(
        s_y_l, s_pl, s_y_t, s_pt, s_y_a, s_pa, p_mix_s_l.max(axis=1),
        y_l_ood=sy_y_l[ood_mask], p_l_ood=ood_pl, conf_ood=p_mix_ood_l.max(axis=1),
        y_l_adv=sy_y_l[ood_mask], p_l_adv=ood_pl, conf_adv=p_mix_ood_l.max(axis=1),
        y_l_multi=sy_y_l[ood_mask][multi_mask], p_l_multi=ood_pl[multi_mask],
    )
    print("\n=== ZERO-LEAKAGE OOD BENCHMARK (TF-IDF Linear + LGBM/XGB) ===", scores, flush=True)

    # Save OOD OOF predictions for stacking
    np.savez_compressed(
        OUT_DIR / "oof_baseline_v4.npz",
        s_lin_l=oof_lin_s_l, s_lin_t=oof_lin_s_t, s_lin_a=oof_lin_s_a,
        s_gb_l=oof_gb_s_l, s_gb_t=oof_gb_s_t, s_gb_a=oof_gb_s_a,
        ood_mask=ood_mask,
        ood_lin_l=pl_ood, ood_lin_t=pt_ood, ood_lin_a=pa_ood,
        ood_gb_l=pgl_ood, ood_gb_t=pgt_ood, ood_gb_a=pga_ood,
    )

    # ---- FIT FULL MODELS ON ALL 64K + SAMPLE AND STREAM 500K PREDICTIONS ----
    X_full_lin = vstack([X_lin_sy, X_lin_s]).tocsr()
    y_full_l = np.concatenate([sy_y_l, s_y_l])
    y_full_t = np.vstack([sy_y_t, s_y_t])
    y_full_a = np.concatenate([sy_y_a, s_y_a])
    _, (full_l, full_ts, full_a) = fit_predict_linear_stack(X_full_lin, y_full_l, y_full_t, y_full_a, [])

    X_full_tab = np.vstack([X_tab_sy, np.repeat(X_tab_s, 3, axis=0)])
    yg_full_l = np.concatenate([sy_y_l, np.repeat(s_y_l, 3)])
    yg_full_t = np.vstack([sy_y_t, np.repeat(s_y_t, 3, axis=0)])
    yg_full_a = np.concatenate([sy_y_a, np.repeat(s_y_a, 3)])
    full_gb = fit_gbdt_multitask(X_full_tab, yg_full_l, yg_full_t, yg_full_a, seed=999, fast=False)
    ckpt_dir = CACHE_DIR / "ckpt_v4"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump({"vec_w": vec_w, "vec_c": vec_c, "svd": svd, "scaler": scaler, "lin": (full_l, full_ts, full_a), "gb": full_gb}, ckpt_dir / "baseline_v4.joblib", compress=3)

    te_f = np.load(CACHE_DIR / "test_v4_feats.npy", mmap_mode="r")
    n_te = len(te_f)
    lin_l = np.zeros((n_te, 6), dtype=np.float32)
    lin_t = np.zeros((n_te, 6), dtype=np.float32)
    lin_a = np.zeros((n_te, 6), dtype=np.float32)
    lgb_l = np.zeros((n_te, 6), dtype=np.float32)
    xgb_l = np.zeros((n_te, 6), dtype=np.float32)
    cb_l = np.zeros((n_te, 6), dtype=np.float32)
    gb_t = np.zeros((n_te, 6), dtype=np.float32)
    gb_a = np.zeros((n_te, 6), dtype=np.float32)

    chunk = 50000
    offset = 0
    for df_c in pd.read_csv(CACHE_DIR / "test_v4_clean.csv", chunksize=chunk):
        m = len(df_c)
        txts = make_turn_texts(df_c["clean_text"].fillna(""))
        X_tw = hstack([vec_w.transform(txts), vec_c.transform(txts)]).tocsr()
        F_c = scaler.transform(te_f[offset:offset + m]).astype(np.float32)
        Z_c = svd.transform(X_tw).astype(np.float32)
        X_lc = hstack([X_tw, csr_matrix(F_c * 0.25)]).tocsr()
        X_tc = np.hstack([F_c, Z_c]).astype(np.float32)

        lin_l[offset:offset + m] = full_l.predict_proba(X_lc).astype(np.float32)
        lin_t[offset:offset + m] = np.column_stack([c.predict_proba(X_lc)[:, 1] for c in full_ts]).astype(np.float32)
        pa_sub = full_a.predict_proba(X_lc).astype(np.float32)
        for ci, cls_id in enumerate(full_a.classes_):
            lin_a[offset:offset + m, int(cls_id)] = pa_sub[:, ci]

        pl1, px1, pcb1, _, pgt1, pga1 = predict_gbdt(full_gb, X_tc)
        lgb_l[offset:offset + m] = pl1
        xgb_l[offset:offset + m] = px1
        cb_l[offset:offset + m] = pcb1
        gb_t[offset:offset + m] = pgt1
        gb_a[offset:offset + m] = pga1
        offset += m
        print(f"  Predicted {offset}/{n_te} test rows...", flush=True)

    np.savez_compressed(
        OUT_DIR / "test_baseline_v4.npz",
        lin_l=lin_l, lin_t=lin_t, lin_a=lin_a,
        lgb_l=lgb_l, xgb_l=xgb_l, cb_l=cb_l, gb_t=gb_t, gb_a=gb_a,
    )

    # Blend inductive rules_v4 signals softly with Linear + GBDT for sharp Task B/C without rigid hard overrides
    t_ru = np.load(CACHE_DIR / "test_v4_rule_pred.npz")
    te_ids = pd.read_csv(CACHE_DIR / "test_v4_clean.csv", usecols=["id"])["id"].values
    r_pl = np.full((n_te, 6), 0.03, dtype=np.float32)
    r_pl[np.arange(n_te), t_ru["lbl"].astype(int)] = 0.85
    r_pt = t_ru["tvec"].astype(np.float32) * 0.84 + 0.08
    r_pa = np.zeros((n_te, 6), dtype=np.float32)
    ra_int = t_ru["act"].astype(int)
    pos_ra = ra_int > 0
    r_pa[pos_ra, 1:] = 0.0375
    r_pa[np.where(pos_ra)[0], ra_int[pos_ra]] = 0.85

    p_l = 0.28 * lin_l + 0.24 * lgb_l + 0.20 * xgb_l + 0.16 * cb_l + 0.12 * r_pl
    p_t = 0.28 * lin_t + 0.48 * gb_t + 0.24 * r_pt
    p_a = 0.28 * lin_a + 0.48 * gb_a + 0.24 * r_pa
    pl, pt, pa = decode_consistent(p_l, p_t, p_a, tech_th=0.45)
    maxp = p_l.max(axis=1)
    is_noise = te_f[:, 15] > 0.5
    conf = np.clip(0.85 * maxp + 0.12 * (1.0 - (te_f[:, 19] < 0.35).astype(np.float32)), 0.05, 0.992)
    conf = np.where(is_noise, np.minimum(conf, 0.90), conf)
    sub = pd.DataFrame({
        "id": te_ids,
        "label": pl,
        "techniques": [vec_to_tech_str(pt[i]) for i in range(n_te)],
        "action": pa,
        "confidence": np.round(conf, 4),
    })
    sub_path = WORK_DIR / "sub_v6b_v5.csv"
    sub.to_csv(sub_path, index=False)
    print(f"Saved {sub_path} in {time.time()-t0:.1f}s!", flush=True)

In [13]:
import argparse
import gc
import json
import os
import re
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup




def compute_class_weights(yl, ya):
    # Balanced label weights tempered with sqrt to avoid destabilizing CE while maximizing Macro-F1
    cnt_l = np.bincount(yl, minlength=6).astype(np.float32)
    w_l = np.sqrt(cnt_l.sum() / (6.0 * np.maximum(cnt_l, 1.0)))
    w_l = w_l / w_l.mean()

    scam_ya = ya[(ya >= 1) & (ya <= 5)]
    cnt_a = np.ones(6, dtype=np.float32)
    for c in range(1, 6):
        cnt_a[c] = max(1.0, float((scam_ya == c).sum()))
    w_a = np.ones(6, dtype=np.float32)
    w_a[1:] = np.sqrt(cnt_a[1:].sum() / (5.0 * cnt_a[1:]))
    w_a[1:] = np.clip(w_a[1:] / w_a[1:].mean(), 0.7, 2.2)
    return torch.tensor(w_l, dtype=torch.float32, device=DEV), torch.tensor(w_a, dtype=torch.float32, device=DEV)


def train_one_teacher_fold(
    model_name, tag, tr_texts, tr_feats, tr_yl, tr_yt, tr_ya,
    va_texts, va_feats, va_yl, n_sample_va=43,
    pool_mode="mean", freeze_k=6, bs=16, grad_accum=2, max_len=160,
    epochs=2, lr_bb=2.5e-5, lr_hd=1.8e-4, seed=42, add_e5_prefix=False,
):
    free_cuda()
    torch.manual_seed(seed)
    np.random.seed(seed)
    t0 = time.time()

    ckpt_path = CKPT_DIR / f"{tag}.pt"
    oof_path = OUT_DIR / f"oof_{tag}.npz"
    if ckpt_path.exists() and oof_path.exists():
        z = np.load(oof_path)
        print(f"  [{tag}] Resuming from cached {oof_path.name}!", flush=True)
        return (z["p_l"], z["p_t"], z["p_a"]), ckpt_path

    tr_in = ["passage: " + t for t in tr_texts] if add_e5_prefix else tr_texts
    va_in = ["passage: " + t for t in va_texts] if add_e5_prefix else va_texts

    # Use a fixed stratified ~3200-row slice (always including ALL sample_va rows) during intermediate epochs
    rng_v = np.random.RandomState(seed)
    if len(va_in) > 3200 + n_sample_va:
        sub_tail = np.sort(n_sample_va + rng_v.choice(len(va_in) - n_sample_va, size=3200, replace=False))
        mon_idx = np.concatenate([np.arange(n_sample_va), sub_tail])
    else:
        mon_idx = np.arange(len(va_in))
    mon_texts = [va_in[i] for i in mon_idx]
    mon_feats = va_feats[mon_idx]
    mon_yl = va_yl[mon_idx]

    tok = AutoTokenizer.from_pretrained(model_name, cache_dir=CACHE_DIR_HF, local_files_only=HF_LOCAL_ONLY)
    model = HeadConditionedTransformer(
        model_name, n_feats=tr_feats.shape[1], pool_mode=pool_mode,
        proj_dim=256, num_freeze_layers=freeze_k,
    ).to(DEV)

    ds = MultiTaskKDDataset(tr_in, tr_feats, tr_yl, tr_yt, tr_ya)
    sampler = BucketBatchSampler(tr_in, bs=bs, seed=seed, jitter=16.0)
    loader = DataLoader(ds, batch_sampler=sampler, collate_fn=make_collate_fn(tok, max_len=max_len, has_labels=True))

    total_steps = max(1, (len(loader) * epochs) // grad_accum)
    opt = torch.optim.AdamW(model.param_groups(lr_backbone=lr_bb, lr_head=lr_hd, weight_decay=0.01))
    sched = get_cosine_schedule_with_warmup(opt, num_warmup_steps=max(10, int(0.08 * total_steps)), num_training_steps=total_steps)
    scaler = torch.amp.GradScaler("cuda")
    w_l, w_a = compute_class_weights(tr_yl, tr_ya)

    best_score = -1.0

    for ep in range(epochs):
        model.train()
        opt.zero_grad(set_to_none=True)
        running_loss = 0.0
        for step, (ids, mask, f_t, yl, yt, ya) in enumerate(loader):
            ids = ids.cuda(non_blocking=False)
            mask = mask.cuda(non_blocking=False)
            f_t = f_t.cuda(non_blocking=False)
            yl = yl.cuda(non_blocking=False)
            yt = yt.cuda(non_blocking=False)
            ya = ya.cuda(non_blocking=False)

            with torch.amp.autocast("cuda"):
                ll, lt, la = model(ids, mask, f_t)
                loss = compute_multitask_loss(ll, lt, la, yl, yt, ya, label_weights=w_l, act_weights=w_a) / grad_accum
            scaler.scale(loss).backward()
            running_loss += float(loss.item()) * grad_accum

            if (step + 1) % grad_accum == 0 or (step + 1) == len(loader):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                sched.step()
                opt.zero_grad(set_to_none=True)
            if (step + 1) % 300 == 0:
                print(f"    [{tag}] ep={ep+1} step={step+1}/{len(loader)} loss={running_loss/(step+1):.4f} elapsed={time.time()-t0:.1f}s", flush=True)

        p_ml, _, _ = predict_transformer(model, tok, mon_texts, mon_feats, bs=max(bs * 2, 32), max_len=max_len, device=DEV)
        mf1 = 100.0 * f1_score(mon_yl, p_ml.argmax(axis=1), average="macro", zero_division=0)
        print(f"  [{tag}] ep={ep+1}/{epochs} loss={running_loss/max(1,len(loader)):.4f} OOD_mon_F1={mf1:.2f} elapsed={time.time()-t0:.1f}s", flush=True)
        if mf1 >= best_score:
            best_score = mf1
            torch.save({"state_dict": model.state_dict(), "model_name": model_name, "pool_mode": pool_mode, "freeze_k": freeze_k}, ckpt_path)

    # Load best checkpoint and evaluate full OOD fold once cleanly
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["state_dict"], strict=True)
    model = model.to(DEV).eval()
    p_l, p_t, p_a = predict_transformer(model, tok, va_in, va_feats, bs=max(bs * 2, 32), max_len=max_len, device=DEV)
    np.savez_compressed(oof_path, p_l=p_l, p_t=p_t, p_a=p_a)

    del model, opt, sched, scaler, loader, ckpt
    free_cuda()
    return (p_l, p_t, p_a), ckpt_path


def load_predict_ckpt(ckpt_path, texts, feats, bs=48, max_len=160, add_e5_prefix=False):
    free_cuda()
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    mname = ckpt["model_name"]
    tok = AutoTokenizer.from_pretrained(mname, cache_dir=CACHE_DIR_HF, local_files_only=HF_LOCAL_ONLY)
    model = HeadConditionedTransformer(
        mname, n_feats=feats.shape[1], pool_mode=ckpt["pool_mode"], proj_dim=256, num_freeze_layers=ckpt["freeze_k"]
    )
    model.load_state_dict(ckpt["state_dict"], strict=True)
    model = model.to(DEV).eval()
    inp = ["passage: " + t for t in texts] if add_e5_prefix else texts
    pl, pt, pa = predict_transformer(model, tok, inp, feats, bs=bs, max_len=max_len, device=DEV)
    del model, ckpt
    free_cuda()
    return pl, pt, pa


def select_domain_adaptation_prototypes(te_rules, te_base, te_clusters, te_feats, total_n=52000, seed=42):
    """Select diverse high-value training prototypes across cluster hubs, uncertainty frontier, and long-tail."""
    rng = np.random.RandomState(seed)
    n_te = len(te_feats)
    p_base_l = 0.32 * te_base["lin_l"] + 0.26 * te_base["lgb_l"] + 0.22 * te_base["xgb_l"] + 0.20 * te_base["cb_l"]
    p_base_a = 0.35 * te_base["lin_a"] + 0.65 * te_base["gb_a"]
    pred_l = p_base_l.argmax(axis=1)
    pred_a = np.where(np.isin(pred_l, [0, 5]), 0, 1 + p_base_a[:, 1:].argmax(axis=1))
    r_l = te_rules["lbl"].astype(int)
    r_a = te_rules["act"].astype(int)

    selected = []
    used = np.zeros(n_te, dtype=bool)

    def _add(idxs):
        for idx in idxs:
            if not used[idx]:
                used[idx] = True
                selected.append(int(idx))

    # 1. All cluster representatives for clusters >= 2 up to 22,000 (covering bulk of structured families)
    uniq_first = te_clusters["first_idx"]
    counts = te_clusters["counts"]
    ge2 = uniq_first[counts >= 2][:22000]
    _add(ge2)

    # 2. Disagreement & low-confidence frontier between rules_v4 and TFIDF-GBDT + rare action=5/1/4
    disagree_l = (r_l != pred_l)
    disagree_a = (r_a != pred_a) & ((r_l >= 1) & (r_l <= 4) | (pred_l >= 1) & (pred_l <= 4))
    low_conf = p_base_l.max(axis=1) < 0.84
    rare_act = (r_a == 5) | (pred_a == 5) | ((pred_a == 1) & (pred_l != 1)) | ((r_a == 1) & (pred_l != 1))
    frontier_pool = np.where((disagree_l | disagree_a | low_conf | rare_act) & (~used))[0]
    if len(frontier_pool) > 19000:
        # Prioritize severe disagreement + rare action first
        score_f = (
            3.0 * disagree_l[frontier_pool].astype(np.float32)
            + 2.0 * rare_act[frontier_pool].astype(np.float32)
            + (1.0 - p_base_l.max(axis=1)[frontier_pool])
        )
        frontier_pool = frontier_pool[np.argsort(-score_f)[:19000]]
    _add(frontier_pool)

    # 3. Balanced stratified fill from remaining singletons across predicted classes 0..5 & multi-turn >=3
    rem = max(0, total_n - len(selected))
    per_cls = max(1, rem // 6)
    for c in range(6):
        pool_c = np.where((pred_l == c) & (~used))[0]
        if len(pool_c) > 0:
            take = min(len(pool_c), per_cls)
            _add(rng.choice(pool_c, size=take, replace=False))
    if len(selected) < total_n:
        pool_rem = np.where(~used)[0]
        if len(pool_rem) > 0:
            _add(rng.choice(pool_rem, size=min(len(pool_rem), total_n - len(selected)), replace=False))

    idx_arr = np.sort(np.array(selected[:total_n], dtype=np.int32))
    print(f"Selected {len(idx_arr)} domain-adaptation prototypes (clusters>=2={len(ge2)}, frontier={len(frontier_pool)})", flush=True)
    return idx_arr


def run_teachers_v4():
    ap = argparse.ArgumentParser()
    ap.add_argument("--max-per-fold", type=int, default=18000)
    ap.add_argument("--proto-n", type=int, default=48000)
    ap.add_argument("--qwen-max", type=int, default=0)
    ap.add_argument("--epochs-mdeb", type=int, default=2)
    ap.add_argument("--epochs-e5b", type=int, default=2)
    args = ap.parse_args([])

    sdf = pd.read_csv(CACHE_DIR / "sample_v4.csv")
    sf = scale_feats_v4(np.load(CACHE_DIR / "sample_v4_feats.npy"))
    sy = pd.read_csv(CACHE_DIR / "synth_64k_v4.csv")
    syf = scale_feats_v4(np.load(CACHE_DIR / "synth_64k_v4_feats.npy"))

    s_txt = [format_turn_aware(str(x)) for x in sdf["clean_text"].fillna("")]
    sy_txt = [format_turn_aware(str(x)) for x in sy["clean_text"].fillna("")]
    s_yl = sdf["label"].values.astype(int)
    s_yt = np.vstack([tech_str_to_vec(x) for x in sdf["techniques"]])
    s_ya = sdf["action"].values.astype(int)
    s_folds = get_scenario_disjoint_folds(sdf["scenario_family"].values)

    sy_yl = sy["label"].values.astype(int)
    sy_yt = np.vstack([tech_str_to_vec(x) for x in sy["techniques"]])
    sy_ya = sy["action"].values.astype(int)
    sy_folds = get_scenario_disjoint_folds(sy["scenario_family"].values)

    print(f"Scenario-Disjoint folds: sample F0={(s_folds==0).sum()} F1={(s_folds==1).sum()} | synth F0={(sy_folds==0).sum()} F1={(sy_folds==1).sum()}", flush=True)

    oof_mdeb_s_l = np.zeros((len(sdf), 6), dtype=np.float32)
    oof_mdeb_s_t = np.zeros((len(sdf), 6), dtype=np.float32)
    oof_mdeb_s_a = np.zeros((len(sdf), 6), dtype=np.float32)
    oof_e5b_s_l = np.zeros((len(sdf), 6), dtype=np.float32)
    oof_e5b_s_t = np.zeros((len(sdf), 6), dtype=np.float32)
    oof_e5b_s_a = np.zeros((len(sdf), 6), dtype=np.float32)

    oof_mdeb_sy_l = np.zeros((len(sy), 6), dtype=np.float32)
    oof_mdeb_sy_t = np.zeros((len(sy), 6), dtype=np.float32)
    oof_mdeb_sy_a = np.zeros((len(sy), 6), dtype=np.float32)
    oof_e5b_sy_l = np.zeros((len(sy), 6), dtype=np.float32)
    oof_e5b_sy_t = np.zeros((len(sy), 6), dtype=np.float32)
    oof_e5b_sy_a = np.zeros((len(sy), 6), dtype=np.float32)

    mdeb_ckpts, e5b_ckpts = [], []
    rng = np.random.RandomState(42)

    for fold in (0, 1):
        tr_sy_idx = np.where(sy_folds != fold)[0]
        va_sy_idx = np.where(sy_folds == fold)[0]
        tr_s_idx = np.where(s_folds != fold)[0]
        va_s_idx = np.where(s_folds == fold)[0]

        if len(tr_sy_idx) > args.max_per_fold:
            # Stratified cap preserving family diversity
            tr_sy_idx = np.sort(rng.choice(tr_sy_idx, size=args.max_per_fold, replace=False))

        tr_texts = [sy_txt[i] for i in tr_sy_idx] + [s_txt[i] for i in tr_s_idx] * 2
        tr_feats = np.vstack([syf[tr_sy_idx], sf[tr_s_idx], sf[tr_s_idx]])
        tr_yl = np.concatenate([sy_yl[tr_sy_idx], s_yl[tr_s_idx], s_yl[tr_s_idx]])
        tr_yt = np.vstack([sy_yt[tr_sy_idx], s_yt[tr_s_idx], s_yt[tr_s_idx]])
        tr_ya = np.concatenate([sy_ya[tr_sy_idx], s_ya[tr_s_idx], s_ya[tr_s_idx]])

        va_texts = [s_txt[i] for i in va_s_idx] + [sy_txt[i] for i in va_sy_idx]
        va_feats = np.vstack([sf[va_s_idx], syf[va_sy_idx]])
        va_yl = np.concatenate([s_yl[va_s_idx], sy_yl[va_sy_idx]])

        (pl_m, pt_m, pa_m), c_m = train_one_teacher_fold(
            "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", f"mdeberta_f{fold}",
            tr_texts, tr_feats, tr_yl, tr_yt, tr_ya, va_texts, va_feats, va_yl, n_sample_va=len(va_s_idx),
            pool_mode="mean", freeze_k=5, bs=8, grad_accum=4, max_len=152,
            epochs=args.epochs_mdeb, lr_bb=2.5e-5, lr_hd=1.8e-4, seed=100 + fold, add_e5_prefix=False,
        )
        ns = len(va_s_idx)
        oof_mdeb_s_l[va_s_idx], oof_mdeb_s_t[va_s_idx], oof_mdeb_s_a[va_s_idx] = pl_m[:ns], pt_m[:ns], pa_m[:ns]
        oof_mdeb_sy_l[va_sy_idx], oof_mdeb_sy_t[va_sy_idx], oof_mdeb_sy_a[va_sy_idx] = pl_m[ns:], pt_m[ns:], pa_m[ns:]
        mdeb_ckpts.append(c_m)

        (pl_e, pt_e, pa_e), c_e = train_one_teacher_fold(
            "intfloat/multilingual-e5-base", f"me5base_f{fold}",
            tr_texts, tr_feats, tr_yl, tr_yt, tr_ya, va_texts, va_feats, va_yl, n_sample_va=len(va_s_idx),
            pool_mode="cls_mean", freeze_k=4, bs=16, grad_accum=2, max_len=152,
            epochs=args.epochs_e5b, lr_bb=3.0e-5, lr_hd=2.0e-4, seed=200 + fold, add_e5_prefix=True,
        )
        oof_e5b_s_l[va_s_idx], oof_e5b_s_t[va_s_idx], oof_e5b_s_a[va_s_idx] = pl_e[:ns], pt_e[:ns], pa_e[:ns]
        oof_e5b_sy_l[va_sy_idx], oof_e5b_sy_t[va_sy_idx], oof_e5b_sy_a[va_sy_idx] = pl_e[ns:], pt_e[ns:], pa_e[ns:]
        e5b_ckpts.append(c_e)

    # Report true Zero-Leakage Scenario-Disjoint Teacher OOF composite scores
    p_mix_s_l = 0.60 * oof_mdeb_s_l + 0.40 * oof_e5b_s_l
    p_mix_s_t = 0.60 * oof_mdeb_s_t + 0.40 * oof_e5b_s_t
    p_mix_s_a = 0.60 * oof_mdeb_s_a + 0.40 * oof_e5b_s_a
    spl, spt, spa = decode_consistent(p_mix_s_l, p_mix_s_t, p_mix_s_a, tech_th=0.45)

    p_mix_sy_l = 0.60 * oof_mdeb_sy_l + 0.40 * oof_e5b_sy_l
    p_mix_sy_t = 0.60 * oof_mdeb_sy_t + 0.40 * oof_e5b_sy_t
    p_mix_sy_a = 0.60 * oof_mdeb_sy_a + 0.40 * oof_e5b_sy_a
    sypl, sypt, sypa = decode_consistent(p_mix_sy_l, p_mix_sy_t, p_mix_sy_a, tech_th=0.45)
    raw_syf = np.load(CACHE_DIR / "synth_64k_v4_feats.npy")
    multi_mask = raw_syf[:, 2] >= 3
    scores = score_section10_proxy(
        s_yl, spl, s_yt, spt, s_ya, spa, p_mix_s_l.max(axis=1),
        y_l_ood=sy_yl, p_l_ood=sypl, conf_ood=p_mix_sy_l.max(axis=1),
        y_l_adv=sy_yl, p_l_adv=sypl, conf_adv=p_mix_sy_l.max(axis=1),
        y_l_multi=sy_yl[multi_mask], p_l_multi=sypl[multi_mask],
    )
    print("\n=== ZERO-LEAKAGE SCENARIO-DISJOINT TEACHER ENSEMBLE OOF ===", scores, flush=True)

    np.savez_compressed(
        OUT_DIR / "oof_teachers_v4.npz",
        s_mdeb_l=oof_mdeb_s_l, s_mdeb_t=oof_mdeb_s_t, s_mdeb_a=oof_mdeb_s_a,
        s_e5b_l=oof_e5b_s_l, s_e5b_t=oof_e5b_s_t, s_e5b_a=oof_e5b_s_a,
        sy_mdeb_l=oof_mdeb_sy_l, sy_mdeb_t=oof_mdeb_sy_t, sy_mdeb_a=oof_mdeb_sy_a,
        sy_e5b_l=oof_e5b_sy_l, sy_e5b_t=oof_e5b_sy_t, sy_e5b_a=oof_e5b_sy_a,
    )

    # ---- STAGE 2: DOMAIN-ADAPTATION PROTOTYPE PSEUDO-LABELING (Rulebook Section 5) ----
    te_rules = np.load(CACHE_DIR / "test_v4_rule_pred.npz")
    te_base = np.load(OUT_DIR / "test_baseline_v4.npz")
    te_clusters = np.load(CACHE_DIR / "test_skel_clusters.npz")
    te_feats_mm = np.load(CACHE_DIR / "test_v4_feats.npy", mmap_mode="r")

    proto_idx = select_domain_adaptation_prototypes(te_rules, te_base, te_clusters, te_feats_mm, total_n=args.proto_n)
    proto_mask = np.zeros(len(te_feats_mm), dtype=bool)
    proto_mask[proto_idx] = True

    # Copy baseline subset on prototypes then free large 500k baseline arrays to guard 8.24 GB RAM
    pb_l = 0.32 * te_base["lin_l"][proto_idx] + 0.26 * te_base["lgb_l"][proto_idx] + 0.22 * te_base["xgb_l"][proto_idx] + 0.20 * te_base["cb_l"][proto_idx]
    pb_t = 0.35 * te_base["lin_t"][proto_idx] + 0.65 * te_base["gb_t"][proto_idx]
    pb_a = 0.35 * te_base["lin_a"][proto_idx] + 0.65 * te_base["gb_a"][proto_idx]
    r_arg = te_rules["lbl"][proto_idx].astype(int)
    del te_base, te_rules, te_clusters, oof_mdeb_sy_l, oof_mdeb_sy_t, oof_mdeb_sy_a, oof_e5b_sy_l, oof_e5b_sy_t, oof_e5b_sy_a, raw_syf, sy, syf, sy_txt
    gc.collect()

    clean_chunks, raw_chunks = [], []
    off = 0
    for df_c, df_r in zip(
        pd.read_csv(CACHE_DIR / "test_v4_clean.csv", chunksize=50000),
        pd.read_csv(V2_DIR.parents[1] / "Main Preli Round/dataset/test.csv", usecols=["text"], chunksize=50000),
    ):
        m = len(df_c)
        loc = proto_mask[off:off + m]
        if loc.any():
            clean_chunks.extend(df_c["clean_text"].fillna("").values[loc].tolist())
            raw_chunks.extend(df_r["text"].fillna("").values[loc].tolist())
        off += m
    proto_turn_texts = [format_turn_aware(t) for t in clean_chunks]
    proto_feats = scale_feats_v4(te_feats_mm[proto_idx])

    print("Scoring domain-adaptation prototypes with mDeBERTa (2 folds) & mE5-base (2 folds)...", flush=True)
    pm_cache = OUT_DIR / "proto_mdeb_tmp.npz"
    if pm_cache.exists():
        zm = np.load(pm_cache)
        pm_l, pm_t, pm_a = zm["pm_l"], zm["pm_t"], zm["pm_a"]
    else:
        m0_l, m0_t, m0_a = load_predict_ckpt(mdeb_ckpts[0], proto_turn_texts, proto_feats, bs=48, max_len=160, add_e5_prefix=False)
        m1_l, m1_t, m1_a = load_predict_ckpt(mdeb_ckpts[1], proto_turn_texts, proto_feats, bs=48, max_len=160, add_e5_prefix=False)
        pm_l, pm_t, pm_a = 0.5 * (m0_l + m1_l), 0.5 * (m0_t + m1_t), 0.5 * (m0_a + m1_a)
        np.savez_compressed(pm_cache, pm_l=pm_l, pm_t=pm_t, pm_a=pm_a)
        del m0_l, m0_t, m0_a, m1_l, m1_t, m1_a
        gc.collect()

    pe_cache = OUT_DIR / "proto_e5b_tmp.npz"
    if pe_cache.exists():
        ze = np.load(pe_cache)
        pe_l, pe_t, pe_a = ze["pe_l"], ze["pe_t"], ze["pe_a"]
    else:
        e0_l, e0_t, e0_a = load_predict_ckpt(e5b_ckpts[0], proto_turn_texts, proto_feats, bs=64, max_len=160, add_e5_prefix=True)
        e1_l, e1_t, e1_a = load_predict_ckpt(e5b_ckpts[1], proto_turn_texts, proto_feats, bs=64, max_len=160, add_e5_prefix=True)
        pe_l, pe_t, pe_a = 0.5 * (e0_l + e1_l), 0.5 * (e0_t + e1_t), 0.5 * (e0_a + e1_a)
        np.savez_compressed(pe_cache, pe_l=pe_l, pe_t=pe_t, pe_a=pe_a)
        del e0_l, e0_t, e0_a, e1_l, e1_t, e1_a
        gc.collect()

    p_cons_l = 0.50 * pm_l + 0.34 * pe_l + 0.16 * pb_l
    p_cons_t = 0.50 * pm_t + 0.34 * pe_t + 0.16 * pb_t
    p_cons_a = 0.50 * pm_a + 0.34 * pe_a + 0.16 * pb_a

    # Guard Priority 0/5 & non-empty Task B rules cleanly
    pseudo_l, pseudo_t_bin, pseudo_a = decode_consistent(p_cons_l, p_cons_t, p_cons_a, tech_th=0.45)
    pseudo_l = pseudo_l.astype(np.int64)
    pseudo_t_bin = pseudo_t_bin.astype(np.float32)
    pseudo_a = pseudo_a.astype(np.int64)
    conf_cons = p_cons_l.max(axis=1).astype(np.float32)

    np.savez_compressed(
        CACHE_DIR / "domain_pseudo_v4.npz",
        proto_idx=proto_idx,
        clean_texts=np.array(clean_chunks, dtype=object),
        turn_texts=np.array(proto_turn_texts, dtype=object),
        p_l=p_cons_l, p_t=p_cons_t, p_a=p_cons_a,
        pm_l=pm_l, pm_t=pm_t, pm_a=pm_a,
        pe_l=pe_l, pe_t=pe_t, pe_a=pe_a,
        y_l=pseudo_l, y_t=pseudo_t_bin, y_a=pseudo_a,
        conf=conf_cons,
    )
    print(f"Saved domain_pseudo_v4.npz ({len(proto_idx)} rows, mean_conf={conf_cons.mean():.4f}, dist={pd.Series(pseudo_l).value_counts().sort_index().to_dict()})", flush=True)

In [14]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd




def run_build_clean_pseudo_v5():
    dp = np.load(CACHE_DIR / "domain_pseudo_v4.npz", allow_pickle=True)
    ru = np.load(CACHE_DIR / "test_v4_rule_pred.npz")
    ba = np.load(OUT_DIR / "test_baseline_v4.npz")

    idx = dp["proto_idx"]
    # Align refreshed URL-masked test_v4_clean texts on the 45k prototypes
    mask = np.zeros(len(ru["lbl"]), dtype=bool)
    mask[idx] = True
    clean_refreshed = []
    off = 0
    for df_c in pd.read_csv(CACHE_DIR / "test_v4_clean.csv", chunksize=50000):
        m = len(df_c)
        loc = mask[off:off + m]
        if loc.any():
            clean_refreshed.extend(df_c["clean_text"].fillna("").values[loc].tolist())
        off += m

    rl = ru["lbl"][idx].astype(int)
    rt = ru["tvec"][idx].astype(np.float32)
    ra = ru["act"][idx].astype(int)

    pb_l = 0.28 * ba["lin_l"][idx] + 0.24 * ba["lgb_l"][idx] + 0.20 * ba["xgb_l"][idx] + 0.16 * ba["cb_l"][idx] + 0.12 * (
        np.eye(6, dtype=np.float32)[rl] * 0.85 + 0.03
    )
    pb_l /= pb_l.sum(axis=1, keepdims=True)
    pb_t = 0.28 * ba["lin_t"][idx] + 0.48 * ba["gb_t"][idx] + 0.24 * (rt * 0.84 + 0.08)
    pb_a = 0.28 * ba["lin_a"][idx] + 0.48 * ba["gb_a"][idx]
    ra_oh = np.zeros((len(idx), 6), dtype=np.float32)
    pos_a = ra > 0
    ra_oh[pos_a, 1:] = 0.0375
    ra_oh[np.where(pos_a)[0], ra[pos_a]] = 0.85
    pb_a = pb_a + 0.24 * ra_oh

    pm_l, pm_t, pm_a = dp["pm_l"], dp["pm_t"], dp["pm_a"]
    pe_l, pe_t, pe_a = dp["pe_l"], dp["pe_t"], dp["pe_a"]

    # Consensus filtering: protect against half-fold teacher blind spots and secrecy decoys!
    b_arg = pb_l.argmax(axis=1)
    m_arg = pm_l.argmax(axis=1)
    e_arg = pe_l.argmax(axis=1)

    p_cons_l = 0.24 * pm_l + 0.26 * pe_l + 0.50 * pb_l
    p_cons_t = 0.24 * pm_t + 0.26 * pe_t + 0.50 * pb_t
    p_cons_a = 0.24 * pm_a + 0.26 * pe_a + 0.50 * pb_a

    # Keep rows with high-trust consensus across rules_v5 / URL-masked Baseline_v5 / Teachers
    cond1 = (rl == b_arg) & (pb_l.max(axis=1) >= 0.72) & ((m_arg == b_arg) | (e_arg == b_arg) | (pb_l.max(axis=1) >= 0.86))
    cond2 = (m_arg == e_arg) & (e_arg == b_arg) & (p_cons_l.max(axis=1) >= 0.78)
    keep = cond1 | cond2

    k_idx = idx[keep]
    k_clean = [clean_refreshed[i] for i in np.where(keep)[0]]
    pl, pt, pa = decode_consistent(p_cons_l[keep], p_cons_t[keep], p_cons_a[keep], tech_th=0.45)

    np.savez_compressed(
        CACHE_DIR / "domain_pseudo_v5.npz",
        proto_idx=k_idx,
        clean_texts=np.array(k_clean, dtype=object),
        p_l=p_cons_l[keep].astype(np.float32),
        p_t=p_cons_t[keep].astype(np.float32),
        p_a=p_cons_a[keep].astype(np.float32),
        y_l=pl.astype(np.int64),
        y_t=pt.astype(np.float32),
        y_a=pa.astype(np.int64),
        conf=p_cons_l[keep].max(axis=1).astype(np.float32),
    )
    print(
        f"Filtered ultra-clean domain_pseudo_v5.npz: kept {keep.sum()}/{len(idx)} ({100*keep.mean():.1f}%) "
        f"| mean_conf={p_cons_l[keep].max(axis=1).mean():.4f} "
        f"| dist={pd.Series(pl).value_counts().sort_index().to_dict()} "
        f"| act_dist={pd.Series(pa).value_counts().sort_index().to_dict()}"
    )

In [15]:
import argparse
import gc
import os
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup




def compute_student_weights(yl, ya):
    cnt_l = np.bincount(yl, minlength=6).astype(np.float32)
    w_l = np.sqrt(cnt_l.sum() / (6.0 * np.maximum(cnt_l, 1.0)))
    w_l = w_l / w_l.mean()
    scam_ya = ya[(ya >= 1) & (ya <= 5)]
    cnt_a = np.ones(6, dtype=np.float32)
    for c in range(1, 6):
        cnt_a[c] = max(1.0, float((scam_ya == c).sum()))
    w_a = np.ones(6, dtype=np.float32)
    w_a[1:] = np.sqrt(cnt_a[1:].sum() / (5.0 * cnt_a[1:]))
    w_a[1:] = np.clip(w_a[1:] / w_a[1:].mean(), 0.7, 2.2)
    return torch.tensor(w_l, dtype=torch.float32, device=DEV), torch.tensor(w_a, dtype=torch.float32, device=DEV)


def prep_text_list(clean_texts, mode="plain"):
    if mode == "e5":
        return ["passage: " + format_turn_aware(str(t)) for t in clean_texts]
    if mode == "bge":
        return ["Represent this sentence for searching relevant passages: " + bn_to_en_roman_v4(format_turn_aware(str(t))) for t in clean_texts]
    return [format_turn_aware(str(t)) for t in clean_texts]


def train_student_model(
    model_name, tag, tr_texts, tr_feats, tr_yl, tr_yt, tr_ya,
    tr_soft_l=None, tr_soft_t=None, tr_soft_a=None, tr_weights=None,
    va_texts=None, va_feats=None, va_yl=None, n_sample_va=43,
    pool_mode="mean", bs=32, max_len=152, epochs=2, lr_bb=3.5e-5, lr_hd=2.0e-4, seed=42, kd_alpha=0.45,
):
    free_cuda()
    torch.manual_seed(seed)
    np.random.seed(seed)
    t0 = time.time()

    ckpt_path = CKPT_DIR / f"{tag}.pt"
    oof_path = OUT_DIR / f"oof_{tag}.npz"
    if va_texts is not None and ckpt_path.exists() and oof_path.exists():
        z = np.load(oof_path)
        print(f"  [{tag}] Resuming cached {oof_path.name}!", flush=True)
        return (z["p_l"], z["p_t"], z["p_a"]), ckpt_path

    has_soft = tr_soft_l is not None
    has_w = tr_weights is not None
    tok = AutoTokenizer.from_pretrained(model_name, cache_dir=CACHE_DIR_HF, local_files_only=HF_LOCAL_ONLY)
    model = HeadConditionedTransformer(
        model_name, n_feats=tr_feats.shape[1], pool_mode=pool_mode, proj_dim=256, num_freeze_layers=0
    ).to(DEV)

    ds = MultiTaskKDDataset(
        tr_texts, tr_feats, tr_yl, tr_yt, tr_ya,
        soft_l=tr_soft_l, soft_t=tr_soft_t, soft_a=tr_soft_a, weights=tr_weights,
    )
    sampler = BucketBatchSampler(tr_texts, bs=bs, seed=seed, jitter=16.0)
    loader = DataLoader(ds, batch_sampler=sampler, collate_fn=make_collate_fn(tok, max_len=max_len, has_labels=True, has_soft=has_soft, has_weights=has_w))

    mon_texts = va_texts
    mon_feats = va_feats
    mon_yl = va_yl
    if va_texts is not None and len(va_texts) > 3200 + n_sample_va:
        rng_v = np.random.RandomState(seed)
        sub_tail = np.sort(n_sample_va + rng_v.choice(len(va_texts) - n_sample_va, size=3200, replace=False))
        mon_idx = np.concatenate([np.arange(n_sample_va), sub_tail])
        mon_texts = [va_texts[i] for i in mon_idx]
        mon_feats = va_feats[mon_idx]
        mon_yl = va_yl[mon_idx]

    total_steps = max(1, len(loader) * epochs)
    opt = torch.optim.AdamW(model.param_groups(lr_backbone=lr_bb, lr_head=lr_hd, weight_decay=0.01))
    sched = get_cosine_schedule_with_warmup(opt, num_warmup_steps=max(20, int(0.08 * total_steps)), num_training_steps=total_steps)
    scaler = torch.amp.GradScaler("cuda")
    w_l, w_a = compute_student_weights(tr_yl, tr_ya)

    best_score = -1.0

    for ep in range(epochs):
        model.train()
        rloss = 0.0
        for batch in loader:
            ids = batch[0].cuda(non_blocking=False)
            mask = batch[1].cuda(non_blocking=False)
            f_t = batch[2].cuda(non_blocking=False)
            yl = batch[3].cuda(non_blocking=False)
            yt = batch[4].cuda(non_blocking=False)
            ya = batch[5].cuda(non_blocking=False)
            pos = 6
            sl = st = sa = sw = None
            if has_soft:
                sl = batch[pos].cuda(non_blocking=False)
                st = batch[pos + 1].cuda(non_blocking=False)
                sa = batch[pos + 2].cuda(non_blocking=False)
                pos += 3
            if has_w:
                sw = batch[pos].cuda(non_blocking=False)

            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda"):
                ll, lt, la = model(ids, mask, f_t)
                loss = compute_multitask_loss(
                    ll, lt, la, yl, yt, ya,
                    label_weights=w_l, act_weights=w_a, sample_w=sw,
                    soft_l=sl, soft_t=st, soft_a=sa, kd_alpha=kd_alpha,
                )
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            rloss += float(loss.item())

        if va_texts is not None:
            p_ml, _, _ = predict_transformer(model, tok, mon_texts, mon_feats, bs=max(bs * 2, 64), max_len=max_len, device=DEV)
            mf1 = 100.0 * f1_score(mon_yl, p_ml.argmax(axis=1), average="macro", zero_division=0)
            print(f"  [{tag}] ep={ep+1}/{epochs} loss={rloss/max(1,len(loader)):.4f} OOD_mon_F1={mf1:.2f} elapsed={time.time()-t0:.1f}s", flush=True)
            if mf1 >= best_score:
                best_score = mf1
                torch.save({"state_dict": model.state_dict(), "model_name": model_name, "pool_mode": pool_mode}, ckpt_path)
        else:
            print(f"  [{tag}] ep={ep+1}/{epochs} loss={rloss/max(1,len(loader)):.4f} elapsed={time.time()-t0:.1f}s", flush=True)
            torch.save({"state_dict": model.state_dict(), "model_name": model_name, "pool_mode": pool_mode}, ckpt_path)

    best_va = None
    if va_texts is not None:
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        model.load_state_dict(ckpt["state_dict"], strict=True)
        model = model.to(DEV).eval()
        p_l, p_t, p_a = predict_transformer(model, tok, va_texts, va_feats, bs=max(bs * 2, 64), max_len=max_len, device=DEV)
        np.savez_compressed(oof_path, p_l=p_l, p_t=p_t, p_a=p_a)
        best_va = (p_l, p_t, p_a)

    del model, opt, sched, scaler, loader
    free_cuda()
    return best_va, ckpt_path


def stream_predict_500k(ckpt_path, out_npz_path, text_mode="plain", bs=96, max_len=152):
    free_cuda()
    t0 = time.time()
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    mname = ckpt["model_name"]
    tok = AutoTokenizer.from_pretrained(mname, cache_dir=CACHE_DIR_HF, local_files_only=HF_LOCAL_ONLY)
    model = HeadConditionedTransformer(mname, n_feats=58, pool_mode=ckpt["pool_mode"], proj_dim=256)
    model.load_state_dict(ckpt["state_dict"], strict=True)
    model = model.to(DEV).eval()

    te_f = np.load(CACHE_DIR / "test_v4_feats.npy", mmap_mode="r")
    n_te = len(te_f)
    p_l = np.zeros((n_te, 6), dtype=np.float32)
    p_t = np.zeros((n_te, 6), dtype=np.float32)
    p_a = np.zeros((n_te, 6), dtype=np.float32)

    off = 0
    for df_c in pd.read_csv(CACHE_DIR / "test_v4_clean.csv", chunksize=50000):
        m = len(df_c)
        txts = prep_text_list(df_c["clean_text"].fillna("").tolist(), mode=text_mode)
        f_c = scale_feats_v4(te_f[off:off + m])
        pl, pt, pa = predict_transformer(model, tok, txts, f_c, bs=bs, max_len=max_len, device=DEV)
        p_l[off:off + m] = pl
        p_t[off:off + m] = pt
        p_a[off:off + m] = pa
        off += m
        print(f"  [{out_npz_path.name}] {off}/{n_te} in {time.time()-t0:.1f}s", flush=True)

    np.savez_compressed(out_npz_path, p_l=p_l, p_t=p_t, p_a=p_a)
    del model, ckpt
    free_cuda()


def _stratified_synth_idx(yl, max_n, rng):
    if max_n is None or len(yl) <= max_n:
        return np.arange(len(yl), dtype=np.int64)
    parts = []
    for c in range(6):
        idx_c = np.where(yl == c)[0]
        if len(idx_c) == 0:
            continue
        take = min(len(idx_c), max(1, int(round(max_n * len(idx_c) / len(yl)))))
        parts.append(rng.choice(idx_c, size=take, replace=False))
    return np.sort(np.concatenate(parts))


def _balanced_dp_idx(yl, conf, max_n, rng):
    if max_n is None or len(yl) <= max_n:
        return np.arange(len(yl), dtype=np.int64)
    cap_c = max(1, max_n // 5)
    parts = []
    for c in range(6):
        idx_c = np.where(yl == c)[0]
        if len(idx_c) <= cap_c:
            parts.append(idx_c)
        else:
            order = idx_c[np.argsort(-conf[idx_c])]
            n_top = int(cap_c * 0.85)
            n_rem = cap_c - n_top
            top_part = order[:n_top]
            rem_part = rng.choice(order[n_top:], size=n_rem, replace=False) if n_rem > 0 else np.empty(0, dtype=np.int64)
            parts.append(np.concatenate([top_part, rem_part]))
    kept = np.sort(np.concatenate(parts))
    if len(kept) > max_n:
        kept = np.sort(rng.choice(kept, size=max_n, replace=False))
    return kept


def build_distill_corpus(sdf, sf, sy, syf, dp, text_mode="plain", seed=42, max_synth=26000, max_dp=34000):
    """Build combined inductive distillation training set: balanced synth + sample*4 + domain pseudo."""
    rng = np.random.RandomState(seed)

    s_yl = sdf["label"].values.astype(np.int64)
    s_yt = np.vstack([tech_str_to_vec(x) for x in sdf["techniques"]]).astype(np.float32)
    s_ya = sdf["action"].values.astype(np.int64)

    sy_yl_all = sy["label"].values.astype(np.int64)
    sy_keep = _stratified_synth_idx(sy_yl_all, max_synth, rng)
    sy_sub = sy.iloc[sy_keep]
    syf_sub = syf[sy_keep]
    sy_yl = sy_yl_all[sy_keep]
    sy_yt = np.vstack([tech_str_to_vec(x) for x in sy_sub["techniques"]]).astype(np.float32)
    sy_ya = sy_sub["action"].values.astype(np.int64)

    dp_yl_all = dp["y_l"].astype(np.int64)
    dp_conf_all = dp["conf"].astype(np.float32)
    dp_keep = _balanced_dp_idx(dp_yl_all, dp_conf_all, max_dp, rng)

    s_clean = sdf["clean_text"].fillna("").tolist()
    sy_clean = sy_sub["clean_text"].fillna("").tolist()
    dp_clean_all = dp["clean_texts"].tolist()
    dp_clean = [dp_clean_all[i] for i in dp_keep]

    dp_feats = scale_feats_v4(np.load(CACHE_DIR / "test_v4_feats.npy", mmap_mode="r")[dp["proto_idx"][dp_keep]])
    all_texts = prep_text_list(sy_clean, text_mode) + prep_text_list(s_clean, text_mode) * 4 + prep_text_list(dp_clean, text_mode)
    all_feats = np.vstack([syf_sub, sf, sf, sf, sf, dp_feats])

    all_yl = np.concatenate([sy_yl, s_yl, s_yl, s_yl, s_yl, dp_yl_all[dp_keep]])
    all_yt = np.vstack([sy_yt, s_yt, s_yt, s_yt, s_yt, dp["y_t"][dp_keep].astype(np.float32)])
    all_ya = np.concatenate([sy_ya, s_ya, s_ya, s_ya, s_ya, dp["y_a"][dp_keep].astype(np.int64)])

    # For hard-labeled source rows, construct sharpened 1-hot soft targets so KD loss is well-defined uniformly
    def onehot6(y):
        oh = np.full((len(y), 6), 0.005, dtype=np.float32)
        oh[np.arange(len(y)), y] = 0.975
        return oh

    n_src = len(sy_yl) + 4 * len(s_yl)
    src_yl = np.concatenate([sy_yl, s_yl, s_yl, s_yl, s_yl])
    src_yt = np.vstack([sy_yt, s_yt, s_yt, s_yt, s_yt])
    src_ya = np.concatenate([sy_ya, s_ya, s_ya, s_ya, s_ya])
    src_sl = onehot6(src_yl)
    src_st = src_yt * 0.94 + 0.03
    src_sa = onehot6(src_ya)

    all_sl = np.vstack([src_sl, dp["p_l"][dp_keep].astype(np.float32)])
    all_st = np.vstack([src_st, dp["p_t"][dp_keep].astype(np.float32)])
    all_sa = np.vstack([src_sa, dp["p_a"][dp_keep].astype(np.float32)])

    # Confidence weighting: source=1.0, pseudo scaled smoothly [0.35..1.15] by teacher consensus
    w_src = np.ones(n_src, dtype=np.float32)
    w_dp = np.clip((dp_conf_all[dp_keep] - 0.35) / 0.65, 0.35, 1.15).astype(np.float32)
    all_w = np.concatenate([w_src, w_dp])
    return all_texts, all_feats, all_yl, all_yt, all_ya, all_sl, all_st, all_sa, all_w


def run_selective_mdeberta_escalation(top_k=30000):
    """Inductive Teacher Escalation: run mDeBERTa-v3-base live on highest-uncertainty/disagreement rows."""
    p_m = np.load(OUT_DIR / "test_mminilm_v4.npz")
    p_e = np.load(OUT_DIR / "test_me5small_v4.npz")
    p_b = np.load(OUT_DIR / "test_bge_v4.npz")
    da_p = OUT_DIR / "test_da_tab_v4.npz"
    p_base = np.load(da_p if da_p.exists() else (OUT_DIR / "test_baseline_v4.npz"))

    pb_l = 0.32 * p_base["lin_l"] + 0.26 * p_base["lgb_l"] + 0.22 * p_base["xgb_l"] + 0.20 * p_base["cb_l"]
    blend_l = 0.34 * p_m["p_l"] + 0.30 * p_e["p_l"] + 0.20 * p_b["p_l"] + 0.16 * pb_l

    # Purely inductive uncertainty + cross-student disagreement score (works identically on any unseen dataset)
    ent = -np.sum(blend_l * np.log(np.clip(blend_l, 1e-6, 1.0)), axis=1)
    dis_l = (
        (p_m["p_l"].argmax(1) != p_e["p_l"].argmax(1)).astype(np.float32)
        + (p_m["p_l"].argmax(1) != p_b["p_l"].argmax(1)).astype(np.float32)
        + (p_m["p_l"].argmax(1) != pb_l.argmax(1)).astype(np.float32)
    )
    dis_a = (p_m["p_a"].argmax(1) != p_e["p_a"].argmax(1)).astype(np.float32) * np.isin(blend_l.argmax(1), [1, 2, 3, 4])
    score = ent + 0.65 * dis_l + 0.30 * dis_a + (1.0 - blend_l.max(1))

    esc_idx = np.sort(np.argsort(-score)[:top_k].astype(np.int32))
    esc_mask = np.zeros(len(score), dtype=bool)
    esc_mask[esc_idx] = True

    esc_clean = []
    off = 0
    for df_c in pd.read_csv(CACHE_DIR / "test_v4_clean.csv", chunksize=50000):
        m = len(df_c)
        loc = esc_mask[off:off + m]
        if loc.any():
            esc_clean.extend(df_c["clean_text"].fillna("").values[loc].tolist())
        off += m
    esc_texts = [format_turn_aware(t) for t in esc_clean]
    esc_feats = scale_feats_v4(np.load(CACHE_DIR / "test_v4_feats.npy", mmap_mode="r")[esc_idx])

    del p_m, p_e, p_b, p_base, pb_l, blend_l
    free_cuda()

    full_ckpt = CKPT_DIR / "mdeberta_full_distill.pt"
    if full_ckpt.exists():
        # Prefer full-scenario mDeBERTa (zero half-fold blind spot!)
        ck = torch.load(full_ckpt, map_location="cpu", weights_only=False)
        mname = ck["model_name"]
        tok = AutoTokenizer.from_pretrained(mname, cache_dir=CACHE_DIR_HF, local_files_only=HF_LOCAL_ONLY)
        mdl = HeadConditionedTransformer(mname, n_feats=58, pool_mode=ck["pool_mode"], proj_dim=256, num_freeze_layers=5)
        mdl.load_state_dict(ck["state_dict"], strict=True)
        mdl = mdl.to(DEV).eval()
        pl, pt, pa = predict_transformer(mdl, tok, esc_texts, esc_feats, bs=48, max_len=152)
        del mdl, ck
        free_cuda()
    else:
        l0, t0_, a0 = load_predict_ckpt(CKPT_DIR / "mdeberta_f0.pt", esc_texts, esc_feats, bs=48, max_len=152)
        l1, t1_, a1 = load_predict_ckpt(CKPT_DIR / "mdeberta_f1.pt", esc_texts, esc_feats, bs=48, max_len=152)
        pl, pt, pa = 0.5 * (l0 + l1), 0.5 * (t0_ + t1_), 0.5 * (a0 + a1)

    np.savez_compressed(
        OUT_DIR / "test_mdeberta_escalate_v4.npz",
        esc_idx=esc_idx, p_l=pl, p_t=pt, p_a=pa,
    )
    print(f"Saved selective inductive mDeBERTa escalation on top {len(esc_idx)} uncertain rows!", flush=True)


def run_students_v4():
    ap = argparse.ArgumentParser()
    ap.add_argument("--only", type=str, default="all", help="all|ood|mminilm|me5small|bge|mdeberta_full|escalate")
    ap.add_argument("--esc-k", type=int, default=26000)
    ap.add_argument("--force", action="store_true", help="Re-run even if output cache already exists")
    args = ap.parse_args([])

    sdf = pd.read_csv(CACHE_DIR / "sample_v4.csv")
    sf = scale_feats_v4(np.load(CACHE_DIR / "sample_v4_feats.npy"))
    sy = pd.read_csv(CACHE_DIR / "synth_64k_v4.csv")
    syf = scale_feats_v4(np.load(CACHE_DIR / "synth_64k_v4_feats.npy"))

    s_yl = sdf["label"].values.astype(np.int64)
    s_yt = np.vstack([tech_str_to_vec(x) for x in sdf["techniques"]]).astype(np.float32)
    s_ya = sdf["action"].values.astype(np.int64)
    s_folds = get_balanced_3folds(sdf["scenario_family"].values)

    sy_yl = sy["label"].values.astype(np.int64)
    sy_yt = np.vstack([tech_str_to_vec(x) for x in sy["techniques"]]).astype(np.float32)
    sy_ya = sy["action"].values.astype(np.int64)
    sy_folds = get_balanced_3folds(sy["scenario_family"].values)

    # 1. Zero-Leakage Balanced 3-Fold Scenario-Disjoint Student OOF (67% of every class in train!)
    oof_out = OUT_DIR / "oof_students_v4.npz"
    if args.only in ("all", "ood"):
        if oof_out.exists() and not args.force:
            print(f"[ood] {oof_out.name} already exists, skipping (pass --force to re-run).", flush=True)
        else:
            specs = [
                ("mminilm", "MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli", "plain", "mean", 48, 4.0e-5),
                ("me5small", "intfloat/multilingual-e5-small", "e5", "cls_mean", 36, 4.0e-5),
                ("bge", "BAAI/bge-small-en-v1.5", "bge", "cls_mean", 48, 4.0e-5),
            ]
            oof_dict = {}
            rng_sub = np.random.RandomState(42)
            for key, mname, tmode, pmode, bs, lrbb in specs:
                s_txts = prep_text_list(sdf["clean_text"].fillna(""), tmode)
                sy_txts = prep_text_list(sy["clean_text"].fillna(""), tmode)
                o_s_l = np.zeros((len(sdf), 6), dtype=np.float32)
                o_s_t = np.zeros((len(sdf), 6), dtype=np.float32)
                o_s_a = np.zeros((len(sdf), 6), dtype=np.float32)
                o_sy_l = np.zeros((len(sy), 6), dtype=np.float32)
                o_sy_t = np.zeros((len(sy), 6), dtype=np.float32)
                o_sy_a = np.zeros((len(sy), 6), dtype=np.float32)
                for fold in (0, 1, 2):
                    tr_sy = np.where(sy_folds != fold)[0]
                    if len(tr_sy) > 14000:
                        tr_sy = np.sort(rng_sub.choice(tr_sy, size=14000, replace=False))
                    va_sy = np.where(sy_folds == fold)[0]
                    tr_s = np.where(s_folds != fold)[0]
                    va_s = np.where(s_folds == fold)[0]
                    tr_tx = [sy_txts[i] for i in tr_sy] + [s_txts[i] for i in tr_s] * 3
                    tr_f = np.vstack([syf[tr_sy], sf[tr_s], sf[tr_s], sf[tr_s]])
                    va_tx = [s_txts[i] for i in va_s] + [sy_txts[i] for i in va_sy]
                    va_f = np.vstack([sf[va_s], syf[va_sy]])
                    (pl, pt, pa), _ = train_student_model(
                        mname, f"{key}_oof3_f{fold}",
                        tr_tx, tr_f,
                        np.concatenate([sy_yl[tr_sy], s_yl[tr_s], s_yl[tr_s], s_yl[tr_s]]),
                        np.vstack([sy_yt[tr_sy], s_yt[tr_s], s_yt[tr_s], s_yt[tr_s]]),
                        np.concatenate([sy_ya[tr_sy], s_ya[tr_s], s_ya[tr_s], s_ya[tr_s]]),
                        va_texts=va_tx, va_feats=va_f, va_yl=np.concatenate([s_yl[va_s], sy_yl[va_sy]]), n_sample_va=len(va_s),
                        pool_mode=pmode, bs=bs, max_len=144, epochs=2, lr_bb=lrbb, seed=300 + fold, kd_alpha=0.0,
                    )
                    ns = len(va_s)
                    o_s_l[va_s], o_s_t[va_s], o_s_a[va_s] = pl[:ns], pt[:ns], pa[:ns]
                    o_sy_l[va_sy], o_sy_t[va_sy], o_sy_a[va_sy] = pl[ns:], pt[ns:], pa[ns:]
                oof_dict[f"s_{key}_l"] = o_s_l; oof_dict[f"s_{key}_t"] = o_s_t; oof_dict[f"s_{key}_a"] = o_s_a
                oof_dict[f"sy_{key}_l"] = o_sy_l; oof_dict[f"sy_{key}_t"] = o_sy_t; oof_dict[f"sy_{key}_a"] = o_sy_a
            np.savez_compressed(oof_out, **oof_dict)

    # 2. Domain-Adapted Inductive Students (distilled from ultra-clean consensus domain_pseudo_v5)
    dp_path = CACHE_DIR / "domain_pseudo_v5.npz"
    if not dp_path.exists():
        dp_path = CACHE_DIR / "domain_pseudo_v4.npz"

    if args.only in ("all", "mminilm", "me5small", "bge"):
        dp = np.load(dp_path, allow_pickle=True)
        run_list = []
        if args.only in ("all", "mminilm"):
            run_list.append(("mminilm", "MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli", "plain", "mean", 32, 112, 15000, 25000, 2, 3.5e-5))
        if args.only in ("all", "me5small"):
            run_list.append(("me5small", "intfloat/multilingual-e5-small", "e5", "cls_mean", 24, 128, 11000, 23000, 1, 4.2e-5))
        if args.only in ("all", "bge"):
            run_list.append(("bge", "BAAI/bge-small-en-v1.5", "bge", "cls_mean", 32, 160, 11000, 23000, 1, 4.2e-5))

        for key, mname, tmode, pmode, bs_tr, bs_inf, m_syn, m_dp, ep_tr, lr_tr in run_list:
            pred_out = OUT_DIR / f"test_{key}_v4.npz"
            if pred_out.exists() and not args.force:
                print(f"[{key}] {pred_out.name} already exists, skipping (pass --force to re-run).", flush=True)
                continue
            ckpt = CKPT_DIR / f"{key}_distill.pt"
            if ckpt.exists() and not args.force:
                print(f"[{key}] Reusing existing checkpoint {ckpt.name} for 500k inference.", flush=True)
            else:
                tr_tx, tr_f, tr_yl, tr_yt, tr_ya, tr_sl, tr_st, tr_sa, tr_w = build_distill_corpus(
                    sdf, sf, sy, syf, dp, text_mode=tmode, max_synth=m_syn, max_dp=m_dp,
                )
                print(f"[{key}] Distill corpus rows={len(tr_tx)} ep={ep_tr}", flush=True)
                _, ckpt = train_student_model(
                    mname, f"{key}_distill", tr_tx, tr_f, tr_yl, tr_yt, tr_ya,
                    tr_soft_l=tr_sl, tr_soft_t=tr_st, tr_soft_a=tr_sa, tr_weights=tr_w,
                    pool_mode=pmode, bs=bs_tr, max_len=144, epochs=ep_tr, lr_bb=lr_tr, seed=500, kd_alpha=0.45,
                )
            stream_predict_500k(ckpt, pred_out, text_mode=tmode, bs=bs_inf, max_len=144)

    if args.only in ("all", "mdeberta_full"):
        md_ckpt = CKPT_DIR / "mdeberta_full_distill.pt"
        if md_ckpt.exists() and not args.force:
            print(f"[mdeberta_full] {md_ckpt.name} already exists, skipping.", flush=True)
        else:
            dp = np.load(dp_path, allow_pickle=True)
            # Capped cleanly at ~20,344 class-balanced rows (8k synth + 4*86 sample + 12k real dp) -> ~24m total!
            tr_tx, tr_f, tr_yl, tr_yt, tr_ya, tr_sl, tr_st, tr_sa, tr_w = build_distill_corpus(
                sdf, sf, sy, syf, dp, text_mode="plain", seed=2026, max_synth=8000, max_dp=12000,
            )
            print(f"[mdeberta_full] Balanced corpus rows={len(tr_tx)}", flush=True)
            tok_md = AutoTokenizer.from_pretrained("MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", cache_dir=CACHE_DIR_HF, local_files_only=HF_LOCAL_ONLY)
            mdl_md = HeadConditionedTransformer("MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", n_feats=58, pool_mode="mean", proj_dim=256, num_freeze_layers=5).to(DEV)
            ds_md = MultiTaskKDDataset(
                tr_tx, tr_f, tr_yl, tr_yt, tr_ya,
                soft_l=tr_sl, soft_t=tr_st, soft_a=tr_sa, weights=tr_w,
            )
            loader_md = DataLoader(ds_md, batch_sampler=BucketBatchSampler(tr_tx, bs=8, seed=777), collate_fn=make_collate_fn(tok_md, max_len=152, has_labels=True, has_soft=True, has_weights=True))
            opt_md = torch.optim.AdamW(mdl_md.param_groups(lr_backbone=2.5e-5, lr_head=1.8e-4))
            sched_md = get_cosine_schedule_with_warmup(opt_md, num_warmup_steps=80, num_training_steps=max(1, len(loader_md) // 4))
            sc_md = torch.amp.GradScaler("cuda")
            w_l, w_a = compute_student_weights(tr_yl, tr_ya)
            t_md = time.time()
            mdl_md.train(); opt_md.zero_grad(set_to_none=True)
            for st_idx, b in enumerate(loader_md):
                ids, msk, ft, yl, yt, ya, sl, st, sa, sw = [x.to(DEV) for x in b]
                with torch.amp.autocast("cuda"):
                    ll, lt, la = mdl_md(ids, msk, ft)
                    loss = compute_multitask_loss(ll, lt, la, yl, yt, ya, label_weights=w_l, act_weights=w_a, sample_w=sw, soft_l=sl, soft_t=st, soft_a=sa, kd_alpha=0.45) / 4.0
                sc_md.scale(loss).backward()
                if (st_idx + 1) % 4 == 0 or (st_idx + 1) == len(loader_md):
                    sc_md.unscale_(opt_md); torch.nn.utils.clip_grad_norm_(mdl_md.parameters(), 1.0)
                    sc_md.step(opt_md); sc_md.update(); sched_md.step(); opt_md.zero_grad(set_to_none=True)
                if (st_idx + 1) % 500 == 0:
                    print(f"  [mdeberta_full_distill] step={st_idx+1}/{len(loader_md)} elapsed={time.time()-t_md:.1f}s", flush=True)
            torch.save({"state_dict": mdl_md.state_dict(), "model_name": "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", "pool_mode": "mean"}, md_ckpt)
            del mdl_md, opt_md, sc_md, loader_md; free_cuda()

    if args.only in ("all", "escalate"):
        esc_out = OUT_DIR / "test_mdeberta_escalate_v4.npz"
        if esc_out.exists() and not args.force:
            print(f"[escalate] {esc_out.name} already exists, skipping.", flush=True)
        else:
            run_selective_mdeberta_escalation(top_k=args.esc_k)

In [16]:
import argparse
import gc
import os
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from transformers import AutoTokenizer




class BPEBiGRUNet(nn.Module):
    """Non-Transformer Sequence Model: Subword-Hashed BiGRU + Attention/Max Pool + 58 Domain Features."""

    def __init__(self, n_buckets=32000, emb_dim=128, hidden=160, n_feats=58, proj_dim=224):
        super().__init__()
        self.emb = nn.Embedding(n_buckets, emb_dim, padding_idx=0)
        self.emb_drop = nn.Dropout(0.15)
        self.gru = nn.GRU(emb_dim, hidden, num_layers=2, batch_first=True, bidirectional=True, dropout=0.15)
        self.attn = nn.Linear(2 * hidden, 1)
        self.feat_norm = nn.LayerNorm(n_feats)
        self.fuse = nn.Sequential(
            nn.Linear(4 * hidden + n_feats, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.GELU(),
        )
        self.dropouts = nn.ModuleList([nn.Dropout(0.12 + 0.04 * i) for i in range(3)])
        self.head_label = nn.Linear(proj_dim, 6)
        self.head_tech = nn.Linear(proj_dim + 6, 6)
        self.head_action = nn.Linear(proj_dim + 6, 6)

    def forward(self, input_ids, attention_mask, feats):
        h_ids = input_ids % self.emb.num_embeddings
        h_ids = h_ids.masked_fill(attention_mask == 0, 0)
        x = self.emb_drop(self.emb(h_ids))
        out, _ = self.gru(x)
        mask_f = attention_mask.unsqueeze(-1).float()
        w_logits = self.attn(out).squeeze(-1).masked_fill(attention_mask == 0, -1e4)
        w = F.softmax(w_logits, dim=-1).unsqueeze(-1)
        attn_pool = (out * w).sum(dim=1)
        max_pool = (out.masked_fill(mask_f == 0, -1e4)).max(dim=1).values
        fn = self.feat_norm(feats)
        z = self.fuse(torch.cat([attn_pool, max_pool, fn], dim=-1))
        ll = sum(self.head_label(d(z)) for d in self.dropouts) / len(self.dropouts)
        zc = torch.cat([z, F.softmax(ll.detach(), dim=-1)], dim=-1)
        lt = sum(self.head_tech(d(zc)) for d in self.dropouts) / len(self.dropouts)
        la = sum(self.head_action(d(zc)) for d in self.dropouts) / len(self.dropouts)
        return ll, lt, la


def train_eval_bigru_fold(tr_texts, tr_feats, tr_yl, tr_yt, tr_ya, va_texts, va_feats, va_yl=None, sl=None, st=None, sa=None, sw=None, epochs=5, bs=96, seed=42):
    free_cuda()
    torch.manual_seed(seed)
    np.random.seed(seed)
    tok = AutoTokenizer.from_pretrained("MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli", cache_dir=CACHE_DIR_HF, local_files_only=HF_LOCAL_ONLY)
    model = BPEBiGRUNet(n_feats=tr_feats.shape[1]).to(DEV)

    has_soft = sl is not None
    has_w = sw is not None
    ds = MultiTaskKDDataset(tr_texts, tr_feats, tr_yl, tr_yt, tr_ya, sl, st, sa, sw)
    sampler = BucketBatchSampler(tr_texts, bs=bs, seed=seed, jitter=16.0)
    loader = DataLoader(ds, batch_sampler=sampler, collate_fn=make_collate_fn(tok, max_len=160, has_labels=True, has_soft=has_soft, has_weights=has_w))

    opt = torch.optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, len(loader) * epochs))

    cnt_l = np.bincount(tr_yl, minlength=6).astype(np.float32)
    w_l = torch.tensor(np.sqrt(cnt_l.sum() / (6.0 * np.maximum(cnt_l, 1.0))), dtype=torch.float32, device=DEV)
    w_l /= w_l.mean()
    scam_ya = tr_ya[(tr_ya >= 1) & (tr_ya <= 5)]
    w_a_arr = np.ones(6, dtype=np.float32)
    cnt_a = np.array([max(1.0, float((scam_ya == c).sum())) for c in range(1, 6)], dtype=np.float32)
    w_a_arr[1:] = np.clip(np.sqrt(cnt_a.sum() / (5.0 * cnt_a)), 0.7, 2.2)
    w_a = torch.tensor(w_a_arr, dtype=torch.float32, device=DEV)

    best_f1 = -1.0
    best_va = None
    best_sd = None
    for ep in range(epochs):
        model.train()
        for batch in loader:
            ids, mask, f_t = batch[0].to(DEV), batch[1].to(DEV), batch[2].to(DEV)
            yl, yt, ya = batch[3].to(DEV), batch[4].to(DEV), batch[5].to(DEV)
            pos = 6
            bsl = bst = bsa = bsw = None
            if has_soft:
                bsl, bst, bsa = batch[pos].to(DEV), batch[pos + 1].to(DEV), batch[pos + 2].to(DEV)
                pos += 3
            if has_w:
                bsw = batch[pos].to(DEV)
            opt.zero_grad(set_to_none=True)
            ll, lt, la = model(ids, mask, f_t)
            loss = compute_multitask_loss(ll, lt, la, yl, yt, ya, label_weights=w_l, act_weights=w_a, sample_w=bsw, soft_l=bsl, soft_t=bst, soft_a=bsa, kd_alpha=0.45 if has_soft else 0.0)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            opt.step()
            sched.step()

        if va_texts is not None and va_yl is not None:
            pl, pt, pa = predict_bigru(model, tok, va_texts, va_feats, bs=256)
            mf1 = 100.0 * f1_score(va_yl, pl.argmax(1), average="macro", zero_division=0)
            if mf1 >= best_f1:
                best_f1 = mf1
                best_va = (pl, pt, pa)
                best_sd = {k: v.cpu() for k, v in model.state_dict().items()}
    if best_sd is not None:
        model.load_state_dict({k: v.to(DEV) for k, v in best_sd.items()})
    return model, tok, best_va, best_f1


@torch.no_grad()
def predict_bigru(model, tok, texts, feats, bs=256, max_len=160):
    model.eval()
    dev = next(model.parameters()).device
    n = len(texts)
    order = np.argsort([len(str(t)) for t in texts])
    inv = np.argsort(order)
    ot = [texts[i] for i in order]
    of = feats[order]
    pl = np.zeros((n, 6), dtype=np.float32)
    pt = np.zeros((n, 6), dtype=np.float32)
    pa = np.zeros((n, 6), dtype=np.float32)
    for s in range(0, n, bs):
        e = min(n, s + bs)
        enc = tok(ot[s:e], padding=True, truncation=True, max_length=max_len, return_tensors="pt")
        ll, lt, la = model(enc["input_ids"].to(dev), enc["attention_mask"].to(dev), torch.from_numpy(of[s:e]).to(dev))
        pl[s:e] = F.softmax(ll, dim=-1).cpu().numpy()
        pt[s:e] = torch.sigmoid(lt).cpu().numpy()
        pa[s:e, 1:] = F.softmax(la[:, 1:], dim=-1).cpu().numpy()
    return pl[inv], pt[inv], pa[inv]


def run_da_bigru_v4():
    t0 = time.time()
    sdf = pd.read_csv(CACHE_DIR / "sample_v4.csv")
    sf = scale_feats_v4(np.load(CACHE_DIR / "sample_v4_feats.npy"))
    sy = pd.read_csv(CACHE_DIR / "synth_64k_v4.csv")
    syf = scale_feats_v4(np.load(CACHE_DIR / "synth_64k_v4_feats.npy"))

    s_tx = [format_turn_aware(str(x)) for x in sdf["clean_text"].fillna("")]
    sy_tx = [format_turn_aware(str(x)) for x in sy["clean_text"].fillna("")]
    s_yl = sdf["label"].values.astype(np.int64)
    s_yt = np.vstack([tech_str_to_vec(x) for x in sdf["techniques"]]).astype(np.float32)
    s_ya = sdf["action"].values.astype(np.int64)
    s_folds = get_balanced_3folds(sdf["scenario_family"].values)

    sy_yl = sy["label"].values.astype(np.int64)
    sy_yt = np.vstack([tech_str_to_vec(x) for x in sy["techniques"]]).astype(np.float32)
    sy_ya = sy["action"].values.astype(np.int64)
    sy_folds = get_balanced_3folds(sy["scenario_family"].values)

    # 1. Zero-Leakage Balanced 3-Fold Scenario-Disjoint BiGRU-v5 OOF
    o_s_l, o_s_t, o_s_a = np.zeros((len(sdf), 6), np.float32), np.zeros((len(sdf), 6), np.float32), np.zeros((len(sdf), 6), np.float32)
    o_sy_l, o_sy_t, o_sy_a = np.zeros((len(sy), 6), np.float32), np.zeros((len(sy), 6), np.float32), np.zeros((len(sy), 6), np.float32)
    for fold in (0, 1, 2):
        tr_sy, va_sy = np.where(sy_folds != fold)[0], np.where(sy_folds == fold)[0]
        tr_s, va_s = np.where(s_folds != fold)[0], np.where(s_folds == fold)[0]
        tr_tx = [sy_tx[i] for i in tr_sy] + [s_tx[i] for i in tr_s] * 2
        tr_f = np.vstack([syf[tr_sy], sf[tr_s], sf[tr_s]])
        va_tx = [s_tx[i] for i in va_s] + [sy_tx[i] for i in va_sy]
        va_f = np.vstack([sf[va_s], syf[va_sy]])
        m, tok, (pl, pt, pa), mf1 = train_eval_bigru_fold(
            tr_tx, tr_f,
            np.concatenate([sy_yl[tr_sy], s_yl[tr_s], s_yl[tr_s]]),
            np.vstack([sy_yt[tr_sy], s_yt[tr_s], s_yt[tr_s]]),
            np.concatenate([sy_ya[tr_sy], s_ya[tr_s], s_ya[tr_s]]),
            va_tx, va_f, va_yl=np.concatenate([s_yl[va_s], sy_yl[va_sy]]), epochs=4, bs=96, seed=70 + fold,
        )
        ns = len(va_s)
        o_s_l[va_s], o_s_t[va_s], o_s_a[va_s] = pl[:ns], pt[:ns], pa[:ns]
        o_sy_l[va_sy], o_sy_t[va_sy], o_sy_a[va_sy] = pl[ns:], pt[ns:], pa[ns:]
        print(f"BiGRU-v5 Scenario-Disjoint Fold {fold} OOD_F1={mf1:.2f} elapsed={time.time()-t0:.1f}s", flush=True)
        del m; free_cuda()

    np.savez_compressed(
        OUT_DIR / "oof_bigru_v4.npz",
        s_l=o_s_l, s_t=o_s_t, s_a=o_s_a,
        sy_l=o_sy_l, sy_t=o_sy_t, sy_a=o_sy_a,
    )

    # 2. Domain-Adapted BiGRU-v5 distilled from domain_pseudo_v5 + synth + sample -> predict 500k
    dp_p = CACHE_DIR / "domain_pseudo_v5.npz"
    dp = np.load(dp_p if dp_p.exists() else (CACHE_DIR / "domain_pseudo_v4.npz"), allow_pickle=True)
    tr_tx, tr_f, tr_yl, tr_yt, tr_ya, tr_sl, tr_st, tr_sa, tr_w = build_distill_corpus(sdf, sf, sy, syf, dp, text_mode="plain")
    m_full, tok, _, _ = train_eval_bigru_fold(
        tr_tx, tr_f, tr_yl, tr_yt, tr_ya, None, None, None,
        sl=tr_sl, st=tr_st, sa=tr_sa, sw=tr_w, epochs=5, bs=112, seed=909,
    )
    torch.save({"state_dict": m_full.state_dict()}, CKPT_DIR / "bigru_v4_distill.pt")

    te_f = np.load(CACHE_DIR / "test_v4_feats.npy", mmap_mode="r")
    n_te = len(te_f)
    p_l, p_t, p_a = np.zeros((n_te, 6), np.float32), np.zeros((n_te, 6), np.float32), np.zeros((n_te, 6), np.float32)
    off = 0
    for df_c in pd.read_csv(CACHE_DIR / "test_v4_clean.csv", chunksize=50000):
        m = len(df_c)
        txts = [format_turn_aware(str(x)) for x in df_c["clean_text"].fillna("")]
        pl, pt, pa = predict_bigru(m_full, tok, txts, scale_feats_v4(te_f[off:off + m]), bs=256)
        p_l[off:off + m], p_t[off:off + m], p_a[off:off + m] = pl, pt, pa
        off += m
    np.savez_compressed(OUT_DIR / "test_bigru_v4.npz", p_l=p_l, p_t=p_t, p_a=p_a)
    print(f"BiGRU-v4 500k prediction complete in {time.time()-t0:.1f}s!", flush=True)

In [17]:
import os
import sys
import time
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler




def run_da_tab_v4():
    t0 = time.time()
    sdf = pd.read_csv(CACHE_DIR / "sample_v4.csv")
    sf = np.load(CACHE_DIR / "sample_v4_feats.npy")
    sy = pd.read_csv(CACHE_DIR / "synth_64k_v4.csv")
    syf = np.load(CACHE_DIR / "synth_64k_v4_feats.npy")
    dp_p = CACHE_DIR / "domain_pseudo_v5.npz"
    dp = np.load(dp_p if dp_p.exists() else (CACHE_DIR / "domain_pseudo_v4.npz"), allow_pickle=True)
    te_f = np.load(CACHE_DIR / "test_v4_feats.npy", mmap_mode="r")
    dp_f = np.array(te_f[dp["proto_idx"]], dtype=np.float32)

    s_tx = [format_turn_aware(str(x)) for x in sdf["clean_text"].fillna("")]
    sy_tx = [format_turn_aware(str(x)) for x in sy["clean_text"].fillna("")]
    dp_tx = [format_turn_aware(str(x)) for x in dp["clean_texts"].tolist()]

    # Filter high-consensus domain prototypes cleanly for crisp tree/linear targets
    keep_dp = dp["conf"] >= 0.62
    dp_tx_f = [dp_tx[i] for i in range(len(dp_tx)) if keep_dp[i]]
    dp_f_f = dp_f[keep_dp]

    all_tx = sy_tx + s_tx * 2 + dp_tx_f
    all_f = np.vstack([syf, sf, sf, dp_f_f]).astype(np.float32)

    s_yl = sdf["label"].values.astype(int)
    s_yt = np.vstack([tech_str_to_vec(x) for x in sdf["techniques"]])
    s_ya = sdf["action"].values.astype(int)
    sy_yl = sy["label"].values.astype(int)
    sy_yt = np.vstack([tech_str_to_vec(x) for x in sy["techniques"]])
    sy_ya = sy["action"].values.astype(int)

    all_yl = np.concatenate([sy_yl, s_yl, s_yl, dp["y_l"][keep_dp].astype(int)])
    all_yt = np.vstack([sy_yt, s_yt, s_yt, dp["y_t"][keep_dp].astype(int)])
    all_ya = np.concatenate([sy_ya, s_ya, s_ya, dp["y_a"][keep_dp].astype(int)])

    print(f"Fitting Domain-Adapted Linear + LGBM/XGB/CatBoost on {len(all_tx)} rows...", flush=True)
    vec_w = TfidfVectorizer(ngram_range=(1, 2), max_features=85000, sublinear_tf=True, min_df=2, token_pattern=r"(?u)\b\w+\b|[<>\[\]\|]+")
    vec_c = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=55000, sublinear_tf=True, min_df=3)
    X_tw = hstack([vec_w.fit_transform(all_tx), vec_c.fit_transform(all_tx)]).tocsr()

    svd = TruncatedSVD(n_components=128, random_state=42)
    Z_tr = svd.fit_transform(X_tw).astype(np.float32)
    scaler = StandardScaler()
    F_tr = scaler.fit_transform(all_f).astype(np.float32)

    X_lin = hstack([X_tw, csr_matrix(F_tr * 0.25)]).tocsr()
    X_tab = np.hstack([F_tr, Z_tr]).astype(np.float32)

    _, (full_l, full_ts, full_a) = fit_predict_linear_stack(X_lin, all_yl, all_yt, all_ya, [])
    full_gb = fit_gbdt_multitask(X_tab, all_yl, all_yt, all_ya, seed=2026)

    joblib.dump(
        {"vec_w": vec_w, "vec_c": vec_c, "svd": svd, "scaler": scaler, "lin": (full_l, full_ts, full_a), "gb": full_gb},
        CKPT_DIR / "da_tab_v4.joblib",
        compress=3,
    )

    n_te = len(te_f)
    lin_l, lin_t, lin_a = np.zeros((n_te, 6), np.float32), np.zeros((n_te, 6), np.float32), np.zeros((n_te, 6), np.float32)
    lgb_l, xgb_l, cb_l = np.zeros((n_te, 6), np.float32), np.zeros((n_te, 6), np.float32), np.zeros((n_te, 6), np.float32)
    gb_t, gb_a = np.zeros((n_te, 6), np.float32), np.zeros((n_te, 6), np.float32)

    off = 0
    for df_c in pd.read_csv(CACHE_DIR / "test_v4_clean.csv", chunksize=50000):
        m = len(df_c)
        txts = [format_turn_aware(str(x)) for x in df_c["clean_text"].fillna("")]
        X_c = hstack([vec_w.transform(txts), vec_c.transform(txts)]).tocsr()
        F_c = scaler.transform(te_f[off:off + m]).astype(np.float32)
        Z_c = svd.transform(X_c).astype(np.float32)
        X_lc = hstack([X_c, csr_matrix(F_c * 0.25)]).tocsr()
        X_tc = np.hstack([F_c, Z_c]).astype(np.float32)

        lin_l[off:off + m] = full_l.predict_proba(X_lc).astype(np.float32)
        lin_t[off:off + m] = np.column_stack([c.predict_proba(X_lc)[:, 1] for c in full_ts]).astype(np.float32)
        pa_sub = full_a.predict_proba(X_lc).astype(np.float32)
        for ci, cls_id in enumerate(full_a.classes_):
            lin_a[off:off + m, int(cls_id)] = pa_sub[:, ci]

        pl1, px1, pcb1, _, pgt1, pga1 = predict_gbdt(full_gb, X_tc)
        lgb_l[off:off + m], xgb_l[off:off + m], cb_l[off:off + m] = pl1, px1, pcb1
        gb_t[off:off + m], gb_a[off:off + m] = pgt1, pga1
        off += m

    np.savez_compressed(
        OUT_DIR / "test_da_tab_v4.npz",
        lin_l=lin_l, lin_t=lin_t, lin_a=lin_a,
        lgb_l=lgb_l, xgb_l=xgb_l, cb_l=cb_l, gb_t=gb_t, gb_a=gb_a,
    )
    print(f"Saved test_da_tab_v4.npz in {time.time()-t0:.1f}s!", flush=True)

## 8. Zero-Leakage Meta-Calibration, `v8/v9` Adversarial Refinement & Purely Inductive Inference

1. **`run_meta_calibrate_v4`**: Optimizes separate Task-A (`W_L`), Task-B (`W_T`), Task-C (`W_A`) blend weights across all 7 full-coverage families, integrates selective 26k `mDeBERTa-full` escalation, fits a Logistic uncertainty/Brier calibrator on scenario-disjoint OOF folds, and exports `submission_v7_top10.csv`.
2. **`clean_v8` / `clean_v9` + `run_v8_hotfix` + `run_v9_final`**: Repairs newly discovered adversarial failure modes in `test.csv` (removing `"please do not reply"` from `NEG_WARN_RE`, separating `"clearance charge"` prize/job scams from parcel scams, capturing `"select hoyechen"`/`"AnyDesk"`/`"card number + CVV"`/concatenated OTP tokens, removing extreme sample-86 `alpha`/`T1` distortion, re-scoring cleaned rows with `mMiniLM`, and smoothing Brier confidence in `[0.25, 0.985]`).
3. **`run_inductive_inference`**: Strictly inductive raw-text end-to-end inference pipeline satisfying Section 7.3 technical verification.

In [18]:
import sys
import time
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, brier_score_loss
from sklearn.model_selection import StratifiedGroupKFold




def onehot_from_int(y, conf=0.92):
    arr = np.full((len(y), 6), (1.0 - conf) / 5.0, dtype=np.float32)
    arr[np.arange(len(y)), y] = conf
    return arr


def build_uncertainty_features(probs_list, te_feats, rule_lbl):
    """Inductive Brier-calibration feature builder (zero hard 0.985 ceiling)."""
    stack = np.stack(probs_list, axis=0)  # (M, N, 6)
    avg = stack.mean(axis=0)
    sorted_p = np.sort(avg, axis=1)
    p1 = sorted_p[:, -1]
    p2 = sorted_p[:, -2]
    margin = p1 - p2
    ent = -np.sum(avg * np.log(np.clip(avg, 1e-6, 1.0)), axis=1) / np.log(6.0)

    top_pred = avg.argmax(axis=1)
    votes = np.stack([p.argmax(axis=1) == top_pred for p in probs_list], axis=0).mean(axis=0)
    prob_std = stack.std(axis=0)[np.arange(len(avg)), top_pred]
    rule_agree = (rule_lbl == top_pred).astype(np.float32)

    is_noise = te_feats[:, 15].astype(np.float32)
    known_ratio = te_feats[:, 19].astype(np.float32)
    n_turns_norm = np.minimum(te_feats[:, 2], 12.0).astype(np.float32) / 12.0
    return np.column_stack([
        p1, margin, ent, votes, prob_std, rule_agree,
        is_noise, known_ratio, n_turns_norm,
    ]).astype(np.float32)


def run_meta_calibrate_v4():
    import argparse
    
    # ap.add_argument("--out", type=str, default=str(V2_DIR / "submission_v7_top10.csv"))
    out_csv_default = V2_DIR / 'submission_v7_top10.csv'
    t0 = time.time()
    sdf = pd.read_csv(CACHE_DIR / "sample_v4.csv")
    sf = np.load(CACHE_DIR / "sample_v4_feats.npy")
    s_rules = np.load(CACHE_DIR / "sample_v4_rule_pred.npz")

    sy = pd.read_csv(CACHE_DIR / "synth_64k_v4.csv")
    syf = np.load(CACHE_DIR / "synth_64k_v4_feats.npy")
    sy_rules = np.load(CACHE_DIR / "synth_64k_v4_rule_pred.npz")

    # Load all Zero-Leakage Scenario-Disjoint OOF files
    oof_st = np.load(OUT_DIR / "oof_students_v4.npz")
    oof_bg = np.load(OUT_DIR / "oof_bigru_v4.npz")
    oof_ba = np.load(OUT_DIR / "oof_baseline_v4.npz")

    y_s_l = sdf["label"].values.astype(int)
    y_s_t = np.vstack([tech_str_to_vec(x) for x in sdf["techniques"]])
    y_s_a = sdf["action"].values.astype(int)

    y_sy_l = sy["label"].values.astype(int)
    y_sy_t = np.vstack([tech_str_to_vec(x) for x in sy["techniques"]])
    y_sy_a = sy["action"].values.astype(int)
    ood_idx = np.where(oof_ba["ood_mask"])[0]

    s_rule_pl = onehot_from_int(s_rules["lbl"], conf=0.86)
    s_rule_pa = np.zeros((len(sdf), 6), dtype=np.float32)
    for i, a in enumerate(s_rules["act"]):
        if a > 0:
            s_rule_pa[i, a] = 0.85
            for c in range(1, 6):
                if c != a:
                    s_rule_pa[i, c] = 0.0375
    s_rule_pt = s_rules["tvec"].astype(np.float32) * 0.85 + 0.07

    # Order of 7 Full-Coverage Diverse Student+Tab+Rule Families:
    # [0:mMiniLM, 1:mE5small, 2:BGE-glossed, 3:BiGRU-v5, 4:Linear-TFIDF, 5:GBDT-SVD, 6:Rules-v5]
    s_L_models = [
        oof_st["s_mminilm_l"], oof_st["s_me5small_l"], oof_st["s_bge_l"], oof_bg["s_l"],
        oof_ba["s_lin_l"], oof_ba["s_gb_l"], s_rule_pl,
    ]
    s_T_models = [
        oof_st["s_mminilm_t"], oof_st["s_me5small_t"], oof_st["s_bge_t"], oof_bg["s_t"],
        oof_ba["s_lin_t"], oof_ba["s_gb_t"], s_rule_pt,
    ]
    s_A_models = [
        oof_st["s_mminilm_a"], oof_st["s_me5small_a"], oof_st["s_bge_a"], oof_bg["s_a"],
        oof_ba["s_lin_a"], oof_ba["s_gb_a"], s_rule_pa,
    ]

    sy_rule_pl = onehot_from_int(sy_rules["lbl"][ood_idx], conf=0.82)
    sy_rule_pa = np.zeros((len(ood_idx), 6), dtype=np.float32)
    for i, a in enumerate(sy_rules["act"][ood_idx]):
        if a > 0:
            sy_rule_pa[i, a] = 0.85
            for c in range(1, 6):
                if c != a:
                    sy_rule_pa[i, c] = 0.0375
    sy_rule_pt = sy_rules["tvec"][ood_idx].astype(np.float32) * 0.82 + 0.08

    sy_L_models = [
        oof_st["sy_mminilm_l"][ood_idx], oof_st["sy_me5small_l"][ood_idx], oof_st["sy_bge_l"][ood_idx], oof_bg["sy_l"][ood_idx],
        oof_ba["ood_lin_l"], oof_ba["ood_gb_l"], sy_rule_pl,
    ]
    sy_T_models = [
        oof_st["sy_mminilm_t"][ood_idx], oof_st["sy_me5small_t"][ood_idx], oof_st["sy_bge_t"][ood_idx], oof_bg["sy_t"][ood_idx],
        oof_ba["ood_lin_t"], oof_ba["ood_gb_t"], sy_rule_pt,
    ]
    sy_A_models = [
        oof_st["sy_mminilm_a"][ood_idx], oof_st["sy_me5small_a"][ood_idx], oof_st["sy_bge_a"][ood_idx], oof_bg["sy_a"][ood_idx],
        oof_ba["ood_lin_a"], oof_ba["ood_gb_a"], sy_rule_pa,
    ]

    def blend_list(mlist, w):
        out = np.zeros_like(mlist[0])
        for wi, m in zip(w, mlist):
            out += wi * m
        return out

    def opt_task_weights(s_list, sy_list, y_s, y_sy, mode="mc", w_prior=None):
        k = len(s_list)
        w0 = np.array(w_prior if w_prior is not None else [0.24, 0.22, 0.16, 0.10, 0.08, 0.10, 0.10], dtype=np.float64)
        w0 /= w0.sum()
        rng = np.random.RandomState(42)
        best_w = w0.copy()
        best_sc = -1.0
        scam_s = np.isin(y_s_l, [1, 2, 3, 4])
        scam_o = np.isin(y_sy_l[ood_idx], [1, 2, 3, 4])
        for it in range(1500):
            cand = w0 if it == 0 else np.maximum(0.02, best_w + rng.normal(0, 0.04, size=k))
            cand /= cand.sum()
            ps = blend_list(s_list, cand)
            po = blend_list(sy_list, cand)
            if mode == "mc":
                fs = 100.0 * f1_score(y_s, ps.argmax(1), average="macro", zero_division=0)
                fo = 100.0 * f1_score(y_sy, po.argmax(1), average="macro", zero_division=0)
            elif mode == "tech":
                fs = 100.0 * f1_score(y_s[scam_s], (ps[scam_s] >= 0.45).astype(int), average="macro", zero_division=0)
                fo = 100.0 * f1_score(y_sy[scam_o], (po[scam_o] >= 0.45).astype(int), average="macro", zero_division=0)
            else:
                fs = 100.0 * f1_score(y_s[scam_s], 1 + ps[scam_s, 1:].argmax(1), average="macro", zero_division=0)
                fo = 100.0 * f1_score(y_sy[scam_o], 1 + po[scam_o, 1:].argmax(1), average="macro", zero_division=0)
            sc = 0.65 * fs + 0.35 * fo
            if sc > best_sc:
                best_sc = sc
                best_w = cand
        return best_w, best_sc

    w_l_stu, sc_l = opt_task_weights(s_L_models, sy_L_models, y_s_l, y_sy_l[ood_idx], mode="mc")
    w_t_stu, sc_t = opt_task_weights(s_T_models, sy_T_models, y_s_t, y_sy_t[ood_idx], mode="tech", w_prior=[0.22, 0.22, 0.14, 0.10, 0.06, 0.08, 0.18])
    w_a_stu, sc_a = opt_task_weights(s_A_models, sy_A_models, y_s_a, y_sy_a[ood_idx], mode="act", w_prior=[0.22, 0.22, 0.14, 0.10, 0.06, 0.08, 0.18])
    w_l_esc = np.append((1.0 - 0.38) * w_l_stu, 0.38)
    print(f"Task A OOD F1={sc_l:.2f} | W_L={np.round(w_l_stu, 3)}")
    print(f"Task B OOD F1={sc_t:.2f} | W_T={np.round(w_t_stu, 3)}")
    print(f"Task C OOD F1={sc_a:.2f} | W_A={np.round(w_a_stu, 3)}")

    p_s_l = blend_list(s_L_models, w_l_stu)
    p_o_l = blend_list(sy_L_models, w_l_stu)
    p_s_t = blend_list(s_T_models, w_t_stu)
    p_o_t = blend_list(sy_T_models, w_t_stu)
    p_s_a = blend_list(s_A_models, w_a_stu)
    p_o_a = blend_list(sy_A_models, w_a_stu)

    # Regularized Task A class-prior multipliers (bounded [0.88, 1.15] via GroupKFold OOD safety)
    rng = np.random.RandomState(123)
    best_alpha = np.ones(6, dtype=np.float32)
    best_fa_joint = 0.65 * f1_score(y_s_l, p_s_l.argmax(1), average="macro") + 0.35 * f1_score(y_sy_l[ood_idx], p_o_l.argmax(1), average="macro")
    for _ in range(1000):
        a = np.clip(best_alpha + rng.normal(0, 0.03, size=6), 0.88, 1.15).astype(np.float32)
        sc = 0.65 * f1_score(y_s_l, (p_s_l * a).argmax(1), average="macro") + 0.35 * f1_score(y_sy_l[ood_idx], (p_o_l * a).argmax(1), average="macro")
        if sc > best_fa_joint:
            best_fa_joint = sc
            best_alpha = a

    # Regularized Task B thresholds per T1..T6 in [0.36, 0.56]
    tech_th = np.full(6, 0.45, dtype=np.float32)
    for j in range(6):
        scam_s = np.isin(y_s_l, [1, 2, 3, 4])
        scam_o = np.isin(y_sy_l[ood_idx], [1, 2, 3, 4])
        b_th, b_f = 0.45, -1.0
        for th in np.linspace(0.36, 0.56, 21):
            fs = f1_score(y_s_t[scam_s, j], (p_s_t[scam_s, j] >= th).astype(int), zero_division=0)
            fo = f1_score(y_sy_t[ood_idx][scam_o, j], (p_o_t[scam_o, j] >= th).astype(int), zero_division=0)
            if 0.65 * fs + 0.35 * fo > b_f:
                b_f = 0.65 * fs + 0.35 * fo
                b_th = th
        tech_th[j] = b_th

    # Regularized Task C action multipliers [1..5] in [0.85, 1.45] to recover action=5/1/4
    best_beta = np.ones(6, dtype=np.float32)
    scam_s = np.isin(y_s_l, [1, 2, 3, 4])
    scam_o = np.isin(y_sy_l[ood_idx], [1, 2, 3, 4])
    best_fc_joint = (
        0.65 * f1_score(y_s_a[scam_s], 1 + p_s_a[scam_s, 1:].argmax(1), average="macro", zero_division=0)
        + 0.35 * f1_score(y_sy_a[ood_idx][scam_o], 1 + p_o_a[scam_o, 1:].argmax(1), average="macro", zero_division=0)
    )
    for _ in range(1200):
        b = np.clip(best_beta + rng.normal(0, 0.05, size=6), 0.85, 1.45).astype(np.float32)
        b[0] = 1.0
        sc = (
            0.65 * f1_score(y_s_a[scam_s], 1 + (p_s_a[scam_s, 1:] * b[1:]).argmax(1), average="macro", zero_division=0)
            + 0.35 * f1_score(y_sy_a[ood_idx][scam_o], 1 + (p_o_a[scam_o, 1:] * b[1:]).argmax(1), average="macro", zero_division=0)
        )
        if sc > best_fc_joint:
            best_fc_joint = sc
            best_beta = b

    print(f"Tuned regularizers: alpha={np.round(best_alpha, 3)} | tech_th={np.round(tech_th, 3)} | act_beta={np.round(best_beta[1:], 3)}")

    # Fit honest Zero-Leakage OOD Brier Calibrator (Logistic on OOF correctness!)
    X_cal_s = build_uncertainty_features(s_L_models, sf, s_rules["lbl"])
    X_cal_o = build_uncertainty_features(sy_L_models, syf[ood_idx], sy_rules["lbl"][ood_idx])
    y_corr_s = ((p_s_l * best_alpha).argmax(1) == y_s_l).astype(int)
    y_corr_o = ((p_o_l * best_alpha).argmax(1) == y_sy_l[ood_idx]).astype(int)

    # Weight sample OOF slightly stronger + synth OOD OOF for broad coverage of errors/OOV/noise
    X_cal = np.vstack([np.repeat(X_cal_s, 20, axis=0), X_cal_o])
    y_cal = np.concatenate([np.repeat(y_corr_s, 20), y_corr_o])
    cal_lr = LogisticRegression(C=0.5, max_iter=500)
    cal_lr.fit(X_cal, y_cal)

    # Evaluate calibrated OOD Brier & Composite Section 10 proxy!
    conf_s = np.clip(cal_lr.predict_proba(X_cal_s)[:, 1], 0.05, 0.995)
    conf_o = np.clip(cal_lr.predict_proba(X_cal_o)[:, 1], 0.05, 0.995)
    spl, spt, spa = decode_consistent(p_s_l * best_alpha, p_s_t, p_s_a * best_beta, tech_th=tech_th)
    opl, opt, opa = decode_consistent(p_o_l * best_alpha, p_o_t, p_o_a * best_beta, tech_th=tech_th)
    multi_o = syf[ood_idx, 2] >= 3
    final_scores = score_section10_proxy(
        y_s_l, spl, y_s_t, spt, y_s_a, spa, conf_s,
        y_l_ood=y_sy_l[ood_idx], p_l_ood=opl, conf_ood=conf_o,
        y_l_adv=y_sy_l[ood_idx], p_l_adv=opl, conf_adv=conf_o,
        y_l_multi=y_sy_l[ood_idx][multi_o], p_l_multi=opl[multi_o],
    )
    print("\n=== FINAL ZERO-LEAKAGE META-ENSEMBLE SECTION 10 PROXY ===", final_scores, flush=True)

    # ---- APPLY TO 500K TEST PREDICTIONS (WITH DOMAIN-ADAPTED STUDENTS + SELECTIVE INDUCTIVE ESCALATION) ----
    t_m = np.load(OUT_DIR / "test_mminilm_v4.npz")
    t_e = np.load(OUT_DIR / "test_me5small_v4.npz")
    t_b = np.load(OUT_DIR / "test_bge_v4.npz")
    t_bg = np.load(OUT_DIR / "test_bigru_v4.npz")
    da_path = OUT_DIR / "test_da_tab_v4.npz"
    t_ba = np.load(da_path if da_path.exists() else (OUT_DIR / "test_baseline_v4.npz"))
    print("Loaded tabular/linear predictions from:", da_path.name if da_path.exists() else "test_baseline_v4.npz")
    t_rules = np.load(CACHE_DIR / "test_v4_rule_pred.npz")
    te_feats = np.load(CACHE_DIR / "test_v4_feats.npy", mmap_mode="r")
    esc_path = OUT_DIR / "test_mdeberta_escalate_v4.npz"
    t_esc = np.load(esc_path) if esc_path.exists() else None

    n_te = len(te_feats)
    t_rule_pl = onehot_from_int(t_rules["lbl"], conf=0.84)
    t_rule_pt = t_rules["tvec"].astype(np.float32) * 0.84 + 0.08
    t_rule_pa = np.zeros((n_te, 6), dtype=np.float32)
    t_acts = t_rules["act"].astype(int)
    pos_a = t_acts > 0
    t_rule_pa[pos_a, :] = 0.0375
    t_rule_pa[np.where(pos_a)[0], t_acts[pos_a]] = 0.85

    t_gb_l = 0.40 * t_ba["lgb_l"] + 0.35 * t_ba["xgb_l"] + 0.25 * t_ba["cb_l"]
    t_L_stu = [t_m["p_l"], t_e["p_l"], t_b["p_l"], t_bg["p_l"], t_ba["lin_l"], t_gb_l, t_rule_pl]
    t_T_stu = [t_m["p_t"], t_e["p_t"], t_b["p_t"], t_bg["p_t"], t_ba["lin_t"], t_ba["gb_t"], t_rule_pt]
    t_A_stu = [t_m["p_a"], t_e["p_a"], t_b["p_a"], t_bg["p_a"], t_ba["lin_a"], t_ba["gb_a"], t_rule_pa]

    # Non-escalated predictions across all 500k
    P_L = blend_list(t_L_stu, w_l_stu)
    P_T = blend_list(t_T_stu, w_t_stu)
    P_A = blend_list(t_A_stu, w_a_stu)

    # Selective Inductive Escalation: blend escalated mDeBERTa-v3-base logits on top uncertain rows!
    if t_esc is not None:
        ei = t_esc["esc_idx"]
        P_L_esc_row = (1.0 - w_l_esc[-1]) * P_L[ei] + w_l_esc[-1] * t_esc["p_l"]
        P_T_esc_row = (1.0 - w_l_esc[-1]) * P_T[ei] + w_l_esc[-1] * t_esc["p_t"]
        P_A_esc_row = (1.0 - w_l_esc[-1]) * P_A[ei] + w_l_esc[-1] * t_esc["p_a"]
        P_L[ei], P_T[ei], P_A[ei] = P_L_esc_row, P_T_esc_row, P_A_esc_row
        print(f"Applied mDeBERTa-full escalation on {len(ei)} uncertain rows.", flush=True)
    else:
        print("Running pre-escalation 7-family ensemble (mDeBERTa escalation file not ready yet).", flush=True)

    # Calibrate confidence in chunks for low RAM
    conf_te = np.zeros(n_te, dtype=np.float32)
    for s in range(0, n_te, 50000):
        e = min(n_te, s + 50000)
        chunk_L = [m[s:e] for m in t_L_stu]
        Xc = build_uncertainty_features(chunk_L, te_feats[s:e], t_rules["lbl"][s:e])
        conf_te[s:e] = np.clip(cal_lr.predict_proba(Xc)[:, 1], 0.05, 0.996).astype(np.float32)

    final_l, final_t, final_a = decode_consistent(P_L * best_alpha, P_T, P_A * best_beta, tech_th=tech_th)
    ckpt_dir = CACHE_DIR / "ckpt_v4"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(
        {"cal_lr": cal_lr, "w_l_stu": w_l_stu, "w_t_stu": w_t_stu, "w_a_stu": w_a_stu, "w_l_esc": w_l_esc, "best_alpha": best_alpha, "tech_th": tech_th, "best_beta": best_beta},
        ckpt_dir / "meta_calibrator_v4.joblib",
    )
    te_ids = pd.read_csv(CACHE_DIR / "test_v4_clean.csv", usecols=["id"])["id"].values
    sub = pd.DataFrame({
        "id": te_ids,
        "label": final_l,
        "techniques": [vec_to_tech_str(final_t[i]) for i in range(n_te)],
        "action": final_a,
        "confidence": np.round(conf_te, 4),
    })
    out_csv = out_csv_default
    sub.to_csv(out_csv, index=False)
    print(f"Saved final submission {out_csv} ({len(sub)} rows) in {time.time()-t0:.1f}s!", flush=True)

In [19]:
"""Surgical V8 Adversarial Normalization, Priority-Fixed Rules, and Fast Re-Scoring Hotfix."""
import re
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer



NOISE_V8_RE = re.compile(
    r"(?:zzqwpo|wkrjfejejrn|mnbvcx?|qzxvbn|xkcdpl|asdkjh|lkjsdfoiu|xcvbnm|zzqpwoeiru|qundvpyfjqx|wbmtzmh|qwertyz|asdfghj)",
    flags=re.IGNORECASE,
)

TYPO_V8 = {
    "don07": "don't", "d0n07": "don't", "d0n7": "don't", "don0t": "don't", "d0nt": "don't",
    "b4nk": "bank", "s3cur1ty": "security", "53curity": "security", "cl34r4nc3": "clearance",
    "r3del1v3ry": "redelivery", "r3d3l1v3ry": "redelivery", "1mp0r7an7": "important",
    "0rd3r": "order", "r3g1s7r4710n": "registration", "c4rd": "card", "crad": "card",
    "cradnumber": "card number", "cardnumber": "card number", "cvvdin": "cvv din",
    "p4ssw0rd": "password", "l0g1n": "login", "0tp": "otp", "p1n": "pin", "pn": "pin",
    "oepn": "open", "d3n": "den", "w45": "was", "p4y": "pay", "py": "pay", "olny": "only",
    "c4ller": "caller", "s3nd3r": "sender", "7el374lk": "teletalk", "dibenna": "diben na",
    "uilever": "unilever", "gramenphone": "grameenphone", "cnograts": "congrats", "congras": "congrats",
    "sahre": "share", "motrbike": "motorbike", "tranaction": "transaction", "trnasaction": "transaction",
    "debennna": "deben na", "debenna": "deben na", "bolbenna": "bolben na",
    "korbenna": "korben na", "sharekorbenna": "share korben na", "dibina": "dibi na",
    "sharethe": "share the", "tashare": "ta share", "rewardsahre": "reward share",
    "justreceived": "just received", "kyychas": "kyc has", "kychas": "kyc has", "kcy": "kyc",
    "coodeaschhe": "code ashche", "cooodeaschhe": "code ashche", "codeaschhe": "code ashche",
    "pohne": "phone", "nubmer": "number", "oa": "ota", "hoyehcen": "hoyechen",
    "samsungs24": "samsung s24", "wintersae": "winter sale", "takadaly": "taka daily",
    "apnia": "apni a", "erjonno": "er jonno", "bkashkorun": "bkash korun",
    "chargetk": "charge tk", "rfeer": "refer", "j173chen": "jitechen",
    "amotrbike": "a motorbike", "motorbikeer": "motorbike er", "onparcel": "on parcel",
    "incompletepay": "incomplete pay", "messaed": "messaged", "neds": "needs",
    "kimcash": "ki mcash", "apiki": "apni ki", "usekoren": "use koren",
    "froma": "from a", "helplabe": "help lagbe", "f1x": "fix", "korte": "korte",
    "kor3": "kore", "codeta": "code ta", "sedn": "send", "annnnd": "and", "dte": "date",
    "yoooou": "you", "taaaakadaly": "taka daily", "sqqqquarepharma": "squarepharma",
    "lokeo": "lokeo", "anydesk": "anydesk", "user5": "users", "bkah": "bkash", "bksah": "bkash",
    "sh4re": "share", "c0d3": "code", "ver1fy": "verify", "4ccount": "account", "4pnar": "apnar",
    "w1n": "win", "pr1ze": "prize", "cl4im": "claim", "del1very": "delivery", "p4rcel": "parcel",
    "ch4rge": "charge", "f33": "fee", "t4ka": "taka", "f3r0t": "ferot", "v1s4": "visa",
}
TYPO_V8_RE = re.compile(r"\b(" + "|".join(re.escape(k) for k in sorted(TYPO_V8, key=len, reverse=True)) + r")\b", re.I)

SINGLE_FIX = {
    "abbrod": "abroad", "abbroad": "abroad", "abbrooad": "abroad", "opppen": "open", "opppenn": "open",
    "wittth": "with", "witth": "with", "llluucky": "lucky", "luucky": "lucky",
    "llucky": "lucky", "sshhare": "share", "ssend": "send", "hhelllo": "hello",
    "hhhello": "hello", "arrrre": "are", "arre": "are", "taakaa": "taka", "ttaka": "taka",
}


def clean_v8(raw_text: str) -> str:
    if not isinstance(raw_text, str):
        raw_text = "" if raw_text is None else str(raw_text)
    s = ZW_RE.sub("", raw_text)
    s = s.translate(CONFUSABLES)
    # Collapse inline URL defanging first
    s = re.sub(r"\bh[t7x]{2}p(s?)://", r"http\1://", s, flags=re.IGNORECASE)
    s = s.replace("[.]", ".")
    # Strip noise strings anywhere
    s = NOISE_V8_RE.sub(" ", s)
    # Replace emojis with clean spaces for rule checks
    s = EMOJI_RE.sub(" ", s)
    s = ANY_EMOJI_RE.sub(" ", s)
    # Normalize unicode apostrophes and lowercase
    s = s.replace("\u2019", "'").replace("\u2018", "'").lower()
    # Collapse 3+ repeated letters down to 2, and fix common double-collapsed words
    s = re.sub(r"([a-z])\1{2,}", r"\1\1", s)
    # Common typo/leet dictionary passes (2 passes in case double->single reveals token)
    s = TYPO_V8_RE.sub(lambda m: TYPO_V8[m.group(1).lower()], s)
    toks = [SINGLE_FIX.get(t, t) for t in s.split()]
    s = " ".join(toks)
    # Split concatenated compound tokens observed in adversarial tests
    s = re.sub(r"(card|pin|otp|cvv|nid|code)(number|din|den|ta|share|pathan|bolo|bolen|ashche)", r"\1 \2", s)
    s = re.sub(r"(share|send|reply|enter|confirm|pay|open)(the|your|with|tk)", r"\1 \2", s)
    s = TYPO_V8_RE.sub(lambda m: TYPO_V8[m.group(1).lower()], s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


# Fixed Priority-Aware Rule Patterns
SENS_V8_RE = re.compile(
    r"\b(?:pin|otp|password|passcode|cvv|card number|card details|nid|verification code|login code|security code|"
    r"6[- ]?digit(?: code)?|9[- ]?digit(?: code)?|\d[- ]digit code|date of birth|dob|internet banking password|wallet pin|"
    r"anydesk|teamviewer|screen\s*shar(?:e|ing))\b"
    r"|ওটিপি|পিন|পাসওয়ার্ড|এনআইডি|কার্ড নম্বর",
    flags=re.IGNORECASE,
)
CODE_PLAIN_RE = re.compile(r"\bcode\b|কোড", flags=re.IGNORECASE)
ASK_V8_RE = re.compile(
    r"\b(?:send|sending|share|sharing|give|giving|tell|reply|reply with|provide|enter|entering|submit|forward|"
    r"confirm|verify|verify by|read out|open|install|bolen|bolo|bolun|pathan|pathaben|din|den|dao|deben|dibi|ta bolen|ta din|dite hobe|bolte hobe)\b"
    r"|জানান|পাঠান|দিন|বলুন|বলো|দেবেন|দাও|পাঠাও",
    flags=re.IGNORECASE,
)
# CRITICAL FIX: Removed "do not reply / don't reply / please do not reply" so scammer footers don't block rules!
NEG_WARN_V8_RE = re.compile(
    r"\b(?:only with (?:the )?rider|trip pin|do not share|don'?t share|never share|not share|never ask|never call|will never|"
    r"do not pay anyone|reply stop to opt out|"
    r"kauke\b.{0,18}\b(?:share|bolben|korben|diben|dien|bolo)\b.{0,6}\bna|"
    r"(?:bolben|dibi|dien|diben|korben)\s+na|bank er lokeo na)\b"
    r"|কাউকে.{0,22}(?:দেবেন|করবেন|বলবেন|শেয়ার).{0,6}না|কখনো.{0,12}না",
    flags=re.IGNORECASE,
)

# CRITICAL FIX: "clearance charge" removed from standalone PARCEL_V8_RE (must have courier/parcel/customs/delivery!)
PARCEL_V8_RE = re.compile(
    r"\b(?:parcel|package|courier|customs|delivery|deliver|shipment|consignment|tracking|redelivery|storage fee|delivery attempt|out for delivery|was delivered|has been delivered)\b"
    r"|পার্সেল|কুরিয়ার|কাস্টমস|শুল্ক|ডেলিভারি হবে|ডেলিভারি ব্যর্থ|ডেলিভারি চার্জ",
    flags=re.IGNORECASE,
)
PARCEL_PHISH_V8_RE = re.compile(
    r"\b(?:customs|duty|storage fee|redelivery|clearance charge|wrong address|incomplete|address confirmation|confirm(?: new)? address|"
    r"update address|address.{0,12}(?:vul|bhul)|(?:vul|bhul).{0,10}address|"
    r"could not be delivered|couldn'?t be delivered|attempt failed|release|"
    r"held|atke ase|failed|fee|charge|(?<!pathao\s)pay|click|tap|<URL_SUSP>|<URL_SHORT>|<URL>)\b"
    r"|আটকে|ব্যর্থ|চার্জ|ফি|পরিশোধ|শুল্ক|কাস্টমস|ঠিকানা আপডেট|ক্লিক|লিংকে",
    flags=re.IGNORECASE,
)

STRONG_PRIZE_V8_RE = re.compile(
    r"\b(?:you won|you'?ve won|won|winner|winners|win a (?:tk|voucher|prize|gift|iphone|smartphone|car)|jitechen|jitechhen|jiteche|jitar|lucky draw|lottery|giveaway|airdrop|usdt|"
    r"mystery box|spin and win|free iphone|free smartphone|free router|free wi[- ]?fi router|10gb data gift|"
    r"pm education fund|stipend|old[- ]?age allowance|allowance arrears|bumper|"
    r"(?:number\s*ti|motorbike|laptop|toyota\s*car|iphone\s*\d*|smart\s*tv|samsung\s*s?\d*|gift\s*card|prize|bonus|gift|cashback|stipend|allowance|reward).{0,45}select(?:ed)?\s*(?:hoyechen|hoyeche|hoye|hoy|for)|"
    r"selected\s*(?:for|at)\s*(?:a\s+)?(?:motorbike|laptop|smart\s*tv|toyota|iphone|samsung|gift\s*card|reward|prize)|"
    r"loyalty reward|prize|cashback.{0,15}(?:jitechen|won|claim))\b"
    r"|জিতেছেন|জিতেছে|জিতার|লাকি ড্র|লটারি|পুরস্কার",
    flags=re.IGNORECASE,
)
SOFT_PRIZE_V8_RE = re.compile(
    r"\b(?:cashback|gift|reward|rewards|bonus|eid bonus|congratulations|congrats|select\s*hoyechen|selected\s*at)\b"
    r"|ক্যাশব্যাক|উপহার|অভিনন্দন|ঈদ বোনাস|গিফট",
    flags=re.IGNORECASE,
)
JOB_V8_RE = re.compile(
    r"(?<!amar\s)(?<!my\s)\bjob\b(?!\s*jabe)|\b(?:hiring|vacancy|work[- ]?from[- ]?home|work frmo home|part[- ]?time|data entry|package handler|brand ambassador|"
    r"relief volunteer|typing job|agent license|earn(?:ing)?|income|kamai|kamao|salary|beton|task|tasks|watch and like|"
    r"like korle|facebook page|page like|per like|product review|review products|unlock tasks?|"
    r"commission|invest|investment|double your money|crypto signals?|arbitrage|bot|ludo|cash game|"
    r"freelanc(?:ing|e)|abroad job|trading crypto|selected at (?:unilever|walton|grameenphone|beximco|samsung bangladesh|pathao|robi|foodpanda|pran))\b"
    r"|লোক নিচ্ছে|ঘরে বসে|আয়|বেতন|বিনিয়োগ|দ্বিগুণ|ভাতা|হোয়াটসঅ্যাপে যোগ দিন",
    flags=re.IGNORECASE,
)
JOB_TRAP_V8_RE = re.compile(
    r"\b(?:fee|deposit|security deposit|uniform fee|registration fee|clearance charge|service charge|visa fee|kit|"
    r"id card|invest|enroll now|per task|per post|per day|per like|like e \d+|per review|daily|weekly profit|profit per day|"
    r"guarantees?\s+\d+%|deposit tk \d+.{0,35}get tk \d+|10 minutes\.\s*guaranteed|commission|dite hobe|unlock korun|apply korun|proceed|"
    r"join (?:whatsapp|telegram|the group|vip)|group e join|whatsapp e join|join korun|whatsapp us|complete \d+ tasks?)\b"
    r"|রেজিস্ট্রেশন ফি|ডিপোজিট|দ্বিগুণ|নিশ্চিত লাভ|প্রতিদিন|যোগ দিন|যোগাযোগ করুন|সার্ভিস চার্জ|ভিসা ফি|টাকা দিন|হোয়াটসঅ্যাপে|টেলিগ্রামে",
    flags=re.IGNORECASE,
)
HARD_PRIZE_RE = re.compile(r"\b(?:select(?:ed)?\s*(?:hoyechen|for\s+a)|reward|won|lottery|lucky draw)\b|লাকি ড্র|লটারি", re.I)


def extract_rule_v8(raw_text: str):
    c = clean_v8(raw_text)
    masked, uflags = mask_urls_for_text_rules(c)
    t = SPEAKER_PREFIX_RE.sub(" ", masked)

    scam_warn_thread = bool(SCAM_WARN_THREAD_RE.search(t))
    mistake = bool(MISTAKE_RE.search(t)) and bool(RETURN_RE.search(t)) and bool(
        re.search(r"\b(?:tk|bdt|taka|money)\b|টাকা|\d{2,}", t, re.I)
    )

    req_secret = False
    for seg in re.split(r"[.!?\n\|]+", t):
        seg = seg.strip()
        if not seg:
            continue
        has_sens = bool(SENS_V8_RE.search(seg)) or (
            bool(CODE_PLAIN_RE.search(seg))
            and not bool(re.search(r"\b(?:use code|code:|promo|voucher|eid\d+)\b", seg, re.I))
        )
        if has_sens and ASK_V8_RE.search(seg) and not NEG_WARN_V8_RE.search(seg):
            if not HARD_LEGIT_HR_DOC_RE.search(seg):
                req_secret = True
                break
    if not req_secret:
        for seg in re.split(r"[.!?\n\|]+", t):
            seg = seg.strip()
            if seg and not NEG_WARN_V8_RE.search(seg) and re.search(
                r"(?:nid.{0,15}both sides|photo of your nid|nid photo|card number.{0,25}cvv|cvv den|cvv din|"
                r"enter(?:ing)? (?:your )?(?:pin|otp)|confirm(?: your)? (?:internet banking )?(?:password|otp|pin)|"
                r"send (?:your )?(?:pin|otp|nid|wallet pin)|tell me your pin|read out the code|"
                r"anydesk.{0,30}(?:code|digit)|(?:whatsapp|telegram|account|sim).{0,25}deactivated.{0,30}(?:code|otp)|"
                r"phone hariye.{0,45}code.{0,20}(?:bolo|bolen|din|pathan)|"
                r"share\s+(?:the\s+)?(?:\d[- ]?digit\s+)?(?:otp|pin|code))",
                seg,
                re.I,
            ):
                req_secret = True
                break

    neg_warn = bool(NEG_WARN_V8_RE.search(t))
    quoted_scam_framing = bool(
        ("|||" in c and scam_warn_thread)
        or re.search(r"(?:someone|ekta sms|ekjon).{0,60}(?:messaged|ashche|pathaise|je)|that is phishing|eta scam|keu pin chaile", t, re.I)
    )
    is_scamwarn_only = bool(scam_warn_thread and (not req_secret or quoted_scam_framing))
    is_cod = bool(HARD_LEGIT_COD_RE.search(t)) and not req_secret
    is_hr_legit = (
        bool(HARD_LEGIT_HR_DOC_RE.search(t))
        and not req_secret
        and not bool(re.search(r"\b(?:pay tk|registration fee|uniform fee|deposit)\b", t, re.I))
    )
    is_tx_legit = (
        bool(HARD_LEGIT_TX_RE.search(t))
        and not req_secret
        and not mistake
        and not bool(re.search(r"\b(?:verify|claim|click|fee|charge|prosessing)\b|প্রসেসিং|ক্লেইম|<URL_SUSP>|<URL_SHORT>", t, re.I))
    )
    is_otp_legit = (bool(HARD_LEGIT_OTP_RE.search(t)) or bool(re.search(r"\b(?:login code|verification code|pin|otp)\b", t, re.I))) and neg_warn and not req_secret
    is_deliv_legit = (
        bool(HARD_LEGIT_DELIV_RE.search(t))
        and not req_secret
        and not (PARCEL_PHISH_V8_RE.search(t) and not bool(re.search(r"will retry tomorrow|096\d{2}-", t, re.I)))
    )
    is_chat_legit = bool(HARD_LEGIT_CHAT_RE.search(t)) and not req_secret and not mistake
    hard_legit = is_cod or is_hr_legit or is_tx_legit or is_otp_legit or is_deliv_legit or is_chat_legit

    has_parcel = bool(PARCEL_V8_RE.search(t))
    has_parcel_phish = bool(PARCEL_PHISH_V8_RE.search(t))
    has_job = bool(JOB_V8_RE.search(t))
    has_job_trap = bool(JOB_TRAP_V8_RE.search(t))
    has_claim = bool(CLAIM_RE.search(t) or re.search(r"\b(?:charge|fee)\s*(?:pathan|din|den|dite)\b", t, re.I))
    has_promo = bool(PROMO_RE.search(t))

    has_strong_prize = bool(STRONG_PRIZE_V8_RE.search(t))
    has_soft_prize = bool(SOFT_PRIZE_V8_RE.search(t)) and has_claim and not (has_parcel and has_parcel_phish) and not (has_job and has_job_trap)
    has_prize = has_strong_prize or has_soft_prize

    if is_scamwarn_only:
        lbl = 0
    elif req_secret or mistake:
        lbl = 1
    elif hard_legit:
        lbl = 0
    elif has_promo and not (
        (has_parcel and has_parcel_phish) or (has_job and has_job_trap) or uflags["has_susp_tld"] or uflags["has_shortener"]
        or req_secret or has_strong_prize or re.search(r"\b(?:processing fee|survey|airdrop)\b|প্রসেসিং ফি", t, re.I)
    ):
        lbl = 5
    elif has_prize and (has_claim or uflags["has_url"]) and not (has_parcel and has_parcel_phish and (not HARD_PRIZE_RE.search(t) or bool(re.search(r"limited winners", t, re.I)))) and not (has_job and has_job_trap):
        lbl = 2
    elif has_parcel and (has_parcel_phish or uflags["has_susp_tld"] or uflags["has_shortener"]) and not re.search(r"\b(?:package handler|salary|ambassador)\b", t, re.I):
        lbl = 4
    elif has_job and has_job_trap:
        lbl = 3
    elif has_promo:
        lbl = 5
    else:
        lbl = 0

    if lbl in (0, 5):
        act = 0
        tvec = np.zeros(6, dtype=np.int32)
    else:
        req_pii_reply = bool(re.search(r"\b(?:reply|send|share|give)\b.{0,25}\b(?:name and address|full name|address|date of birth|dob)\b", t, re.I))
        has_pay = bool(re.search(
            r"\b(?:(?<!pathao\s)pay|deposit|invest|ferot|return|send (?:it )?back|reverse|enroll now with tk|charge pathan|fee din)\b"
            r"|\b(?:registration|processing|delivery|redelivery|storage|service|clearance)\s*(?:fee|charge|duty)\b.{0,22}\b(?:tk|bdt|\d|dite|(?<!pathao\s)pay|din|baki|pathan)"
            r"|\b(?:tk|bdt|\$)\s*[\d,]+\s*(?:fee|duty|deposit|charge|ferot|dite hobe|bkash korun|nagad korun|din|pathan)\b"
            r"|পরিশোধ|ডিপোজিট|বিনিয়োগ|ফেরত|(?:টাকা|ফি|চার্জ)\s*(?:দিন|দিতে হবে)",
            t, re.I,
        ))
        has_pay_verb = bool(re.search(r"\b(?:(?<!pathao\s)pay|deposit|invest|enroll now with|ferot|send (?:it )?back|reverse|bkash korun|nagad korun|charge pathan|fee din)\b|পরিশোধ|ডিপোজিট|বিনিয়োগ|ফেরত|দিতে হবে|টাকা দিন", t, re.I))
        has_join_apk = bool(re.search(
            r"\b(?:install|download|join (?:the )?(?:group|vip|whatsapp|telegram)|whatsapp e join|telegram e join|group e join|join korun|"
            r"complete \d+ tasks?|watch and like|buy products|review products|rewards app|relief volunteer)\b|<URL_APK>|<URL_TGWA>"
            r"|যোগ দিন|হোয়াটসঅ্যাপে যোগ|টেলিগ্রামে যোগ|গ্রুপে",
            t, re.I,
        ))
        has_url_click = bool(uflags["has_url"] or re.search(r"\b(?:click|tap|link|visit|survey|connect (?:your )?wallet)\b|ক্লিক|লিংকে|যান", t, re.I))
        has_call_wa = bool(re.search(r"\b(?:call|whatsapp|contact|dial)\b|যোগাযোগ|কল করুন", t, re.I))

        if lbl == 1:
            act = 1 if (req_secret or not mistake) else 2
        elif req_pii_reply and not has_pay_verb:
            act = 1
        elif has_call_wa and not has_pay_verb and not has_join_apk and not has_url_click:
            act = 5
        elif has_pay:
            act = 2
        elif has_join_apk:
            act = 4
        elif has_url_click and not (has_call_wa and not uflags["has_url"]):
            act = 3
        elif has_call_wa:
            act = 5
        elif req_pii_reply:
            act = 1
        else:
            act = {2: 3, 3: 4, 4: 3}[lbl]

        t1 = int(bool(ENTITY_T1_RE.search(t)))
        t2 = int(lbl == 1 and (req_secret or not mistake) or req_pii_reply)
        t3 = int(has_pay or mistake or (lbl == 4 and act == 2) or bool(re.search(r"\b(?:fee|charge|duty|deposit|invest|visa fee|registration fee|clearance charge)\b|ফি|চার্জ|শুল্ক|বিনিয়োগ", t, re.I)))
        t4 = int(lbl == 2 or has_strong_prize or (has_soft_prize and not mistake))
        t5 = int(bool(URGENCY_T5_RE.search(t)) or mistake or bool(re.search(r"\b(?:good morning|brother|apu|mama|sir|friend|helpful|sorry|dear customer|last warning|act now|offer expires)\b", t, re.I)))
        t6 = int(lbl == 3 or has_job)
        tvec = np.array([t1, t2, t3, t4, t5, t6], dtype=np.int32)
        if tvec.sum() == 0:
            tvec[{1: 1, 2: 3, 3: 5, 4: 2}[lbl]] = 1

    return lbl, tvec, act, c, is_otp_legit, req_secret, has_strong_prize, has_parcel, has_job, has_job_trap


def run_v8_hotfix():
    t0 = time.time()
    # 1. Verify rules_v8 against ALL 86 sample.csv rows first!
    sdf = pd.read_csv(DATA_DIR / "sample.csv")
    y_sl = sdf["label"].values.astype(int)
    y_st = np.vstack([tech_str_to_vec(x) for x in sdf["techniques"]])
    y_sa = sdf["action"].values.astype(int)
    r_sl, r_st, r_sa = np.zeros(len(sdf), int), np.zeros((len(sdf), 6), int), np.zeros(len(sdf), int)
    for i, tx in enumerate(sdf["text"].fillna("").astype(str)):
        l, tv, a, *_ = extract_rule_v8(tx)
        r_sl[i], r_st[i], r_sa[i] = l, tv, a
    res_s = score_section10_proxy(y_sl, r_sl, y_st, r_st, y_sa, r_sa, np.full(len(sdf), 0.96))
    print(f"rules_v8 on sample.csv (86 rows): {res_s}", flush=True)

    # 2. Load raw test.csv + saved 500k model probabilities
    raw = pd.read_csv(DATA_DIR / "test.csv", usecols=["id", "text"])
    te_ids = raw["id"].values
    texts = raw["text"].fillna("").astype(str).tolist()
    n = len(texts)

    t_m = np.load(OUT_DIR / "test_mminilm_v4.npz")
    t_e = np.load(OUT_DIR / "test_me5small_v4.npz")
    t_b = np.load(OUT_DIR / "test_bge_v4.npz")
    t_bg = np.load(OUT_DIR / "test_bigru_v4.npz")
    t_ba = np.load(OUT_DIR / "test_da_tab_v4.npz")
    t_esc = np.load(OUT_DIR / "test_mdeberta_escalate_v4.npz")
    s4_clean = pd.read_csv(CACHE_DIR / "test_v4_clean.csv", usecols=["clean_text"])["clean_text"].fillna("").astype(str).tolist()

    # Copy mutable arrays for re-scoring
    pm_l, pm_t, pm_a = t_m["p_l"].copy(), t_m["p_t"].copy(), t_m["p_a"].copy()

    # 3. Extract rules_v8 + clean_v8 across all 500,000 rows (~20s)
    r8_l = np.zeros(n, dtype=np.int64)
    r8_t = np.zeros((n, 6), dtype=np.float32)
    r8_a = np.zeros(n, dtype=np.int64)
    c8_list = []
    otp_legit = np.zeros(n, dtype=bool)
    req_sec = np.zeros(n, dtype=bool)
    strong_pr = np.zeros(n, dtype=bool)
    has_par = np.zeros(n, dtype=bool)
    has_jb = np.zeros(n, dtype=bool)
    has_jbt = np.zeros(n, dtype=bool)
    diff_clean = np.zeros(n, dtype=bool)

    for i, raw_tx in enumerate(texts):
        l, tv, a, c8, ol, rs, sp, hp, hj, hjt = extract_rule_v8(raw_tx)
        r8_l[i], r8_t[i], r8_a[i] = l, tv, a
        c8_list.append(c8)
        otp_legit[i], req_sec[i], strong_pr[i] = ol, rs, sp
        has_par[i], has_jb[i], has_jbt[i] = hp, hj, hjt
        # Flag if aggressive cleaning materially repaired words vs v4 clean_text
        if len(c8) > 0 and c8 != s4_clean[i].lower():
            # Compare alpha word counts or rule flip
            diff_clean[i] = True

    old_l = (0.35 * pm_l + 0.25 * t_e["p_l"] + 0.10 * t_b["p_l"] + 0.30 * t_bg["p_l"]).argmax(1)
    prio1 = np.where(r8_l != old_l)[0]
    prio2 = np.where((r8_l == old_l) & diff_clean & (old_l == 0))[0]
    if len(prio1) < 28000:
        re_idx = np.sort(np.unique(np.concatenate([prio1, prio2[:28000 - len(prio1)]])))
    else:
        re_idx = prio1[:28000]
    print(f"rules_v8 done in {time.time()-t0:.1f}s! Diff vs old neural={int((r8_l != old_l).sum())} | Re-scoring mMiniLM on {len(re_idx)} cleaned rows...", flush=True)

    # 4. Re-score repaired/uncertain rows on GPU with mMiniLM_distill.pt (~95s!)
    if len(re_idx) > 0:
        free_cuda()
        ck = torch.load(CKPT_DIR / "mminilm_distill.pt", map_location="cpu", weights_only=False)
        tok = AutoTokenizer.from_pretrained(ck["model_name"], cache_dir=CACHE_DIR_HF, local_files_only=HF_LOCAL_ONLY)
        mdl = HeadConditionedTransformer(ck["model_name"], n_feats=58, pool_mode=ck["pool_mode"], proj_dim=256).to(DEV).eval()
        mdl.load_state_dict(ck["state_dict"], strict=True)
        te_feats_m = np.load(CACHE_DIR / "test_v4_feats.npy", mmap_mode="r")[re_idx].copy()
        sc_f = scale_feats_v4(te_feats_m)
        sub_txt = [format_turn_aware(c8_list[i]) for i in re_idx]
        pl_new, pt_new, pa_new = predict_transformer(mdl, tok, sub_txt, sc_f, bs=112, max_len=144, device=DEV)
        pm_l[re_idx], pm_t[re_idx], pm_a[re_idx] = pl_new, pt_new, pa_new
        del mdl, ck
        free_cuda()

    # 5. Balanced Un-Distorted Ensemble Blend (No extreme 0.88..1.15 sample overfitting!)
    t_gb_l = 0.40 * t_ba["lgb_l"] + 0.35 * t_ba["xgb_l"] + 0.25 * t_ba["cb_l"]
    r8_pl = np.full((n, 6), 0.03, dtype=np.float32)
    r8_pl[np.arange(n), r8_l] = 0.85
    r8_pa = np.zeros((n, 6), dtype=np.float32)
    pos_a = r8_a > 0
    r8_pa[pos_a] = 0.035
    r8_pa[np.where(pos_a)[0], r8_a[pos_a]] = 0.86
    r8_pt = r8_t * 0.85 + 0.07

    # Balanced weights prioritizing clean neural ensemble + rules_v8
    P_L = 0.26 * pm_l + 0.17 * t_e["p_l"] + 0.10 * t_b["p_l"] + 0.16 * t_bg["p_l"] + 0.06 * t_ba["lin_l"] + 0.09 * t_gb_l + 0.16 * r8_pl
    P_T = 0.28 * pm_t + 0.24 * t_e["p_t"] + 0.14 * t_b["p_t"] + 0.06 * t_bg["p_t"] + 0.06 * t_ba["lin_t"] + 0.08 * t_ba["gb_t"] + 0.14 * r8_pt
    P_A = 0.22 * pm_a + 0.19 * t_e["p_a"] + 0.14 * t_b["p_a"] + 0.08 * t_bg["p_a"] + 0.06 * t_ba["lin_a"] + 0.06 * t_ba["gb_a"] + 0.25 * r8_pa

    # Apply mDeBERTa escalation with 0.42 weight on escalated uncertain rows
    ei = t_esc["esc_idx"]
    P_L[ei] = 0.58 * P_L[ei] + 0.42 * t_esc["p_l"]
    P_T[ei] = 0.58 * P_T[ei] + 0.42 * t_esc["p_t"]
    P_A[ei] = 0.58 * P_A[ei] + 0.42 * t_esc["p_a"]

    # High-Precision Guide Priority & Adversarial Repair Gates
    # Gate 1: Legitimate OTP warning with don't-share negation -> Class 0
    g_otp0 = otp_legit & (~req_sec)
    P_L[g_otp0, 0] = np.maximum(P_L[g_otp0, 0], 0.82)
    # Gate 2: Explicit secret harvest (OTP/PIN/CVV/NID/AnyDesk) -> Priority 1
    g_p1 = req_sec & (r8_l == 1) & (~otp_legit)
    P_L[g_p1, 1] = np.maximum(P_L[g_p1, 1], 0.78)
    # Gate 3: Reward/Selected/Job + clearance charge misclassified as Parcel (4) -> 2 or 3
    g_rew_clr = (r8_l == 2) & strong_pr & (~has_par) & (~has_jb)
    P_L[g_rew_clr, 2] = np.maximum(P_L[g_rew_clr, 2], 0.82)
    g_job_clr = (r8_l == 3) & has_jb & has_jbt & (~has_par)
    P_L[g_job_clr, 3] = np.maximum(P_L[g_job_clr, 3], 0.82)

    # Smooth balanced thresholds (fixing T1=0.36 over-firing!)
    tech_th = np.array([0.48, 0.46, 0.49, 0.45, 0.50, 0.48], dtype=np.float32)
    final_l, final_t, final_a = decode_consistent(P_L, P_T, P_A, tech_th=tech_th)

    # Calibrated Brier confidence: smooth agreement + margin + max_prob (no overconfident 0.996 saturation!)
    pred_set = [pm_l.argmax(1), t_e["p_l"].argmax(1), t_b["p_l"].argmax(1), t_bg["p_l"].argmax(1), t_gb_l.argmax(1), r8_l]
    votes = np.mean([(p == final_l) for p in pred_set], axis=0)
    sp = np.sort(P_L, axis=1)
    margin = sp[:, -1] - sp[:, -2]
    conf = np.clip(0.32 + 0.34 * votes + 0.24 * P_L.max(1) + 0.10 * margin, 0.25, 0.985)
    conf[np.isin(np.arange(n), re_idx)] = np.minimum(conf[np.isin(np.arange(n), re_idx)], 0.96)

    sub = pd.DataFrame({
        "id": te_ids,
        "label": final_l,
        "techniques": [vec_to_tech_str(final_t[i]) for i in range(n)],
        "action": final_a,
        "confidence": np.round(conf, 4),
    })
    out_csv = V2_DIR / "submission_v8_hotfix.csv"
    sub.to_csv(out_csv, index=False)
    s7 = pd.read_csv(V2_DIR / "submission_v7_top10.csv")
    print(f"Saved {out_csv} in {time.time()-t0:.1f}s! Changed vs s7: lbl={(sub['label']!=s7['label']).sum()} act={(sub['action']!=s7['action']).sum()} tech={(sub['techniques']!=s7['techniques']).sum()}", flush=True)

In [20]:
"""Ultra-precision V9 Surgical Polish over V8 Ensemble (Zero False Positives, ~25s CPU)."""
import re
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd



EXTRA_REPAIRS = {
    **EXACT_TYPO_REPAIRS,
    "piin": "pin", "piiinn": "pin", "currentp1n": "current pin", "contiune": "continue",
    "sellect": "select", "hoyeechen": "hoyechen", "tasharre": "ta share", "korunn": "korun",
    "offeer": "offer", "rewardshare": "reward share", "rewardsahre": "reward share",
    "justreceived": "just received", "shhare": "share",
}
EXTRA_RE = re.compile(r"\b(" + "|".join(re.escape(k) for k in sorted(EXTRA_REPAIRS, key=len, reverse=True)) + r")\b", re.I)

def clean_v9(raw_text: str) -> str:
    s = clean_v8(raw_text)
    s = EXTRA_RE.sub(lambda m: EXTRA_REPAIRS[m.group(1).lower()], s)
    # Single-pass collapse of residual >1 repeats inside specific keywords
    s = re.sub(r"([a-z])\1+", lambda m: m.group(1) if m.group(0) in {"ll", "ee", "oo", "ss", "nn"} and len(m.group(0)) > 2 else m.group(0), s)
    s = s.replace("rewardsahre", "reward share").replace("rewardshare", "reward share").replace("justreceived", "just received").replace("tashare", "ta share")
    s = re.sub(r"\s+", " ", s).strip()
    return s

NEG_GUARD = re.compile(
    r"(?:never share|do not share|don'?t share|kauke.{0,20}(?:diben|bolben|korben|share).{0,6}na|bank er lokeo na|messaged me:|that is phishing|eta scam|will never call)",
    re.I,
)
PARCEL_REAL = re.compile(r"\b(?:parcel|package|courier|customs|delivery|shipment|consignment|tracking|redelivery|atke ase|could not be delivered)\b|পার্সেল|কুরিয়ার|কাস্টমস", re.I)
CARD_CVV_OTP = re.compile(
    r"(?:\bshare\s+the\s+(?:\d[- ]?digit\s+)?(?:otp|pin|code)\b"
    r"|\b(?:otp|pin).{0,20}\bshare\s+korun\b"
    r"|\banydesk.{0,30}(?:\d[- ]?digit|code)\b"
    r"|\bcard\s*number.{0,25}\bcvv\b"
    r"|\b(?:your\s+)?(?:whatsapp|telegram|sim|account)\s+(?:will be|hobe)\s+deactivated.{0,40}\bcode\b"
    r"|\b(?:pohne|phone).{0,15}hariyeche.{0,35}\bcode\b)",
    re.I,
)
REW_CLR = re.compile(
    r"(?:selected\s+for\s+(?:a\s+)+.*(?:reward|laptop|motorbike|smart\s*tv|toyota|iphone|samsung|gift\s*card)"
    r"|apni\s+(?:a\s+)?(?:motorbike|laptop|samsung\s*s?\d*|iphone|toyota|smart\s*tv).{0,35}select\s*hoy.*?(?:fee|charge))",
    re.I,
)
JOB_CLR = re.compile(
    r"(?:work\s*from\s*home|package\s*handler|data\s*entry|vacancy|hiring).{0,80}clearance\s*charge",
    re.I,
)


def run_v9_final():
    t0 = time.time()
    sub = pd.read_csv(V2_DIR / "submission_v8_hotfix.csv")
    raw = pd.read_csv(DATA_DIR / "test.csv", usecols=["text"])["text"].fillna("").astype(str).tolist()
    n = len(sub)
    lbl = sub["label"].values.copy()
    act = sub["action"].values.copy()
    tech = sub["techniques"].values.tolist()
    conf = sub["confidence"].values.copy()

    fixes = {"p1": 0, "rew2": 0, "job3": 0}
    # Only scan rows where s8 is 0, 2, 3, 4 (fast surgical pass)
    cand = np.where(np.isin(lbl, [0, 2, 3, 4]))[0]
    for i in cand:
        c = clean_v9(raw[i])
        if NEG_GUARD.search(c):
            continue
        # Fix 1: Uncaught high-precision Priority-1 credential harvest (219, 2032, 300035, 1038, 1280, 1660)
        if lbl[i] != 1 and CARD_CVV_OTP.search(c):
            lbl[i] = 1
            act[i] = 1
            tv = [int(bool(ENTITY_T1_RE.search(c))), 1, 0, int(bool(re.search(r"won|reward|select|lucky|jitechen", c, re.I))), int(bool(URGENCY_T5_RE.search(c)) or "offer" in c), int(bool(re.search(r"job|work from home|income", c, re.I)))]
            tech[i] = vec_to_tech_str(np.array(tv, dtype=int))
            conf[i] = min(conf[i], 0.88)
            fixes["p1"] += 1
            continue
        # Fix 2: Reward/Selected + clearance charge / fee predicted 0 or 4 instead of 2 (602, 814, 1225, 3057)
        if lbl[i] in (0, 3, 4) and REW_CLR.search(c) and not PARCEL_REAL.search(c):
            lbl[i] = 2
            act[i] = 2 if re.search(r"\b(?:pay|fee|charge|din|pathan|bkash|nagad)\b", c, re.I) else 3
            tv = [int(bool(ENTITY_T1_RE.search(c))), 0, int(act[i] == 2), 1, int(bool(URGENCY_T5_RE.search(c))), 0]
            tech[i] = vec_to_tech_str(np.array(tv, dtype=int))
            conf[i] = min(conf[i], 0.88)
            fixes["rew2"] += 1
            continue
        # Fix 3: Job + clearance charge predicted 4 instead of 3
        if lbl[i] == 4 and JOB_CLR.search(c) and not PARCEL_REAL.search(c):
            lbl[i] = 3
            act[i] = 2
            tv = [int(bool(ENTITY_T1_RE.search(c))), 0, 1, 0, int(bool(URGENCY_T5_RE.search(c))), 1]
            tech[i] = vec_to_tech_str(np.array(tv, dtype=int))
            conf[i] = min(conf[i], 0.88)
            fixes["job3"] += 1

    # Strict Section 9 invariants
    mask05 = np.isin(lbl, [0, 5])
    act[mask05] = 0
    for i in np.where(mask05)[0]:
        tech[i] = "NONE"

    out = pd.DataFrame({
        "id": sub["id"].values,
        "label": lbl,
        "techniques": tech,
        "action": act,
        "confidence": np.round(conf, 4),
    })
    out_p = WORK_DIR / "submission.csv"
    out.to_csv(out_p, index=False)
    print(f"Saved {out_p} in {time.time()-t0:.1f}s! Fixes={fixes} | Spot 219,602,814,2032,300035 labels={lbl[[219,602,814,2032,300035]].tolist()}", flush=True)

In [21]:
# Live V9 adversarial repair comparison (raw -> clean_for_model v4 -> final clean_v9)
adv_examples = [
    "mnbvcx Your bKash PIIIN will expire today Rpely witttth your currentP1N ad a new PIN to contiune service",
    "866970 is y\u200bournagad lоgіncode don07 share it w\u200bith anyoe even mnbvcx na\u200bgad 5taff",
    "khala ami,pohne hariyeche 💰 Tomr nubmer e je cooodeaschhe oa bOLo",
    "You have been selected for a a laptop reward. Pay Tk 3,500 clearance charge to +880 015XXXXXX86.",
]
for idx_ex, ex in enumerate(adv_examples, 1):
    print(f"[{idx_ex}] v4: {clean_for_model(ex)}")
    print(f"    v9: {clean_v9(ex)}")

[1] v4: your bkash pin will expire today reply with your currentp1n and a new pin to continue service
    v9: your bkash pin will expire today reply with your current pin and a new pin to continue service
[2] v4: 866970 is your nagad login code don07 share it with anyoe even nagad staff
    v9: 866970 is yournagad logincode don't share it with anyoe even nagad 5taff
[3] v4: khala ami,pohne hariyeche <EMOJI_MONEY> tomr nubmer e je codeaschhe oa bolo
    v9: khala ami,phone hariyeche tomr number e je code ashche ota bolo
[4] v4: you have been selected for a a laptop reward. pay tk 3,500 clearance charge to +880 015XXXXXX86.
    v9: you have been selected for a a laptop reward. pay tk 3,500 clearance charge to +880 015xx86.


In [22]:
"""Purely Inductive Section 7.3 End-to-End Inference Harness for ScamShield BD v4."""
import argparse
import gc
import sys
import time
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import torch
from scipy.sparse import hstack, csr_matrix


from transformers import AutoTokenizer


def run_inductive_inference(input_csv: Path, output_csv: Path, esc_frac: float = 0.055):
    t0 = time.time()
    df = pd.read_csv(input_csv)
    raw_texts = df["text"].fillna("").astype(str).tolist()
    n = len(raw_texts)

    cleans = []
    feats = np.zeros((n, 58), dtype=np.float32)
    r_lbls = np.zeros(n, dtype=np.int64)
    r_tvecs = np.zeros((n, 6), dtype=np.float32)
    r_acts = np.zeros(n, dtype=np.int64)
    for i, raw in enumerate(raw_texts):
        cleans.append(clean_for_model(raw))
        l, tv, a, f = extract_rule_prediction(raw)
        r_lbls[i], r_tvecs[i], r_acts[i], feats[i] = l, tv, a, f

    turns = [format_turn_aware(c) for c in cleans]
    e5_texts = ["passage: " + t for t in turns]
    bge_texts = ["Represent this sentence for searching relevant passages: " + bn_to_en_roman_v4(t) for t in turns]
    sc_feats = scale_feats_v4(feats)

    # 1. Domain-Adapted Tabular Linear + LGBM/XGB/CatBoost
    tab = joblib.load(CKPT_DIR / "da_tab_v4.joblib")
    lin_l = np.zeros((n, 6), np.float32)
    lin_t = np.zeros((n, 6), np.float32)
    lin_a = np.zeros((n, 6), np.float32)
    gb_l = np.zeros((n, 6), np.float32)
    gb_t = np.zeros((n, 6), np.float32)
    gb_a = np.zeros((n, 6), np.float32)

    for s in range(0, n, 50000):
        e = min(n, s + 50000)
        X_tw = hstack([tab["vec_w"].transform(turns[s:e]), tab["vec_c"].transform(turns[s:e])]).tocsr()
        F_c = tab["scaler"].transform(feats[s:e]).astype(np.float32)
        Z_c = tab["svd"].transform(X_tw).astype(np.float32)
        X_lc = hstack([X_tw, csr_matrix(F_c * 0.25)]).tocsr()
        X_tc = np.hstack([F_c, Z_c]).astype(np.float32)

        fl, fts, fa = tab["lin"]
        lin_l[s:e] = fl.predict_proba(X_lc).astype(np.float32)
        lin_t[s:e] = np.column_stack([c.predict_proba(X_lc)[:, 1] for c in fts]).astype(np.float32)
        pa_sub = fa.predict_proba(X_lc).astype(np.float32)
        for ci, cls_id in enumerate(fa.classes_):
            lin_a[s:e, int(cls_id)] = pa_sub[:, ci]

        gm = tab["gb"]
        pl1 = gm["lgb_l"].predict_proba(X_tc).astype(np.float32)
        px1 = gm["xgb_l"].predict_proba(X_tc).astype(np.float32)
        if gm.get("cb_l") is not None:
            pcb1 = gm["cb_l"].predict_proba(X_tc).astype(np.float32)
            gb_l[s:e] = 0.40 * pl1 + 0.35 * px1 + 0.25 * pcb1
        else:
            gb_l[s:e] = 0.55 * pl1 + 0.45 * px1
        gb_t[s:e] = np.column_stack([c.predict_proba(X_tc)[:, 1] for c in gm["lgbs_t"]]).astype(np.float32)
        pa_g = gm["lgb_a"].predict_proba(X_tc).astype(np.float32)
        for ci, cls_id in enumerate(gm["lgb_a"].classes_):
            gb_a[s:e, int(cls_id) + 1] = pa_g[:, ci]
    del tab
    gc.collect()

    _dev_unused = "cuda" if torch.cuda.is_available() else "cpu"

    # 2. Transformer & BiGRU Students (streaming one model at a time for strict <2GB VRAM / <4GB RAM)
    def run_ckpt(ckpt_name, in_texts, bs=96, max_len=152):
        free_cuda()
        ck = torch.load(CKPT_DIR / ckpt_name, map_location="cpu", weights_only=False)
        mname = ck["model_name"]
        tok = AutoTokenizer.from_pretrained(mname, cache_dir=CACHE_DIR_HF, local_files_only=HF_LOCAL_ONLY)
        mdl = HeadConditionedTransformer(mname, n_feats=58, pool_mode=ck["pool_mode"], proj_dim=256)
        mdl.load_state_dict(ck["state_dict"], strict=True)
        mdl = mdl.to(DEV).eval()
        pl, pt, pa = predict_transformer(mdl, tok, in_texts, sc_feats, bs=bs, max_len=max_len, device=DEV)
        del mdl, ck
        free_cuda()
        return pl, pt, pa

    m_l, m_t, m_a = run_ckpt("mminilm_distill.pt", turns, bs=96)
    e_l, e_t, e_a = run_ckpt("me5small_distill.pt", e5_texts, bs=96)
    b_l, b_t, b_a = run_ckpt("bge_distill.pt", bge_texts, bs=96)

    # BiGRU-v4
    free_cuda()
    tok_bg = AutoTokenizer.from_pretrained("MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli", cache_dir=CACHE_DIR_HF, local_files_only=HF_LOCAL_ONLY)
    bg = BPEBiGRUNet(n_feats=58).to(DEV).eval()
    bg.load_state_dict(torch.load(CKPT_DIR / "bigru_v4_distill.pt", map_location="cpu", weights_only=False)["state_dict"], strict=True)
    bg_l, bg_t, bg_a = predict_bigru(bg, tok_bg, turns, sc_feats, bs=256)
    del bg
    free_cuda()

    # 3. Rule priors & Meta Stacking
    r_pl = onehot_from_int(r_lbls, conf=0.84)
    r_pt = r_tvecs * 0.84 + 0.08
    r_pa = np.zeros((n, 6), np.float32)
    pos_a = r_acts > 0
    r_pa[pos_a, :] = 0.0375
    r_pa[np.where(pos_a)[0], r_acts[pos_a]] = 0.85

    meta = joblib.load(CKPT_DIR / "meta_calibrator_v4.joblib")
    L_stu = [m_l, e_l, b_l, bg_l, lin_l, gb_l, r_pl]
    T_stu = [m_t, e_t, b_t, bg_t, lin_t, gb_t, r_pt]
    A_stu = [m_a, e_a, b_a, bg_a, lin_a, gb_a, r_pa]

    P_L = sum(w * x for w, x in zip(meta["w_l_stu"], L_stu))
    P_T = sum(w * x for w, x in zip(meta.get("w_t_stu", meta["w_l_stu"]), T_stu))
    P_A = sum(w * x for w, x in zip(meta.get("w_a_stu", meta["w_l_stu"]), A_stu))

    # 4. Selective Inductive mDeBERTa-v3-base Escalation on top uncertain frontier
    k_esc = max(1, int(round(n * esc_frac)))
    ent = -np.sum(P_L * np.log(np.clip(P_L, 1e-6, 1.0)), axis=1)
    dis_l = (m_l.argmax(1) != e_l.argmax(1)).astype(np.float32) + (m_l.argmax(1) != b_l.argmax(1)).astype(np.float32) + (m_l.argmax(1) != gb_l.argmax(1)).astype(np.float32)
    score_esc = ent + 0.65 * dis_l + (1.0 - P_L.max(1))
    esc_idx = np.sort(np.argsort(-score_esc)[:k_esc])

    esc_tx = [turns[i] for i in esc_idx]
    esc_f = sc_feats[esc_idx]
    full_ck = CKPT_DIR / "mdeberta_full_distill.pt"
    if full_ck.exists():
        free_cuda()
        ck = torch.load(full_ck, map_location="cpu", weights_only=False)
        tok_d = AutoTokenizer.from_pretrained(ck["model_name"], cache_dir=CACHE_DIR_HF, local_files_only=HF_LOCAL_ONLY)
        mdl_d = HeadConditionedTransformer(ck["model_name"], n_feats=58, pool_mode=ck["pool_mode"], proj_dim=256, num_freeze_layers=5)
        mdl_d.load_state_dict(ck["state_dict"], strict=True)
        mdl_d = mdl_d.to(DEV).eval()
        esc_l, esc_t, esc_a = predict_transformer(mdl_d, tok_d, esc_tx, esc_f, bs=48, max_len=152, device=DEV)
        del mdl_d, ck
        free_cuda()
    else:
        for fold_tag, wt in [("mdeberta_f0.pt", 0.5), ("mdeberta_f1.pt", 0.5)]:
            free_cuda()
            ck = torch.load(CKPT_DIR / fold_tag, map_location="cpu", weights_only=False)
            tok_d = AutoTokenizer.from_pretrained(ck["model_name"], cache_dir=CACHE_DIR_HF, local_files_only=HF_LOCAL_ONLY)
            mdl_d = HeadConditionedTransformer(ck["model_name"], n_feats=58, pool_mode=ck["pool_mode"], proj_dim=256, num_freeze_layers=ck["freeze_k"])
            mdl_d.load_state_dict(ck["state_dict"], strict=True)
            mdl_d = mdl_d.to(DEV).eval()
            dl, dt, da = predict_transformer(mdl_d, tok_d, esc_tx, esc_f, bs=48, max_len=152, device=DEV)
            if fold_tag.endswith("f0.pt"):
                esc_l, esc_t, esc_a = wt * dl, wt * dt, wt * da
            else:
                esc_l += wt * dl; esc_t += wt * dt; esc_a += wt * da
            del mdl_d, ck
            free_cuda()

    w_esc = meta["w_l_esc"][-1]
    P_L[esc_idx] = (1.0 - w_esc) * P_L[esc_idx] + w_esc * esc_l
    P_T[esc_idx] = (1.0 - w_esc) * P_T[esc_idx] + w_esc * esc_t
    P_A[esc_idx] = (1.0 - w_esc) * P_A[esc_idx] + w_esc * esc_a

    X_cal = build_uncertainty_features(L_stu, feats, r_lbls)
    conf = np.clip(meta["cal_lr"].predict_proba(X_cal)[:, 1], 0.05, 0.996).astype(np.float32)

    final_l, final_t, final_a = decode_consistent(P_L * meta["best_alpha"], P_T, P_A * meta["best_beta"], tech_th=meta["tech_th"])
    out = pd.DataFrame({
        "id": df["id"].values,
        "label": final_l,
        "techniques": [vec_to_tech_str(final_t[i]) for i in range(n)],
        "action": final_a,
        "confidence": np.round(conf, 4),
    })
    out.to_csv(output_csv, index=False)
    print(f"Inductive prediction complete on {n} rows -> {output_csv} in {time.time()-t0:.1f}s!", flush=True)

In [23]:
# Master end-to-end execution guard (set RUN_FULL_TRAINING=True to reproduce everything from scratch)
if RUN_FULL_TRAINING:
    run_prepare_cache()
    run_baseline_v5()
    run_teachers_v4()
    run_build_clean_pseudo_v5()
    run_da_bigru_v4()
    run_da_tab_v4()
    run_students_v4()
    run_meta_calibrate_v4()
    run_v8_hotfix()
    run_v9_final()
else:
    print("RUN_FULL_TRAINING=False -> Using verified checkpoints/predictions for audit & validation cells below.")

RUN_FULL_TRAINING=False -> Using verified checkpoints/predictions for audit & validation cells below.


## 9. Validation Benchmarks, Error Analysis & Final Verification (Executed)

### 9.1 Zero-Leakage Scenario-Disjoint Validation Summary
Evaluates saved OOD predictions (`oof_*.npz`) if local caches are present, and demonstrates why multi-task Neural models complement TF-IDF/GBDT cleanly (`BiGRU-v5` & Transformers gain `+6` to `+18` F1 on Task B `techniques` and Task C `action`, while `rules` + GBDT stabilize Task A primary classes).

In [24]:
oof_ba_p = OUT_DIR / "oof_baseline_v4.npz"
oof_bg_p = OUT_DIR / "oof_bigru_v4.npz"
oof_st_p = OUT_DIR / "oof_students_v4.npz"
s_cache_p = CACHE_DIR / "sample_v4.csv"

if all(p.exists() for p in [oof_ba_p, oof_bg_p, oof_st_p, s_cache_p]):
    sdf_v4 = pd.read_csv(s_cache_p)
    ys_l = sdf_v4["label"].values.astype(int)
    ys_t = np.vstack([tech_str_to_vec(x) for x in sdf_v4["techniques"]])
    ys_a = sdf_v4["action"].values.astype(int)
    ba = np.load(oof_ba_p); bg = np.load(oof_bg_p); st = np.load(oof_st_p)
    rows_eval = []
    for nm, pl, pt, pa in [
        ("TF-IDF Linear (OOD)", ba["s_lin_l"], ba["s_lin_t"], ba["s_lin_a"]),
        ("SVD+GBDT Stack (OOD)", ba["s_gb_l"], ba["s_gb_t"], ba["s_gb_a"]),
        ("BiGRU-v5 (3-Fold OOD)", bg["s_l"], bg["s_t"], bg["s_a"]),
        ("mMiniLM-v5 (3-Fold OOD)", st["s_mminilm_l"], st["s_mminilm_t"], st["s_mminilm_a"]),
        ("mE5-small-v5 (3-Fold OOD)", st["s_me5small_l"], st["s_me5small_t"], st["s_me5small_a"]),
        ("BGE-small-v5 (3-Fold OOD)", st["s_bge_l"], st["s_bge_t"], st["s_bge_a"]),
    ]:
        dl, dt, da = decode_consistent(pl, pt, pa)
        sc = score_section10_proxy(ys_l, dl, ys_t, dt, ys_a, da, pl.max(1))
        rows_eval.append({"Model": nm, **sc})
    print(pd.DataFrame(rows_eval)[["Model", "Composite", "F_A_primary", "F_B_tech", "F_C_action", "Brier", "Acc_primary"]].to_string(index=False))

    # Optional inductive end-to-end check on sample.csv if all model weights & GBDT libs are present locally
    if lgb is not None and xgb is not None and (CKPT_DIR / "meta_calibrator_v4.joblib").exists() and (CKPT_DIR / "mdeberta_full_distill.pt").exists():
        ind_tmp = WORK_DIR / "_tmp_inductive_sample.csv"
        run_inductive_inference(DATA_DIR / "sample.csv", ind_tmp)
        chk = pd.read_csv(ind_tmp)
        ind_sc = score_section10_proxy(
            sample_df["label"].values, chk["label"].values,
            np.vstack([tech_str_to_vec(x) for x in sample_df["techniques"]]),
            np.vstack([tech_str_to_vec(x) for x in chk["techniques"]]),
            sample_df["action"].values, chk["action"].values, chk["confidence"].values,
        )
        print("\nPurely Inductive End-to-End Check on sample.csv (Section 7.3):", ind_sc)
        if ind_tmp.exists():
            ind_tmp.unlink()
else:
    print("Cached OOD arrays not found in standalone directory — skipped cached OOD table.")

                    Model  Composite  F_A_primary  F_B_tech  F_C_action   Brier  Acc_primary
      TF-IDF Linear (OOD)     91.983        95.55     78.13       85.10 0.02647        95.35
     SVD+GBDT Stack (OOD)     91.881        97.33     76.05       74.94 0.03015        96.51
    BiGRU-v5 (3-Fold OOD)     94.456        96.50     88.07       88.56 0.03175        96.51
  mMiniLM-v5 (3-Fold OOD)     91.995        93.68     82.48       92.88 0.04876        93.02
mE5-small-v5 (3-Fold OOD)     89.743        91.57     82.29       86.11 0.06254        90.70
BGE-small-v5 (3-Fold OOD)     87.609        88.53     83.38       85.13 0.07581        87.21


### 9.2 Error Analysis, Adversarial Normalization & Limitations

1. **Class Confusion & Rule Priority Audit**: Early rules misclassified *"selected for a laptop/Smart TV reward + clearance charge"* and *"work from home job + clearance charge"* as Class 4 (Parcel) because `clearance charge` sat inside `PARCEL_RE`; restricting `PARCEL_RE` to true parcel/courier/customs context restored Classes 2 and 3. Similarly, `"please do not reply"` sat inside `NEG_WARN_RE`, allowing scam footers to mask deceptive requests (`idx=7050`).
2. **Adversarial & Multilingual Robustness**: `test.csv` systematically combines zero-width joiners (`54,310` rows), Cyrillic/Greek confusables (`42,538` rows), random noise tokens (`asdkjh`, `wkrjfejejrn`, `mnbvcx`, `zzqwpo`), leetspeak (`b4nk`, `don07`, `s3cur1ty`), character repeats (`taaaakadaly`), and token merging (`sharethe`, `coodeaschhe`, `cvvdin`, `dibenna`). Adding `clean_v8`/`clean_v9` and re-scoring repaired rows with `mMiniLM` lifted Kaggle Public LB from `0.93177 -> 0.93552 -> 0.93661`.
3. **Hard Legitimate Separation**: Genuine OTP warnings (`"866970 is your nagad login code don07 share it with anyone"` or `"Standard Chrtered: PIN, OTP ba password kauke dibenna, b4nk er lokeo na"`), COD delivery notices, transaction alerts (`TrxID`), and friend scam-warning threads (`"someone from ... messaged me ... eta scam"`) contain high-overlap scam vocabulary and are protected through explicit negation-window scoping and contextual features.
4. **Justification & Honest Limitations**: Under a strict 4 GB VRAM single-GPU constraint, 2-fold teachers with aggressive domain holdout initially experienced scenario blind spots (`~74.9` OOD), motivating balanced 3-fold OOD splits (`67%` of all classes retained per fold), consensus prototype filtering (`39,618 / 45,000`), compact distilled Transformer students, and selective `26k` `mDeBERTa` escalation. Future improvements would expand synthetic multi-turn/adversarial coverage and distill `mMiniLM`/`mE5-small` directly on `clean_v9` text.

### 9.3 Final Submission Verification (`submission.csv`, Executed)

In [25]:
import sys
import numpy as np
import pandas as pd


def verify_submission(sub_path: str, sample_sub_path: str = str(DATA_DIR / "sample_submission.csv")):
    print(f"Verifying {sub_path} against {sample_sub_path} ...")
    ref_ids = pd.read_csv(sample_sub_path, usecols=["id"])["id"].values
    ref_cols = list(pd.read_csv(sample_sub_path, nrows=1).columns)

    df = pd.read_csv(sub_path, keep_default_na=False)
    assert list(df.columns) == ref_cols, f"Column mismatch: {list(df.columns)} != {ref_cols}"
    assert len(df) == len(ref_ids), f"Row count mismatch: {len(df)} != {len(ref_ids)}"
    assert np.array_equal(df["id"].values, ref_ids), "ID alignment mismatch!"

    # Check zero blanks / NaNs
    for col in df.columns:
        s = df[col].astype(str)
        assert not (s == "").any(), f"Blank values in {col}"
        assert not (s.str.lower() == "nan").any(), f"NaN string in {col}"

    # Validate label in 0..5
    lbl = pd.to_numeric(df["label"], errors="raise").values
    assert np.isin(lbl, [0, 1, 2, 3, 4, 5]).all(), "Invalid label outside 0..5!"

    # Validate action in 0..5 and consistency with label
    act = pd.to_numeric(df["action"], errors="raise").values
    assert np.isin(act, [0, 1, 2, 3, 4, 5]).all(), "Invalid action outside 0..5!"
    is_np = np.isin(lbl, [0, 5])
    assert (act[is_np] == 0).all(), f"Found {(act[is_np] != 0).sum()} rows with label in {{0,5}} but action != 0!"
    assert (act[~is_np] != 0).all(), f"Found {(act[~is_np] == 0).sum()} scam rows (1..4) with action == 0!"

    # Validate techniques syntax and consistency
    valid_codes = {"T1", "T2", "T3", "T4", "T5", "T6"}
    tech = df["techniques"].astype(str).values
    assert (tech[is_np] == "NONE").all(), f"Found {(tech[is_np] != 'NONE').sum()} rows with label in {{0,5}} but techniques != NONE!"
    assert (tech[~is_np] != "NONE").all(), f"Found {(tech[~is_np] == 'NONE').sum()} scam rows (1..4) with techniques == NONE!"
    bad_syntax = 0
    for s in set(tech):
        if s == "NONE":
            continue
        parts = s.split("|")
        if len(parts) != len(set(parts)) or not all(p in valid_codes for p in parts) or parts != sorted(parts):
            bad_syntax += 1
    assert bad_syntax == 0, f"Found {bad_syntax} invalid technique strings!"

    # Validate confidence in (0, 1]
    conf = pd.to_numeric(df["confidence"], errors="raise").values
    assert np.isfinite(conf).all(), "Non-finite confidence!"
    assert ((conf >= 0.0) & (conf <= 1.0)).all(), "Confidence outside [0, 1]!"

    print("ALL ASSERTIONS PASSED SUCCESSFULLY!")
    print("Label distribution (%):")
    vc_l = df["label"].value_counts().sort_index()
    for k, v in vc_l.items():
        print(f"  label={k}: {v:7d} ({100*v/len(df):.2f}%)")
    print("Action distribution (%):")
    vc_a = df["action"].value_counts().sort_index()
    for k, v in vc_a.items():
        print(f"  action={k}: {v:7d} ({100*v/len(df):.2f}%)")
    print(f"Top 10 techniques: {df['techniques'].value_counts().head(10).to_dict()}")
    print(f"Confidence stats: min={conf.min():.4f}, p25={np.percentile(conf,25):.4f}, p50={np.percentile(conf,50):.4f}, mean={conf.mean():.4f}, p75={np.percentile(conf,75):.4f}, max={conf.max():.4f}")


sub_target = WORK_DIR / "submission.csv"
verify_submission(str(sub_target), str(DATA_DIR / "sample_submission.csv"))
raw_tx_check = test_raw_df["text"].fillna("").astype(str)
sub_df_check = pd.read_csv(sub_target)
print("\nFinal submission SHA-256:", hashlib.sha256(sub_target.read_bytes()).hexdigest())
for name, pat in [
    ("Bangla script", r"[\u0980-\u09FF]"),
    ("Multi-turn |||", r"\|\|\|"),
    ("Zero-width", r"[\u200b-\u200f\u2060\ufeff\ufe0f\u00ad]"),
    ("Cyrillic/Greek", r"[\u0400-\u04ff\u0370-\u03ff]"),
]:
    m = raw_tx_check.str.contains(pat, regex=True).values
    print(f"  Subset {name:15s} (n={int(m.sum()):6d}) labels: {np.bincount(sub_df_check['label'].values[m], minlength=6).tolist()}")

Verifying d:\Datathon\nsucec_neurahackl\final_package_turtlers\submission.csv against D:\Datathon\nsucec_neurahackl\Main Preli Round\dataset\sample_submission.csv ...


ALL ASSERTIONS PASSED SUCCESSFULLY!
Label distribution (%):
  label=0:  163458 (32.69%)
  label=1:  110257 (22.05%)
  label=2:   45513 (9.10%)
  label=3:   43184 (8.64%)
  label=4:   47457 (9.49%)
  label=5:   90131 (18.03%)
Action distribution (%):
  action=0:  253589 (50.72%)
  action=1:  101257 (20.25%)
  action=2:  115834 (23.17%)
  action=3:   14757 (2.95%)
  action=4:   10694 (2.14%)
  action=5:    3869 (0.77%)
Top 10 techniques: {'NONE': 253589, 'T1|T2|T5': 82678, 'T1|T3': 28928, 'T1|T3|T4': 28290, 'T1|T3|T6': 12726, 'T1|T3|T5|T6': 11115, 'T3|T5': 9362, 'T1|T3|T5': 9233, 'T1|T4': 8915, 'T1|T2': 7538}
Confidence stats: min=0.3926, p25=0.9842, p50=0.9850, mean=0.9702, p75=0.9850, max=0.9850



Final submission SHA-256: 1c9206daeecbd8a19004d1bbeadaa04ca86945e5fd5c6bfb358987c5a87b0989


  Subset Bangla script   (n= 54837) labels: [16478, 6488, 10209, 6285, 9325, 6052]
  Subset Multi-turn |||  (n=107248) labels: [32539, 27958, 7173, 10768, 9648, 19162]


  Subset Zero-width      (n= 54310) labels: [19263, 12353, 4587, 5240, 4910, 7957]


  Subset Cyrillic/Greek  (n= 42538) labels: [15408, 10192, 3367, 3706, 3692, 6173]
